# Module 04: Federated EV Energy Modelling

## Research report and executable analysis

This notebook models trip-level EV energy consumption in the Munich EV trip dataset and compares centralized and federated Elastic Net learning. It preserves the original executable experiments and recorded outputs while organizing them into the final research workflow.

**Scope.** Munich provides rich driving, battery, thermal, HVAC, environmental, and trip-context telemetry for 70 trips. It has no persistent vehicle, driver, or user identifier, and no direct state-of-health (SOH), capacity, or degradation target. Accordingly, this module concerns driving-behaviour and energy-consumption modelling, not direct battery-SOH prediction. State of charge (SOC) is an observed battery-charge state and is not SOH.

### Navigation

1. Dataset and Experimental Context  
2. Data Loading and Validation  
3. Telemetry Quality Audit  
4. Trip-Level Feature Engineering  
5. Target Construction and Validation  
6. Feature Audit and Reduction  
7. Centralized Elastic Net Baseline  
8. Federated Experimental Design  
9. Simulated Client Construction  
10. FedAvg Elastic Net  
11. FedAvg Optimizer and Hyperparameter Validation  
12. FedPer-Style Personalized Elastic Net  
13. Final Model Comparison  
14. Research Findings  
15. Limitations and Conclusion


In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment ready.")

Environment ready.


## 1. Dataset and Experimental Context

The analysis begins by locating the 70 trip files and overview table. The purpose is to establish a reproducible dataset boundary before any feature engineering or modelling.

## 2. Data Loading and Validation

The following cells load the overview and trip telemetry, then verify one-to-one agreement between the metadata and telemetry files. These checks ensure that all reported analyses use the complete, expected trip population.


In [2]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

MODULE_DIR = Path(r"C:\Users\gayat\projects\get_into\FIP\module_01")

OVERVIEW_PATH = MODULE_DIR / "Overview.xlsx"

TRIP_FILES = sorted(
    list(MODULE_DIR.glob("TripA*.csv")) +
    list(MODULE_DIR.glob("TripB*.csv"))
)

print(f"Module directory : {MODULE_DIR}")
print(f"Overview file    : {OVERVIEW_PATH}")
print(f"Trip files found : {len(TRIP_FILES)}")

assert OVERVIEW_PATH.exists(), "Overview.xlsx not found."
assert len(TRIP_FILES) == 70, f"Expected 70 trip files, found {len(TRIP_FILES)}."

Module directory : C:\Users\gayat\projects\get_into\FIP\module_01
Overview file    : C:\Users\gayat\projects\get_into\FIP\module_01\Overview.xlsx
Trip files found : 70


In [3]:
# ============================================================
# LOAD TRIP OVERVIEW
# ============================================================

overview_df = pd.read_excel(OVERVIEW_PATH)

print("Original shape:", overview_df.shape)
print("\nColumns:")
print(overview_df.columns.tolist())

overview_df.head()

Original shape: (72, 16)

Columns:
['Trip', 'Date', 'Route/Area', 'Weather', 'Battery Temperature (Start) [°C]', 'Battery Temperature (End)', 'Battery State of Charge (Start)', 'Battery State of Charge (End)', 'Unnamed: 8', 'Ambient Temperature (Start) [°C]', 'Target Cabin Temperature', 'Distance [km]', 'Duration [min]', 'Unnamed: 13', 'Fan', 'Note']


,Trip,Date,Route/Area,Weather,Battery Temperature (Start) [°C],Battery Temperature (End),Battery State of Charge (Start),Battery State of Charge (End),Unnamed: 8,Ambient Temperature (Start) [°C],Target Cabin Temperature,Distance [km],Duration [min],Unnamed: 13,Fan,Note
0,TripA01,2019-06-25_13-21-14,Munich East,sunny,21.0,22.0,0.863,0.803,0.060,25.5,23.0,7.427690,16.820000,NaN,"Automatic, Level 1",NaN
1,TripA02,2019-06-25_14-05-31,Munich East,sunny,23.0,26.0,0.803,0.673,0.130,32.0,23.0,23.509709,23.550000,NaN,"Automatic, Level 1",Target Cabin Temperature changed
2,TripA03,2019-06-28_10-02-15,Munich East,sunny,24.0,25.0,0.835,0.751,0.084,21.5,27.0,12.820846,11.180000,NaN,"Automatic, Level 1",Target Cabin Temperature changed
3,TripA04,2019-06-28_10-13-30,Munich East,sunny,25.0,27.0,0.751,0.667,0.084,24.0,22.0,10.727491,6.870000,NaN,"Automatic, Level 1",NaN
4,TripA05,2019-06-28_10-20-26,Munich East,sunny,27.0,27.0,0.667,0.602,0.065,24.5,24.0,12.393223,22.776667,NaN,"Automatic, Level 1",NaN


In [4]:
# ============================================================
# CLEAN OVERVIEW TABLE
# ============================================================

# Remove completely empty rows
overview_df = overview_df.dropna(how="all").reset_index(drop=True)

# Remove completely empty columns
overview_df = overview_df.dropna(axis=1, how="all")

# Rename the derived SOC-change column
if "Unnamed: 8" in overview_df.columns:
    overview_df = overview_df.rename(
        columns={"Unnamed: 8": "SoC Change"}
    )

if "Unnamed: 13" in overview_df.columns:
    overview_df = overview_df.drop(columns=["Unnamed: 13"])

print("Cleaned shape:", overview_df.shape)
print("\nColumns:")
print(overview_df.columns.tolist())

Cleaned shape: (70, 15)

Columns:
['Trip', 'Date', 'Route/Area', 'Weather', 'Battery Temperature (Start) [°C]', 'Battery Temperature (End)', 'Battery State of Charge (Start)', 'Battery State of Charge (End)', 'SoC Change', 'Ambient Temperature (Start) [°C]', 'Target Cabin Temperature', 'Distance [km]', 'Duration [min]', 'Fan', 'Note']


In [5]:
# ============================================================
# VERIFY TRIP FILES AGAINST OVERVIEW
# ============================================================

overview_trips = set(
    overview_df["Trip"].astype(str).str.strip()
)

file_trips = set(
    path.stem.strip()
    for path in TRIP_FILES
)

missing_files = sorted(overview_trips - file_trips)
extra_files = sorted(file_trips - overview_trips)

print("Trips in overview :", len(overview_trips))
print("Telemetry files   :", len(file_trips))
print("Missing files     :", missing_files)
print("Extra files       :", extra_files)

assert not missing_files
assert not extra_files

Trips in overview : 70
Telemetry files   : 70
Missing files     : []
Extra files       : []


In [6]:
# ============================================================
# LOAD TELEMETRY DATA
# ============================================================

trip_data = {}

for i, path in enumerate(TRIP_FILES, start=1):

    trip_name = path.stem

    df = pd.read_csv(
        path,
        sep=";",
        encoding="cp1252"
    )

    trip_data[trip_name] = df

    if i % 10 == 0 or i == len(TRIP_FILES):
        print(f"Loaded {i}/{len(TRIP_FILES)} trips")

print("\nTotal trips loaded:", len(trip_data))

Loaded 10/70 trips
Loaded 20/70 trips
Loaded 30/70 trips
Loaded 40/70 trips
Loaded 50/70 trips
Loaded 60/70 trips
Loaded 70/70 trips

Total trips loaded: 70


## 3. Telemetry Quality Audit

Trip-level row counts, column counts, missingness, and durations are audited before deriving features. The subsequent column-name diagnostics record the correction of a known telemetry header typo in `TripB38`; they are retained as a useful data-quality trace.


In [7]:
# ============================================================
# BASIC TELEMETRY VALIDATION
# ============================================================

validation_rows = []

for trip, df in trip_data.items():

    validation_rows.append({
        "Trip": trip,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Cells": int(df.isna().sum().sum()),
        "Time Start": df["Time [s]"].iloc[0],
        "Time End": df["Time [s]"].iloc[-1],
        "Duration [s]": df["Time [s]"].iloc[-1] - df["Time [s]"].iloc[0]
    })

telemetry_audit = pd.DataFrame(validation_rows)

print(telemetry_audit.describe(include="all"))

           Trip          Rows    Columns  Missing Cells  Time Start  \
count        70     70.000000  70.000000      70.000000        70.0   
unique       70           NaN        NaN            NaN         NaN   
top     TripA01           NaN        NaN            NaN         NaN   
freq          1           NaN        NaN            NaN         NaN   
mean        NaN  15639.900000  37.428571    3627.028571         0.0   
std         NaN   8878.963218  11.720874   15581.814735         0.0   
min         NaN   3545.000000  22.000000       0.000000         0.0   
25%         NaN  10116.250000  23.000000       0.000000         0.0   
50%         NaN  14079.500000  48.000000       0.000000         0.0   
75%         NaN  18313.750000  48.000000      64.000000         0.0   
max         NaN  56102.000000  48.000000  100200.000000         0.0   

           Time End  Duration [s]  
count     70.000000     70.000000  
unique          NaN           NaN  
top             NaN           NaN  
fre

## 4. Trip-Level Feature Engineering

Telemetry time series are summarized into trip-level predictors spanning driving, battery, thermal, HVAC, environmental, and context characteristics. The diagnostic cells that follow validate the required columns and test the feature function before applying it to all trips.


In [8]:
# ============================================================
# TRIP-LEVEL FEATURE ENGINEERING
# ============================================================

def safe_mean(series):
    return series.mean()

def safe_std(series):
    return series.std()

def safe_min(series):
    return series.min()

def safe_max(series):
    return series.max()

def safe_median(series):
    return series.median()

def fraction_above(series, threshold):
    return (series > threshold).mean()

def fraction_below(series, threshold):
    return (series < threshold).mean()


def engineer_trip_features(trip, df, overview_row):

    features = {
        "Trip": trip,

        # ----------------------------------------------------
        # Context
        # ----------------------------------------------------
        "Distance [km]": overview_row["Distance [km]"],
        "Duration [min]": overview_row["Duration [min]"],

        # ----------------------------------------------------
        # SOC
        # ----------------------------------------------------
        "soc_start": overview_row["Battery State of Charge (Start)"],
        "soc_end": overview_row["Battery State of Charge (End)"],
        "soc_change": (
            overview_row["Battery State of Charge (Start)"]
            - overview_row["Battery State of Charge (End)"]
        ),
    }

    # --------------------------------------------------------
    # Helper for statistical features
    # --------------------------------------------------------

    feature_columns = {
        "speed": "Velocity [km/h]",
        "elevation": "Elevation [m]",
        "throttle": "Throttle [%]",
        "torque": "Motor Torque [Nm]",
        "acceleration": "Longitudinal Acceleration [m/s^2]",
        "battery_voltage": "Battery Voltage [V]",
        "battery_current": "Battery Current [A]",
        "battery_temp": "Battery Temperature [°C]",
        "battery_temp_max": "max. Battery Temperature [°C]",
        "soc": "SoC [%]",
        "heating_power": "Heating Power CAN [kW]",
        "requested_heating": "Requested Heating Power [W]",
        "aircon_power": "AirCon Power [kW]",
        "heater_signal": "Heater Signal",
        "ambient_temp": "Ambient Temperature [°C]",
        "heat_exchanger_temp": "Heat Exchanger Temperature [°C]",
        "cabin_temp": "Cabin Temperature Sensor [°C]",
    }

    for prefix, column in feature_columns.items():

        if column not in df.columns:
            continue

        s = pd.to_numeric(df[column], errors="coerce")

        features[f"{prefix}_mean"] = s.mean()
        features[f"{prefix}_std"] = s.std()
        features[f"{prefix}_min"] = s.min()
        features[f"{prefix}_max"] = s.max()
        features[f"{prefix}_median"] = s.median()

    # --------------------------------------------------------
    # Driving behaviour
    # --------------------------------------------------------

    speed = pd.to_numeric(
        df["Velocity [km/h]"], errors="coerce"
    )

    acceleration = pd.to_numeric(
        df["Longitudinal Acceleration [m/s^2]"],
        errors="coerce"
    )

    throttle = pd.to_numeric(
        df["Throttle [%]"], errors="coerce"
    )

    elevation = pd.to_numeric(
        df["Elevation [m]"], errors="coerce"
    )

    features["speed_stationary_fraction"] = (
        speed <= 1
    ).mean()

    features["speed_above_50_fraction"] = (
        speed > 50
    ).mean()

    features["hard_acceleration_fraction"] = (
        acceleration > 2
    ).mean()

    features["hard_deceleration_fraction"] = (
        acceleration < -2
    ).mean()

    features["jerk_mean_abs"] = (
        acceleration.diff().abs().mean()
    )

    # --------------------------------------------------------
    # HVAC / heater activity
    # --------------------------------------------------------

    heater_signal = pd.to_numeric(
        df["Heater Signal"],
        errors="coerce"
    )

    features["heater_active_fraction"] = (
        heater_signal > 0
    ).mean()

    return features

In [10]:
# Check the exact column names in the loaded telemetry
sample_trip = next(iter(trip_data))

print("Sample trip:", sample_trip)
print("\nActual columns:")
for i, col in enumerate(trip_data[sample_trip].columns):
    print(i, repr(col))

Sample trip: TripA01

Actual columns:
0 'Time [s]'
1 'Velocity [km/h]'
2 'Elevation [m]'
3 'Throttle [%]'
4 'Motor Torque [Nm]'
5 'Longitudinal Acceleration [m/s^2]'
6 'Regenerative Braking Signal '
7 'Battery Voltage [V]'
8 'Battery Current [A]'
9 'Battery Temperature [°C]'
10 'max. Battery Temperature [°C]'
11 'SoC [%]'
12 'displayed SoC [%]'
13 'min. SoC [%]'
14 'max. SoC [%)'
15 'Heating Power CAN [kW]'
16 'Heating Power LIN [W]'
17 'Requested Heating Power [W]'
18 'AirCon Power [kW]'
19 'Heater Signal'
20 'Heater Voltage [V]'
21 'Heater Current [A]'
22 'Ambient Temperature [°C]'
23 'Coolant Temperature Heatercore [°C]'
24 'Requested Coolant Temperature [°C]'
25 'Coolant Temperature Inlet [°C]'
26 'Heat Exchanger Temperature [°C]'
27 'Cabin Temperature Sensor [°C]'


In [11]:
# Specifically find columns related to velocity/speed
sample_df = trip_data[sample_trip]

print([
    repr(c)
    for c in sample_df.columns
    if any(word in c.lower() for word in ["velocity", "speed"])
])

["'Velocity [km/h]'"]


In [12]:
# ============================================================
# CHECK REQUIRED COLUMNS ACROSS ALL TRIPS
# ============================================================

required_columns = [
    "Velocity [km/h]",
    "Elevation [m]",
    "Throttle [%]",
    "Motor Torque [Nm]",
    "Longitudinal Acceleration [m/s^2]",
    "Battery Voltage [V]",
    "Battery Current [A]",
    "Battery Temperature [°C]",
    "max. Battery Temperature [°C]",
    "SoC [%]",
    "Heating Power CAN [kW]",
    "Requested Heating Power [W]",
    "AirCon Power [kW]",
    "Heater Signal",
    "Ambient Temperature [°C]",
    "Heat Exchanger Temperature [°C]",
    "Cabin Temperature Sensor [°C]",
]

column_audit = []

for trip, df in trip_data.items():

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    column_audit.append({
        "Trip": trip,
        "Missing Required Columns": missing,
        "Number Missing": len(missing)
    })

column_audit_df = pd.DataFrame(column_audit)

problem_trips = column_audit_df[
    column_audit_df["Number Missing"] > 0
]

print("Trips with missing required columns:")
display(problem_trips)

Trips with missing required columns:


,Trip,Missing Required Columns,Number Missing
69,TripB38,[Velocity [km/h]],1


In [13]:
# Check exact column-name consistency
for trip, df in trip_data.items():

    for col in required_columns:

        matches = [
            c for c in df.columns
            if c.strip() == col.strip()
        ]

        if len(matches) != 1:
            print(
                f"{trip}: expected {repr(col)}, "
                f"found matches = {matches}"
            )

TripB38: expected 'Velocity [km/h]', found matches = []


In [14]:
b38 = trip_data["TripB38"]

print("TripB38 shape:", b38.shape)
print("\nTripB38 columns:")
for i, col in enumerate(b38.columns):
    print(i, repr(col))

TripB38 shape: (16429, 48)

TripB38 columns:
0 'Time [s]'
1 'Velocity [km/h]]]'
2 'Elevation [m]'
3 'Throttle [%]'
4 'Motor Torque [Nm]'
5 'Longitudinal Acceleration [m/s^2]'
6 'Regenerative Braking Signal '
7 'Battery Voltage [V]'
8 'Battery Current [A]'
9 'Battery Temperature [°C]'
10 'max. Battery Temperature [°C]'
11 'SoC [%]'
12 'displayed SoC [%]'
13 'min. SoC [%]'
14 'max. SoC [%)'
15 'Heating Power CAN [kW]'
16 'Heating Power LIN [W]'
17 'Requested Heating Power [W]'
18 'AirCon Power [kW]'
19 'Heater Signal'
20 'Heater Voltage [V]'
21 'Heater Current [A]'
22 'Ambient Temperature [°C]'
23 'Ambient Temperature Sensor [°C]'
24 'Coolant Temperature Heatercore [°C]'
25 'Requested Coolant Temperature [°C]'
26 'Coolant Temperature Inlet [°C]'
27 'Coolant Volume Flow +500 [l/h]'
28 'Heat Exchanger Temperature [°C]'
29 'Cabin Temperature Sensor [°C]'
30 'Temperature Coolant Heater Inlet [°C]'
31 'Temperature Coolant Heater Outlet [°C]'
32 'Temperature Heat Exchanger Outlet [°C]'
33 'Tem

In [15]:
# Normalize known column-name typo in TripB38
for trip, df in trip_data.items():
    df.columns = df.columns.str.strip()

    if "Velocity [km/h]]]" in df.columns:
        df = df.rename(columns={
            "Velocity [km/h]]]": "Velocity [km/h]"
        })

    trip_data[trip] = df

In [16]:
print("TripB38 velocity column:")
print([
    repr(c)
    for c in trip_data["TripB38"].columns
    if "Velocity" in c
])

TripB38 velocity column:
["'Velocity [km/h]'"]


In [17]:
problem_trips = []

for trip, df in trip_data.items():
    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        problem_trips.append((trip, missing))

print("Problem trips:", problem_trips)

Problem trips: []


In [23]:
import inspect

print(inspect.signature(engineer_trip_features))
print("\nFunction source:\n")
print(inspect.getsource(engineer_trip_features))

(trip, df, overview_row)

Function source:

def engineer_trip_features(trip, df, overview_row):

    features = {
        "Trip": trip,

        # ----------------------------------------------------
        # Context
        # ----------------------------------------------------
        "Distance [km]": overview_row["Distance [km]"],
        "Duration [min]": overview_row["Duration [min]"],

        # ----------------------------------------------------
        # SOC
        # ----------------------------------------------------
        "soc_start": overview_row["Battery State of Charge (Start)"],
        "soc_end": overview_row["Battery State of Charge (End)"],
        "soc_change": (
            overview_row["Battery State of Charge (Start)"]
            - overview_row["Battery State of Charge (End)"]
        ),
    }

    # --------------------------------------------------------
    # Helper for statistical features
    # --------------------------------------------------------

 

In [24]:
# Test feature engineering on one trip

test_trip = "TripA01"

test_df = trip_data[test_trip]

test_overview_row = overview_df[
    overview_df["Trip"] == test_trip
].iloc[0]

test_features = engineer_trip_features(
    test_trip,
    test_df,
    test_overview_row
)

print("Feature engineering succeeded!")
print("Number of features:", len(test_features))
print("\nFeatures:")
display(pd.Series(test_features))

Feature engineering succeeded!
Number of features: 97

Features:


Trip                           TripA01
Distance [km]                  7.42769
Duration [min]                   16.82
soc_start                        0.863
soc_end                          0.803
                                ...   
speed_above_50_fraction       0.015956
hard_acceleration_fraction     0.00773
hard_deceleration_fraction     0.00109
jerk_mean_abs                 0.085961
heater_active_fraction             1.0
Length: 97, dtype: object

In [25]:
# ============================================================
# ENGINEER FEATURES FOR ALL TRIPS
# ============================================================

feature_rows = []
feature_errors = []

for trip, df in trip_data.items():

    try:
        overview_match = overview_df[
            overview_df["Trip"] == trip
        ]

        if len(overview_match) != 1:
            raise ValueError(
                f"Expected exactly 1 overview row, "
                f"found {len(overview_match)}"
            )

        overview_row = overview_match.iloc[0]

        features = engineer_trip_features(
            trip,
            df,
            overview_row
        )

        feature_rows.append(features)

    except Exception as e:
        feature_errors.append({
            "Trip": trip,
            "Error": f"{type(e).__name__}: {e}"
        })

features_df = pd.DataFrame(feature_rows)
feature_errors_df = pd.DataFrame(feature_errors)

print("Feature table shape:", features_df.shape)
print("Successfully engineered:", len(features_df))
print("Errors:", len(feature_errors_df))

if len(feature_errors_df) > 0:
    display(feature_errors_df)

Feature table shape: (70, 97)
Successfully engineered: 70
Errors: 0


In [26]:
# Basic feature-table validation

print("Shape:", features_df.shape)
print("Trips:", features_df["Trip"].nunique())

print("\nMissing values:")
display(
    features_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nFirst 5 rows:")
display(features_df.head())

Shape: (70, 97)
Trips: 70

Missing values:


Trip                        0
battery_temp_max_max        0
heater_signal_mean          0
aircon_power_median         0
aircon_power_max            0
aircon_power_min            0
aircon_power_std            0
aircon_power_mean           0
requested_heating_median    0
requested_heating_max       0
requested_heating_min       0
requested_heating_std       0
requested_heating_mean      0
heating_power_median        0
heating_power_max           0
dtype: int64


First 5 rows:


,Trip,Distance [km],Duration [min],soc_start,soc_end,soc_change,speed_mean,speed_std,speed_min,speed_max,...,cabin_temp_std,cabin_temp_min,cabin_temp_max,cabin_temp_median,speed_stationary_fraction,speed_above_50_fraction,hard_acceleration_fraction,hard_deceleration_fraction,jerk_mean_abs,heater_active_fraction
0,TripA01,7.427690,16.820000,0.863,0.803,0.060,26.502170,15.497032,0.00,53.30,...,1.249755,22.17,25.84,24.00,0.125372,0.015956,0.007730,0.001090,0.085961,1.0
1,TripA02,23.509709,23.550000,0.803,0.673,0.130,59.909267,32.868059,1.11,132.70,...,1.053332,19.84,23.34,22.34,0.000000,0.484218,0.005591,0.005379,0.116010,1.0
2,TripA03,12.820846,11.180000,0.835,0.751,0.084,68.826493,38.047229,0.00,143.82,...,1.362068,22.84,27.01,24.84,0.041455,0.631226,0.007307,0.003728,0.124570,1.0
3,TripA04,10.727491,6.870000,0.751,0.667,0.084,93.645415,41.693047,0.00,136.94,...,0.348660,21.84,23.00,22.00,0.088506,0.811106,0.017701,0.000000,0.113252,1.0
4,TripA05,12.393223,22.776667,0.667,0.602,0.065,32.646337,18.151337,0.00,80.83,...,0.582391,22.00,24.50,23.34,0.066364,0.142094,0.004463,0.005049,0.110878,1.0


## 5. Target Construction and Validation

Net battery energy is integrated over each trip and combined with trip distance to construct the target: **Energy Consumption [kWh/km]**. Direction and consistency checks are performed before modelling; this is necessary because energy consumption is meaningful here only for valid discharge-dominant trips.


In [27]:
# ============================================================
# CALCULATE TRIP-LEVEL ENERGY
# ============================================================

energy_rows = []

for trip, df in trip_data.items():

    time = pd.to_numeric(df["Time [s]"], errors="coerce")
    voltage = pd.to_numeric(df["Battery Voltage [V]"], errors="coerce")
    current = pd.to_numeric(df["Battery Current [A]"], errors="coerce")

    valid = (
        time.notna()
        & voltage.notna()
        & current.notna()
    )

    time = time[valid].to_numpy()
    voltage = voltage[valid].to_numpy()
    current = current[valid].to_numpy()

    # Net electrical energy
    net_energy_kwh = np.trapz(
        voltage * current / 1000,
        time
    ) / 3600

    # Absolute electrical energy
    absolute_energy_kwh = np.trapz(
        np.abs(voltage * current / 1000),
        time
    ) / 3600

    # Net battery charge throughput
    net_current_ah = np.trapz(
        current,
        time
    ) / 3600

    energy_rows.append({
        "Trip": trip,
        "Net Energy [kWh]": net_energy_kwh,
        "Absolute Energy [kWh]": absolute_energy_kwh,
        "Net Current [Ah]": net_current_ah
    })

energy_df = pd.DataFrame(energy_rows)

print("Energy table shape:", energy_df.shape)
display(energy_df.head())

Energy table shape: (70, 4)


,Trip,Net Energy [kWh],Absolute Energy [kWh],Net Current [Ah]
0,TripA01,-1.289181,2.056004,-3.350259
1,TripA02,-2.934639,6.235537,-8.000599
2,TripA03,-1.954406,2.864096,-5.200250
3,TripA04,-1.899911,2.420973,-5.161216
4,TripA05,-1.504167,2.855926,-4.021570


In [28]:
# ============================================================
# MERGE FEATURES + ENERGY
# ============================================================

features_df = features_df.merge(
    energy_df,
    on="Trip",
    how="left",
    validate="one_to_one"
)

print("Merged shape:", features_df.shape)

print("\nMissing energy values:")
print(
    features_df[
        [
            "Net Energy [kWh]",
            "Absolute Energy [kWh]",
            "Net Current [Ah]"
        ]
    ].isna().sum()
)

Merged shape: (70, 100)

Missing energy values:
Net Energy [kWh]         0
Absolute Energy [kWh]    0
Net Current [Ah]         0
dtype: int64


In [29]:
# ============================================================
# ENERGY / SOC DIRECTION CHECK
# ============================================================

features_df["SOC Direction"] = np.where(
    features_df["soc_change"] > 0,
    "Discharge",
    np.where(
        features_df["soc_change"] < 0,
        "Charge",
        "No SOC Change"
    )
)

features_df["Energy Direction"] = np.where(
    features_df["Net Energy [kWh]"] < 0,
    "Discharge",
    np.where(
        features_df["Net Energy [kWh]"] > 0,
        "Charge / Net Positive",
        "Zero"
    )
)

direction_check = pd.crosstab(
    features_df["SOC Direction"],
    features_df["Energy Direction"]
)

display(direction_check)

Energy Direction,Charge / Net Positive,Discharge
SOC Direction,,
Charge,1,1
Discharge,1,67


The consistency checks exclude three trips—`TripA31`, `TripB04`, and `TripB09`—without attributing causes beyond the documented checks. The final modelling population is therefore **67 valid discharge-dominant trips**.


In [30]:
# ============================================================
# IDENTIFY VALID DISCHARGE TRIPS
# ============================================================

valid_discharge = (
    (features_df["soc_change"] > 0)
    & (features_df["Net Energy [kWh]"] < 0)
    & (features_df["Distance [km]"] > 0)
)

model_df = features_df[valid_discharge].copy()

excluded_df = features_df[~valid_discharge].copy()

print("Total trips:", len(features_df))
print("Valid discharge trips:", len(model_df))
print("Excluded trips:", len(excluded_df))

print("\nExcluded trips:")
display(
    excluded_df[
        [
            "Trip",
            "soc_start",
            "soc_end",
            "soc_change",
            "Distance [km]",
            "Net Energy [kWh]",
            "SOC Direction",
            "Energy Direction"
        ]
    ]
)

Total trips: 70
Valid discharge trips: 67
Excluded trips: 3

Excluded trips:


,Trip,soc_start,soc_end,soc_change,Distance [km],Net Energy [kWh],SOC Direction,Energy Direction
30,TripA31,0.810,0.825,-0.015,15.976052,-2.113023,Charge,Discharge
35,TripB04,0.451,0.692,-0.241,16.595300,5.783560,Charge,Charge / Net Positive
40,TripB09,0.700,0.460,0.240,54.182500,15.871137,Discharge,Charge / Net Positive


In [32]:
# ============================================================
# CONSTRUCT ENERGY-CONSUMPTION TARGET
# ============================================================

model_df["Energy Consumption [kWh]"] = (
    -model_df["Net Energy [kWh]"]
)

model_df["Energy Consumption [kWh/km]"] = (
    model_df["Energy Consumption [kWh]"]
    / model_df["Distance [km]"]
)

model_df["Energy Direction"] = "Discharge"

print("Modeling dataset shape:", model_df.shape)

display(
    model_df[
        [
            "Trip",
            "Distance [km]",
            "Net Energy [kWh]",
            "Energy Consumption [kWh]",
            "Energy Consumption [kWh/km]"
        ]
    ].head()
)

Modeling dataset shape: (67, 104)


,Trip,Distance [km],Net Energy [kWh],Energy Consumption [kWh],Energy Consumption [kWh/km]
0,TripA01,7.427690,-1.289181,1.289181,0.173564
1,TripA02,23.509709,-2.934639,2.934639,0.124827
2,TripA03,12.820846,-1.954406,1.954406,0.152440
3,TripA04,10.727491,-1.899911,1.899911,0.177107
4,TripA05,12.393223,-1.504167,1.504167,0.121370


In [33]:
# ============================================================
# TARGET VALIDATION
# ============================================================

target = model_df["Energy Consumption [kWh/km]"]

print("Number of observations:", len(target))
print("Missing targets:", target.isna().sum())
print("Infinite targets:", np.isinf(target).sum())
print("Non-positive targets:", (target <= 0).sum())

print("\nTarget statistics:")
print(f"Mean:   {target.mean():.6f} kWh/km")
print(f"Std:    {target.std():.6f} kWh/km")
print(f"Min:    {target.min():.6f} kWh/km")
print(f"Median: {target.median():.6f} kWh/km")
print(f"Max:    {target.max():.6f} kWh/km")

Number of observations: 67
Missing targets: 0
Infinite targets: 0
Non-positive targets: 0

Target statistics:
Mean:   0.165799 kWh/km
Std:    0.033835 kWh/km
Min:    0.095763 kWh/km
Median: 0.165180 kWh/km
Max:    0.233014 kWh/km


## 6. Feature Audit and Reduction

The initial predictor set contains 97 features. Constant and redundant predictors are removed, then median summaries are removed to obtain a compact 77-feature representation. Scaling is applied later using a scaler fitted on training data only, preventing test-set information from entering model fitting.


In [34]:
# ============================================================
# FEATURE CLEANUP
# ============================================================

# Keep target separately
target_columns = [
    "Trip",
    "Net Energy [kWh]",
    "Absolute Energy [kWh]",
    "Net Current [Ah]",
    "Energy Consumption [kWh]",
    "Energy Consumption [kWh/km]",
    "Energy Direction"
]

# Create predictor matrix
X = model_df.drop(columns=target_columns, errors="ignore")
y = model_df["Energy Consumption [kWh/km]"]

print("Initial feature matrix:", X.shape)
print("Target shape:", y.shape)

Initial feature matrix: (67, 97)
Target shape: (67,)


In [35]:
# ============================================================
# REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

print("Constant features:", constant_features)

X_base = X.drop(columns=constant_features)

print("After removing constants:", X_base.shape)

Constant features: ['throttle_min', 'heater_signal_std', 'SOC Direction']
After removing constants: (67, 94)


In [36]:
# ============================================================
# COMPACT FEATURE REPRESENTATION
# ============================================================

median_features = [
    col for col in X_base.columns
    if col.endswith("_median")
]

X_compact = X_base.drop(columns=median_features)

print("Original predictors:", X.shape[1])
print("After constant removal:", X_base.shape[1])
print("Median features removed:", len(median_features))
print("Compact predictors:", X_compact.shape[1])

print("\nRemoved median features:")
print(median_features)

Original predictors: 97
After constant removal: 94
Median features removed: 17
Compact predictors: 77

Removed median features:
['speed_median', 'elevation_median', 'throttle_median', 'torque_median', 'acceleration_median', 'battery_voltage_median', 'battery_current_median', 'battery_temp_median', 'battery_temp_max_median', 'soc_median', 'heating_power_median', 'requested_heating_median', 'aircon_power_median', 'heater_signal_median', 'ambient_temp_median', 'heat_exchanger_temp_median', 'cabin_temp_median']


In [37]:
# ============================================================
# FINAL FEATURE / TARGET SANITY CHECK
# ============================================================

print("X_compact shape:", X_compact.shape)
print("y shape:", y.shape)

print("\nMissing predictor values:", X_compact.isna().sum().sum())
print("Missing target values:", y.isna().sum())
print("Infinite predictor values:", np.isinf(X_compact.select_dtypes(include=np.number)).sum().sum())
print("Infinite target values:", np.isinf(y).sum())

assert X_compact.shape == (67, 77)
assert y.shape == (67,)
assert X_compact.isna().sum().sum() == 0
assert y.isna().sum() == 0
assert np.isinf(y).sum() == 0

print("\n✓ Feature matrix and target are ready.")

X_compact shape: (67, 77)
y shape: (67,)

Missing predictor values: 0
Missing target values: 0
Infinite predictor values: 0
Infinite target values: 0

✓ Feature matrix and target are ready.


## 7. Centralized Elastic Net Baseline

An 80/20 fixed split creates 53 training trips and 14 untouched test trips. This fixed holdout is the reference protocol for the direct comparison with FedAvg and FedPer-style learning. The model uses Elastic Net with `alpha=0.001` and `l1_ratio=0.3`.


In [38]:
# ============================================================
# GLOBAL TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_compact,
    y,
    test_size=0.20,
    random_state=42
)

print("Global training set:", X_train.shape)
print("Global test set:", X_test.shape)

print("\nTarget statistics:")
print(f"Train mean: {y_train.mean():.6f}")
print(f"Test mean:  {y_test.mean():.6f}")

print("\nTrip overlap check:")

train_trips = set(model_df.loc[X_train.index, "Trip"])
test_trips = set(model_df.loc[X_test.index, "Trip"])

print("Train trips:", len(train_trips))
print("Test trips:", len(test_trips))
print("Overlap:", len(train_trips & test_trips))

assert len(train_trips & test_trips) == 0

print("\n✓ Global train/test split is clean.")

Global training set: (53, 77)
Global test set: (14, 77)

Target statistics:
Train mean: 0.169918
Test mean:  0.150204

Trip overlap check:
Train trips: 53
Test trips: 14
Overlap: 0

✓ Global train/test split is clean.


In [39]:
# ============================================================
# CENTRALIZED ELASTIC NET BASELINE
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

centralized_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.001,
        l1_ratio=0.3,
        max_iter=10000,
        random_state=42
    ))
])

# Train ONLY on global training data
centralized_model.fit(X_train, y_train)

# Evaluate on untouched global test set
y_pred_centralized = centralized_model.predict(X_test)

centralized_mae = mean_absolute_error(y_test, y_pred_centralized)
centralized_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_centralized)
)
centralized_r2 = r2_score(y_test, y_pred_centralized)

print("CENTRALIZED ELASTIC NET")
print("=" * 40)
print(f"MAE  : {centralized_mae:.6f} kWh/km")
print(f"RMSE : {centralized_rmse:.6f} kWh/km")
print(f"R²   : {centralized_r2:.6f}")

CENTRALIZED ELASTIC NET
MAE  : 0.012569 kWh/km
RMSE : 0.014817 kWh/km
R²   : 0.805558


**Fixed-holdout result.** Centralized Elastic Net achieves MAE = **0.012569**, RMSE = **0.014817**, and R² = **0.805558**. An earlier repeated 5×10 CV experiment produced MAE = 0.009282, RMSE = 0.011554, and R² = 0.853985; these values use a different evaluation protocol and are not contradictory. The fixed holdout is retained for federated comparison.


In [40]:
# ============================================================
# TEST-SET PREDICTIONS
# ============================================================

test_results = model_df.loc[
    X_test.index,
    ["Trip", "Energy Consumption [kWh/km]"]
].copy()

test_results["Predicted"] = y_pred_centralized
test_results["Absolute Error"] = (
    test_results["Energy Consumption [kWh/km]"]
    - test_results["Predicted"]
).abs()

test_results = test_results.sort_values("Trip")

display(test_results)

print("\nMean absolute error:",
      test_results["Absolute Error"].mean())

,Trip,Energy Consumption [kWh/km],Predicted,Absolute Error
0,TripA01,0.173564,0.154563,0.019001
4,TripA05,0.121370,0.134957,0.013587
5,TripA06,0.109203,0.118650,0.009447
9,TripA10,0.124348,0.116745,0.007603
12,TripA13,0.151130,0.145006,0.006124
16,TripA17,0.158784,0.180944,0.022161
25,TripA26,0.095763,0.130295,0.034532
28,TripA29,0.108976,0.120626,0.011650
38,TripB07,0.184585,0.175086,0.009499
43,TripB12,0.138392,0.131764,0.006629



Mean absolute error: 0.012568869130619805


## 8. Federated Experimental Design

Munich does not identify persistent vehicles, drivers, or users; all clients in this notebook are therefore simulated rather than real participants. The early allocation cells below are retained as exploratory partition diagnostics. They are not the final client construction and their results are not used for the final comparison.


In [41]:
# ============================================================
# SIMULATED FEDERATED CLIENT PARTITION
# ============================================================

from numpy.random import default_rng

NUM_CLIENTS = 5
DIRICHLET_ALPHA = 0.5
RANDOM_STATE = 42

rng = default_rng(RANDOM_STATE)

# ------------------------------------------------------------
# Use ONLY the global training set
# ------------------------------------------------------------

train_indices = np.array(X_train.index)

print("Total training trips:", len(train_indices))
print("Total test trips:", len(X_test))
print("Number of simulated clients:", NUM_CLIENTS)
print("Dirichlet alpha:", DIRICHLET_ALPHA)

# ------------------------------------------------------------
# Dirichlet allocation
# ------------------------------------------------------------

# Generate heterogeneous client proportions
proportions = rng.dirichlet(
    [DIRICHLET_ALPHA] * NUM_CLIENTS
)

print("\nGenerated client proportions:")
for i, p in enumerate(proportions):
    print(f"Client {i+1}: {p:.4f}")

# ------------------------------------------------------------
# Convert proportions into trip counts
# ------------------------------------------------------------

raw_counts = proportions * len(train_indices)
client_counts = np.floor(raw_counts).astype(int)

# Distribute remaining samples according to largest remainders
remaining = len(train_indices) - client_counts.sum()

remainders = raw_counts - client_counts

for idx in np.argsort(remainders)[::-1][:remaining]:
    client_counts[idx] += 1

print("\nClient sample counts:")
for i, count in enumerate(client_counts):
    print(f"Client {i+1}: {count}")

assert client_counts.sum() == len(train_indices)

Total training trips: 53
Total test trips: 14
Number of simulated clients: 5
Dirichlet alpha: 0.5

Generated client proportions:
Client 1: 0.2807
Client 2: 0.4467
Client 3: 0.0031
Client 4: 0.2637
Client 5: 0.0057

Client sample counts:
Client 1: 15
Client 2: 24
Client 3: 0
Client 4: 14
Client 5: 0


In [42]:
# ============================================================
# ASSIGN TRAINING TRIPS TO CLIENTS
# ============================================================

shuffled_indices = rng.permutation(train_indices)

client_data = {}

start = 0

for client_id, count in enumerate(client_counts, start=1):

    client_indices = shuffled_indices[start:start + count]

    client_data[client_id] = {
        "indices": client_indices,
        "X": X_train.loc[client_indices].copy(),
        "y": y_train.loc[client_indices].copy()
    }

    start += count

print("Client datasets created.\n")

for client_id, data in client_data.items():
    print(
        f"Client {client_id}: "
        f"{len(data['indices'])} trips"
    )

Client datasets created.

Client 1: 15 trips
Client 2: 24 trips
Client 3: 0 trips
Client 4: 14 trips
Client 5: 0 trips


In [43]:
# ============================================================
# CLIENT PARTITION VALIDATION
# ============================================================

all_client_indices = np.concatenate([
    data["indices"]
    for data in client_data.values()
])

print("Total client samples:", len(all_client_indices))
print("Unique client samples:", len(np.unique(all_client_indices)))

print(
    "Expected training samples:",
    len(X_train)
)

# Check complete coverage
assert len(all_client_indices) == len(X_train)
assert len(np.unique(all_client_indices)) == len(X_train)

# Check exact training-set membership
assert set(all_client_indices) == set(X_train.index)

# Check no test leakage
assert len(set(all_client_indices) & set(X_test.index)) == 0

print("\n✓ Every training trip assigned exactly once.")
print("✓ No duplicate client assignments.")
print("✓ No test trips leaked into clients.")
print("✓ Test set remains completely untouched.")

Total client samples: 53
Unique client samples: 53
Expected training samples: 53

✓ Every training trip assigned exactly once.
✓ No duplicate client assignments.
✓ No test trips leaked into clients.
✓ Test set remains completely untouched.


In [44]:
# ============================================================
# CLIENT HETEROGENEITY AUDIT
# ============================================================

client_summary = []

for client_id, data in client_data.items():

    client_y = data["y"]

    client_summary.append({
        "Client": client_id,
        "Trips": len(client_y),
        "Target Mean": client_y.mean(),
        "Target Std": client_y.std(),
        "Target Min": client_y.min(),
        "Target Max": client_y.max()
    })

client_summary_df = pd.DataFrame(client_summary)

display(client_summary_df)

print("\nGlobal training target:")
print(f"Mean: {y_train.mean():.6f}")
print(f"Std : {y_train.std():.6f}")
print(f"Min : {y_train.min():.6f}")
print(f"Max : {y_train.max():.6f}")

,Client,Trips,Target Mean,Target Std,Target Min,Target Max
0,1,15,0.173997,0.033960,0.122876,0.217394
1,2,24,0.165048,0.032915,0.098834,0.233014
2,3,0,NaN,NaN,NaN,NaN
3,4,14,0.173897,0.032036,0.114801,0.212259
4,5,0,NaN,NaN,NaN,NaN



Global training target:
Mean: 0.169918
Std : 0.032654
Min : 0.098834
Max : 0.233014


In [45]:
# ============================================================
# SIMULATED CLIENT PARTITION
# GUARANTEED NON-EMPTY CLIENTS
# ============================================================

NUM_CLIENTS = 5
DIRICHLET_ALPHA = 0.5
MIN_TRIPS_PER_CLIENT = 5
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

train_indices = np.array(X_train.index)

n_train = len(train_indices)

assert n_train >= NUM_CLIENTS * MIN_TRIPS_PER_CLIENT

# ------------------------------------------------------------
# Reserve minimum number of trips for every client
# ------------------------------------------------------------

remaining = n_train - (
    NUM_CLIENTS * MIN_TRIPS_PER_CLIENT
)

# Dirichlet allocation for remaining trips
proportions = rng.dirichlet(
    [DIRICHLET_ALPHA] * NUM_CLIENTS
)

additional_counts = np.floor(
    proportions * remaining
).astype(int)

# Distribute rounding remainder
remainder = remaining - additional_counts.sum()

fractional = (
    proportions * remaining
    - additional_counts
)

for idx in np.argsort(fractional)[::-1][:remainder]:
    additional_counts[idx] += 1

client_counts = (
    MIN_TRIPS_PER_CLIENT
    + additional_counts
)

print("Client proportions from Dirichlet:")
for i, p in enumerate(proportions):
    print(f"Client {i+1}: {p:.4f}")

print("\nFinal client counts:")
for i, count in enumerate(client_counts):
    print(f"Client {i+1}: {count}")

print("\nTotal:", client_counts.sum())

assert client_counts.sum() == n_train
assert np.all(client_counts >= MIN_TRIPS_PER_CLIENT)

Client proportions from Dirichlet:
Client 1: 0.2807
Client 2: 0.4467
Client 3: 0.0031
Client 4: 0.2637
Client 5: 0.0057

Final client counts:
Client 1: 13
Client 2: 18
Client 3: 5
Client 4: 12
Client 5: 5

Total: 53


In [46]:
# ============================================================
# CREATE CLIENT DATASETS
# ============================================================

shuffled_indices = rng.permutation(train_indices)

client_data = {}

start = 0

for client_id, count in enumerate(client_counts, start=1):

    client_indices = shuffled_indices[
        start:start + count
    ]

    client_data[client_id] = {
        "indices": client_indices,
        "X": X_train.loc[client_indices].copy(),
        "y": y_train.loc[client_indices].copy()
    }

    start += count

print("Client datasets created.\n")

for client_id, data in client_data.items():
    print(
        f"Client {client_id}: "
        f"{len(data['indices'])} trips"
    )

Client datasets created.

Client 1: 13 trips
Client 2: 18 trips
Client 3: 5 trips
Client 4: 12 trips
Client 5: 5 trips


In [47]:
# ============================================================
# CLIENT PARTITION VALIDATION
# ============================================================

all_client_indices = np.concatenate([
    data["indices"]
    for data in client_data.values()
])

assert len(all_client_indices) == len(X_train)
assert len(np.unique(all_client_indices)) == len(X_train)

assert set(all_client_indices) == set(X_train.index)

# Absolutely no test leakage
assert len(
    set(all_client_indices) & set(X_test.index)
) == 0

# Every client must contain data
assert all(
    len(data["indices"]) > 0
    for data in client_data.values()
)

print("Total client samples:", len(all_client_indices))
print("Unique client samples:", len(np.unique(all_client_indices)))
print("Expected training samples:", len(X_train))

print("\n✓ Every training trip assigned exactly once.")
print("✓ No duplicate client assignments.")
print("✓ No test trips leaked into clients.")
print("✓ Every client contains training data.")
print("✓ Test set remains completely untouched.")

Total client samples: 53
Unique client samples: 53
Expected training samples: 53

✓ Every training trip assigned exactly once.
✓ No duplicate client assignments.
✓ No test trips leaked into clients.
✓ Every client contains training data.
✓ Test set remains completely untouched.


In [48]:
# ============================================================
# CLIENT HETEROGENEITY AUDIT
# ============================================================

client_summary = []

for client_id, data in client_data.items():

    client_y = data["y"]

    client_summary.append({
        "Client": client_id,
        "Trips": len(client_y),
        "Target Mean": client_y.mean(),
        "Target Std": client_y.std(),
        "Target Min": client_y.min(),
        "Target Max": client_y.max()
    })

client_summary_df = pd.DataFrame(client_summary)

display(client_summary_df)

print("\nGlobal training target:")
print(f"Mean: {y_train.mean():.6f}")
print(f"Std : {y_train.std():.6f}")
print(f"Min : {y_train.min():.6f}")
print(f"Max : {y_train.max():.6f}")

,Client,Trips,Target Mean,Target Std,Target Min,Target Max
0,1,13,0.172236,0.035011,0.122876,0.217394
1,2,18,0.165645,0.033713,0.098834,0.233014
2,3,5,0.171583,0.040522,0.118895,0.211049
3,4,12,0.164192,0.030600,0.114801,0.204793
4,5,5,0.191353,0.019002,0.173580,0.212259



Global training target:
Mean: 0.169918
Std : 0.032654
Min : 0.098834
Max : 0.233014


### Archived preliminary FedAvg experiments

The following cells preserve early Dirichlet-partition and preliminary FedAvg attempts. They are diagnostic development history, not final results. In particular, neither the preliminary partition nor its hyperparameters defines the final experiment; the final balanced condition-based partition begins below.


In [49]:
# ============================================================
# FEDAVG ELASTIC NET
# LOCAL OPTIMIZATION SETUP
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# Federated hyperparameters
# ------------------------------------------------------------

FED_ALPHA = 0.001
FED_L1_RATIO = 0.3

NUM_ROUNDS = 30
LOCAL_EPOCHS = 20
LEARNING_RATE = 0.01

print("FedAvg Elastic Net")
print("=" * 50)
print("Clients:", NUM_CLIENTS)
print("Rounds:", NUM_ROUNDS)
print("Local epochs:", LOCAL_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Alpha:", FED_ALPHA)
print("L1 ratio:", FED_L1_RATIO)

FedAvg Elastic Net
Clients: 5
Rounds: 30
Local epochs: 20
Learning rate: 0.01
Alpha: 0.001
L1 ratio: 0.3


In [50]:
# ============================================================
# GLOBAL TRAINING SCALER
# ============================================================

global_scaler = StandardScaler()

X_train_scaled = global_scaler.fit_transform(X_train)
X_test_scaled = global_scaler.transform(X_test)

print("Training matrix:", X_train_scaled.shape)
print("Test matrix:", X_test_scaled.shape)

print("\n✓ Scaler fitted only on training data.")

Training matrix: (53, 77)
Test matrix: (14, 77)

✓ Scaler fitted only on training data.


In [51]:
# ============================================================
# ELASTIC NET LOCAL OPTIMIZER
# ============================================================

def soft_threshold(value, threshold):
    """
    Proximal operator for the L1 penalty.
    """
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_elastic_net(
    X,
    y,
    coef,
    intercept,
    alpha=0.001,
    l1_ratio=0.3,
    learning_rate=0.01,
    epochs=20
):
    """
    Perform local Elastic Net optimization using
    gradient descent + proximal L1 update.

    L2 penalty is included in the gradient.
    L1 penalty is handled through soft thresholding.
    """

    beta = coef.copy()
    b = float(intercept)

    n = len(y)

    for _ in range(epochs):

        # Prediction
        prediction = X @ beta + b

        # Residual
        error = prediction - y

        # Gradient of squared-error loss
        grad_beta = (X.T @ error) / n
        grad_intercept = error.mean()

        # L2 component of Elastic Net penalty
        grad_beta += (
            learning_rate
            * 0
        )

        # Gradient descent for smooth part
        beta -= learning_rate * (
            grad_beta
            + alpha * (1 - l1_ratio) * beta
        )

        b -= learning_rate * grad_intercept

        # Proximal L1 step
        beta = soft_threshold(
            beta,
            learning_rate * alpha * l1_ratio
        )

    return beta, b

In [52]:
# ============================================================
# FINAL FEDAVG ELASTIC NET LOCAL OPTIMIZER
# ============================================================

def soft_threshold(value, threshold):
    """
    Proximal operator for the L1 penalty.
    """
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_elastic_net(
    X,
    y,
    coef,
    intercept,
    alpha=0.001,
    l1_ratio=0.3,
    learning_rate=0.01,
    epochs=20
):
    """
    Local Elastic Net optimization.

    Objective:
        (1 / 2n) * ||y - Xβ - b||²
        + alpha * l1_ratio * ||β||₁
        + alpha * (1-l1_ratio)/2 * ||β||²

    Smooth terms:
        squared error + L2 penalty

    Non-smooth term:
        L1 penalty handled using the proximal
        soft-thresholding operator.
    """

    beta = coef.copy()
    b = float(intercept)

    n = X.shape[0]

    for _ in range(epochs):

        # ----------------------------------------------------
        # Prediction and residual
        # ----------------------------------------------------

        prediction = X @ beta + b
        error = prediction - y

        # ----------------------------------------------------
        # Gradient of squared-error loss
        # ----------------------------------------------------

        grad_beta = (X.T @ error) / n
        grad_intercept = error.mean()

        # ----------------------------------------------------
        # Add L2 component
        # ----------------------------------------------------

        grad_beta += (
            alpha
            * (1 - l1_ratio)
            * beta
        )

        # ----------------------------------------------------
        # Gradient step for smooth component
        # ----------------------------------------------------

        beta -= learning_rate * grad_beta
        b -= learning_rate * grad_intercept

        # ----------------------------------------------------
        # Proximal L1 step
        # ----------------------------------------------------

        beta = soft_threshold(
            beta,
            learning_rate
            * alpha
            * l1_ratio
        )

    return beta, b

In [53]:
# ============================================================
# FEDAVG GLOBAL MODEL INITIALIZATION
# ============================================================

n_features = X_train_scaled.shape[1]

global_coef = np.zeros(n_features)
global_intercept = y_train.mean()

print("Number of features:", n_features)
print("Initial intercept:", global_intercept)
print("Initial non-zero coefficients:",
      np.count_nonzero(global_coef))

Number of features: 77
Initial intercept: 0.16991800825069805
Initial non-zero coefficients: 0


In [54]:
# ============================================================
# PREPARE SCALED CLIENT DATA
# ============================================================

federated_clients = {}

for client_id, data in client_data.items():

    indices = data["indices"]

    # Find positions within X_train
    positions = [
        X_train.index.get_loc(idx)
        for idx in indices
    ]

    federated_clients[client_id] = {
        "X": X_train_scaled[positions],
        "y": y_train.loc[indices].to_numpy(),
        "indices": indices
    }

for client_id, data in federated_clients.items():
    print(
        f"Client {client_id}: "
        f"X={data['X'].shape}, "
        f"y={data['y'].shape}"
    )

Client 1: X=(13, 77), y=(13,)
Client 2: X=(18, 77), y=(18,)
Client 3: X=(5, 77), y=(5,)
Client 4: X=(12, 77), y=(12,)
Client 5: X=(5, 77), y=(5,)


In [55]:
# ============================================================
# FEDAVG TRAINING
# ============================================================

history = []

global_coef = np.zeros(n_features)
global_intercept = y_train.mean()

for round_num in range(1, NUM_ROUNDS + 1):

    local_models = []
    total_samples = 0

    # --------------------------------------------------------
    # Local training
    # --------------------------------------------------------

    for client_id, data in federated_clients.items():

        local_coef, local_intercept = local_elastic_net(
            X=data["X"],
            y=data["y"],
            coef=global_coef,
            intercept=global_intercept,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=LEARNING_RATE,
            epochs=LOCAL_EPOCHS
        )

        n_samples = len(data["y"])

        local_models.append({
            "client_id": client_id,
            "coef": local_coef,
            "intercept": local_intercept,
            "samples": n_samples
        })

        total_samples += n_samples

    # --------------------------------------------------------
    # FedAvg weighted aggregation
    # --------------------------------------------------------

    new_coef = np.zeros_like(global_coef)
    new_intercept = 0.0

    for model in local_models:

        weight = model["samples"] / total_samples

        new_coef += weight * model["coef"]
        new_intercept += weight * model["intercept"]

    global_coef = new_coef
    global_intercept = new_intercept

    # --------------------------------------------------------
    # Global training-set evaluation
    # --------------------------------------------------------

    train_pred = (
        X_train_scaled @ global_coef
        + global_intercept
    )

    train_mae = mean_absolute_error(
        y_train,
        train_pred
    )

    history.append({
        "Round": round_num,
        "Train MAE": train_mae,
        "Nonzero Coefficients": np.count_nonzero(
            np.abs(global_coef) > 1e-10
        )
    })

    if (
        round_num == 1
        or round_num % 5 == 0
        or round_num == NUM_ROUNDS
    ):
        print(
            f"Round {round_num:02d} | "
            f"Train MAE: {train_mae:.6f} | "
            f"Non-zero: "
            f"{np.count_nonzero(np.abs(global_coef) > 1e-10)}"
        )

print("\n✓ FedAvg training completed.")

Round 01 | Train MAE: 0.011919 | Non-zero: 77
Round 05 | Train MAE: 0.008333 | Non-zero: 77
Round 10 | Train MAE: 0.006938 | Non-zero: 77
Round 15 | Train MAE: 0.006368 | Non-zero: 77
Round 20 | Train MAE: 0.006007 | Non-zero: 77
Round 25 | Train MAE: 0.005766 | Non-zero: 77
Round 30 | Train MAE: 0.005582 | Non-zero: 77

✓ FedAvg training completed.


In [56]:
# ============================================================
# FEDAVG GLOBAL TEST EVALUATION
# ============================================================

fedavg_test_pred = (
    X_test_scaled @ global_coef
    + global_intercept
)

fedavg_mae = mean_absolute_error(
    y_test,
    fedavg_test_pred
)

fedavg_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        fedavg_test_pred
    )
)

fedavg_r2 = r2_score(
    y_test,
    fedavg_test_pred
)

print("\nFEDAVG ELASTIC NET")
print("=" * 40)
print(f"MAE  : {fedavg_mae:.6f} kWh/km")
print(f"RMSE : {fedavg_rmse:.6f} kWh/km")
print(f"R²   : {fedavg_r2:.6f}")


FEDAVG ELASTIC NET
MAE  : 0.013082 kWh/km
RMSE : 0.016569 kWh/km
R²   : 0.756872


In [57]:
# ============================================================
# FEDAVG GLOBAL TEST EVALUATION
# ============================================================

fedavg_test_pred = (
    X_test_scaled @ global_coef
    + global_intercept
)

fedavg_mae = mean_absolute_error(
    y_test,
    fedavg_test_pred
)

fedavg_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        fedavg_test_pred
    )
)

fedavg_r2 = r2_score(
    y_test,
    fedavg_test_pred
)

print("\nFEDAVG ELASTIC NET")
print("=" * 40)
print(f"MAE  : {fedavg_mae:.6f} kWh/km")
print(f"RMSE : {fedavg_rmse:.6f} kWh/km")
print(f"R²   : {fedavg_r2:.6f}")


FEDAVG ELASTIC NET
MAE  : 0.013082 kWh/km
RMSE : 0.016569 kWh/km
R²   : 0.756872


In [58]:
# ============================================================
# CENTRALIZED VS FEDAVG
# ============================================================

comparison = pd.DataFrame({
    "Model": [
        "Centralized Elastic Net",
        "FedAvg Elastic Net"
    ],
    "MAE": [
        centralized_mae,
        fedavg_mae
    ],
    "RMSE": [
        centralized_rmse,
        fedavg_rmse
    ],
    "R²": [
        centralized_r2,
        fedavg_r2
    ]
})

display(comparison)

,Model,MAE,RMSE,R²
0,Centralized Elastic Net,0.012569,0.014817,0.805558
1,FedAvg Elastic Net,0.013082,0.016569,0.756872


In [74]:
# ============================================================
# FEDAVG TRAINING - 2 COMMUNICATION ROUNDS
# ============================================================

NUM_ROUNDS = 70

history = []

# Reinitialize global model
global_coef = np.zeros(n_features)
global_intercept = y_train.mean()

for round_num in range(1, NUM_ROUNDS + 1):

    local_models = []
    total_samples = 0

    # --------------------------------------------------------
    # Local training on every client
    # --------------------------------------------------------

    for client_id, data in federated_clients.items():

        local_coef, local_intercept = local_elastic_net(
            X=data["X"],
            y=data["y"],
            coef=global_coef,
            intercept=global_intercept,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=LEARNING_RATE,
            epochs=LOCAL_EPOCHS
        )

        n_samples = len(data["y"])

        local_models.append({
            "client_id": client_id,
            "coef": local_coef,
            "intercept": local_intercept,
            "samples": n_samples
        })

        total_samples += n_samples

    # --------------------------------------------------------
    # FedAvg aggregation
    # --------------------------------------------------------

    new_coef = np.zeros_like(global_coef)
    new_intercept = 0.0

    for model in local_models:

        weight = model["samples"] / total_samples

        new_coef += weight * model["coef"]
        new_intercept += weight * model["intercept"]

    global_coef = new_coef
    global_intercept = new_intercept

    # --------------------------------------------------------
    # Evaluate global model on training data
    # --------------------------------------------------------

    train_pred = (
        X_train_scaled @ global_coef
        + global_intercept
    )

    train_mae = mean_absolute_error(
        y_train,
        train_pred
    )

    history.append({
        "Round": round_num,
        "Train MAE": train_mae,
        "Nonzero Coefficients": np.count_nonzero(
            np.abs(global_coef) > 1e-10
        )
    })

    print(
        f"Round {round_num} | "
        f"Train MAE: {train_mae:.6f} | "
        f"Non-zero coefficients: "
        f"{np.count_nonzero(np.abs(global_coef) > 1e-10)}"
    )

print("\n✓ 2 FedAvg communication rounds completed.")

Round 1 | Train MAE: 0.011919 | Non-zero coefficients: 77
Round 2 | Train MAE: 0.010389 | Non-zero coefficients: 77
Round 3 | Train MAE: 0.009483 | Non-zero coefficients: 77
Round 4 | Train MAE: 0.008817 | Non-zero coefficients: 77
Round 5 | Train MAE: 0.008333 | Non-zero coefficients: 77
Round 6 | Train MAE: 0.007933 | Non-zero coefficients: 77
Round 7 | Train MAE: 0.007596 | Non-zero coefficients: 77
Round 8 | Train MAE: 0.007324 | Non-zero coefficients: 77
Round 9 | Train MAE: 0.007108 | Non-zero coefficients: 77
Round 10 | Train MAE: 0.006938 | Non-zero coefficients: 77
Round 11 | Train MAE: 0.006799 | Non-zero coefficients: 77
Round 12 | Train MAE: 0.006674 | Non-zero coefficients: 77
Round 13 | Train MAE: 0.006563 | Non-zero coefficients: 77
Round 14 | Train MAE: 0.006461 | Non-zero coefficients: 77
Round 15 | Train MAE: 0.006368 | Non-zero coefficients: 77
Round 16 | Train MAE: 0.006282 | Non-zero coefficients: 77
Round 17 | Train MAE: 0.006204 | Non-zero coefficients: 77
Round 

In [75]:
# ============================================================
# FEDAVG ROUND 12 TEST EVALUATION
# ============================================================

fedavg_test_pred = (
    X_test_scaled @ global_coef
    + global_intercept
)

fedavg_mae = mean_absolute_error(
    y_test,
    fedavg_test_pred
)

fedavg_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        fedavg_test_pred
    )
)

fedavg_r2 = r2_score(
    y_test,
    fedavg_test_pred
)

print("FEDAVG ELASTIC NET - ROUND 12")
print("=" * 45)
print(f"MAE  : {fedavg_mae:.6f} kWh/km")
print(f"RMSE : {fedavg_rmse:.6f} kWh/km")
print(f"R²   : {fedavg_r2:.6f}")

FEDAVG ELASTIC NET - ROUND 12
MAE  : 0.013109 kWh/km
RMSE : 0.015952 kWh/km
R²   : 0.774648


In [77]:
# ============================================================
# FEDAVG CONVERGENCE ANALYSIS
# ============================================================

rounds_to_test = [1, 2, 3, 5, 7, 10, 12, 15, 20, 25, 30, 38, 47, 55, 67, 71]

round_results = []

# Reinitialize the global model
global_coef = np.zeros(n_features)
global_intercept = y_train.mean()

for round_num in range(1, max(rounds_to_test) + 1):

    local_models = []
    total_samples = 0

    # --------------------------------------------------------
    # Local training on every client
    # --------------------------------------------------------

    for client_id, data in federated_clients.items():

        local_coef, local_intercept = local_elastic_net(
            X=data["X"],
            y=data["y"],
            coef=global_coef,
            intercept=global_intercept,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=LEARNING_RATE,
            epochs=LOCAL_EPOCHS
        )

        n_samples = len(data["y"])

        local_models.append({
            "coef": local_coef,
            "intercept": local_intercept,
            "samples": n_samples
        })

        total_samples += n_samples

    # --------------------------------------------------------
    # FedAvg weighted aggregation
    # --------------------------------------------------------

    global_coef = sum(
        (m["samples"] / total_samples) * m["coef"]
        for m in local_models
    )

    global_intercept = sum(
        (m["samples"] / total_samples) * m["intercept"]
        for m in local_models
    )

    # --------------------------------------------------------
    # Training predictions
    # --------------------------------------------------------

    train_pred = (
        X_train_scaled @ global_coef
        + global_intercept
    )

    train_mae = mean_absolute_error(
        y_train,
        train_pred
    )

    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_pred)
    )

    train_r2 = r2_score(
        y_train,
        train_pred
    )

    # --------------------------------------------------------
    # Test predictions
    # --------------------------------------------------------

    test_pred = (
        X_test_scaled @ global_coef
        + global_intercept
    )

    test_mae = mean_absolute_error(
        y_test,
        test_pred
    )

    test_rmse = np.sqrt(
        mean_squared_error(y_test, test_pred)
    )

    test_r2 = r2_score(
        y_test,
        test_pred
    )

    # --------------------------------------------------------
    # Store selected rounds
    # --------------------------------------------------------

    if round_num in rounds_to_test:

        round_results.append({
            "Rounds": round_num,

            "Train MAE": train_mae,
            "Train RMSE": train_rmse,
            "Train R²": train_r2,

            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R²": test_r2
        })

round_results_df = pd.DataFrame(round_results)

display(
    round_results_df.round(6)
)

,Rounds,Train MAE,Train RMSE,Train R²,Test MAE,Test RMSE,Test R²
0,1,0.011919,0.014416,0.801351,0.018810,0.023995,0.490061
1,2,0.010389,0.012447,0.851908,0.016984,0.022271,0.560741
2,3,0.009483,0.011378,0.876250,0.016021,0.021283,0.598833
3,5,0.008333,0.010048,0.903483,0.015247,0.020050,0.643961
4,7,0.007596,0.009228,0.918610,0.014673,0.019314,0.669612
5,10,0.006938,0.008470,0.931426,0.014201,0.018656,0.691760
6,12,0.006674,0.008130,0.936815,0.013979,0.018327,0.702529
7,15,0.006368,0.007753,0.942541,0.013695,0.017878,0.716944
8,20,0.006007,0.007325,0.948713,0.013401,0.017285,0.735383
9,25,0.005766,0.007044,0.952568,0.013214,0.016872,0.747889


In [78]:
# ============================================================
# FEDAVG CONVERGENCE VS CENTRALIZED BASELINE
# ============================================================

print("Centralized Elastic Net baseline:")
print(f"MAE  : {centralized_mae:.6f}")
print(f"RMSE : {centralized_rmse:.6f}")
print(f"R²   : {centralized_r2:.6f}")

print("\nFedAvg convergence:")
display(round_results_df.round(6))

Centralized Elastic Net baseline:
MAE  : 0.012569
RMSE : 0.014817
R²   : 0.805558

FedAvg convergence:


,Rounds,Train MAE,Train RMSE,Train R²,Test MAE,Test RMSE,Test R²
0,1,0.011919,0.014416,0.801351,0.018810,0.023995,0.490061
1,2,0.010389,0.012447,0.851908,0.016984,0.022271,0.560741
2,3,0.009483,0.011378,0.876250,0.016021,0.021283,0.598833
3,5,0.008333,0.010048,0.903483,0.015247,0.020050,0.643961
4,7,0.007596,0.009228,0.918610,0.014673,0.019314,0.669612
5,10,0.006938,0.008470,0.931426,0.014201,0.018656,0.691760
6,12,0.006674,0.008130,0.936815,0.013979,0.018327,0.702529
7,15,0.006368,0.007753,0.942541,0.013695,0.017878,0.716944
8,20,0.006007,0.007325,0.948713,0.013401,0.017285,0.735383
9,25,0.005766,0.007044,0.952568,0.013214,0.016872,0.747889


### Archived preliminary FedPer experiments

These cells document superseded personalized-linear-model explorations. They are retained for reproducibility but are not final benchmarks. The final FedPer-style approach is the conservative intercept-only formulation reported later.


In [79]:
# ============================================================
# FEDPER ELASTIC NET INITIALIZATION
# ============================================================

FEDPER_ALPHA = 0.001
FEDPER_L1_RATIO = 0.3

FEDPER_ROUNDS = 3
FEDPER_LOCAL_EPOCHS = 20
FEDPER_LEARNING_RATE = 0.01

n_features = X_train_scaled.shape[1]

# Shared/global component
fedper_global_coef = np.zeros(n_features)
fedper_global_intercept = y_train.mean()

# Client-specific personalized components
fedper_personal = {}

for client_id in federated_clients:

    fedper_personal[client_id] = {
        "coef": np.zeros(n_features),
        "intercept": 0.0
    }

print("FedPer initialized")
print("Clients:", len(federated_clients))
print("Features:", n_features)
print("Rounds:", FEDPER_ROUNDS)
print("Local epochs:", FEDPER_LOCAL_EPOCHS)

FedPer initialized
Clients: 5
Features: 77
Rounds: 3
Local epochs: 20


In [80]:
# ============================================================
# FEDPER LOCAL OPTIMIZATION
# ============================================================

def local_fedper(
    X,
    y,
    global_coef,
    global_intercept,
    personal_coef,
    personal_intercept,
    alpha=0.001,
    l1_ratio=0.3,
    learning_rate=0.01,
    epochs=20
):

    shared = global_coef.copy()
    personal = personal_coef.copy()

    shared_bias = float(global_intercept)
    personal_bias = float(personal_intercept)

    n = X.shape[0]

    for _ in range(epochs):

        # ----------------------------------------------------
        # Combined personalized prediction
        # ----------------------------------------------------

        prediction = (
            X @ (shared + personal)
            + shared_bias
            + personal_bias
        )

        error = prediction - y

        # ----------------------------------------------------
        # Gradient
        # ----------------------------------------------------

        grad = (X.T @ error) / n
        grad_bias = error.mean()

        # ----------------------------------------------------
        # Update shared component
        # ----------------------------------------------------

        shared -= learning_rate * (
            grad
            + alpha * (1 - l1_ratio) * shared
        )

        # ----------------------------------------------------
        # Update personalized component
        # ----------------------------------------------------

        personal -= learning_rate * (
            grad
            + alpha * (1 - l1_ratio) * personal
        )

        # ----------------------------------------------------
        # L1 proximal step
        # ----------------------------------------------------

        shared = soft_threshold(
            shared,
            learning_rate * alpha * l1_ratio
        )

        personal = soft_threshold(
            personal,
            learning_rate * alpha * l1_ratio
        )

        # ----------------------------------------------------
        # Bias updates
        # ----------------------------------------------------

        shared_bias -= learning_rate * grad_bias

        personal_bias -= learning_rate * grad_bias

    return (
        shared,
        shared_bias,
        personal,
        personal_bias
    )

In [81]:
# ============================================================
# FEDPER TRAINING
# ============================================================

fedper_history = []

for round_num in range(1, FEDPER_ROUNDS + 1):

    local_shared_models = []

    total_samples = 0

    # --------------------------------------------------------
    # Local training
    # --------------------------------------------------------

    for client_id, data in federated_clients.items():

        result = local_fedper(
            X=data["X"],
            y=data["y"],
            global_coef=fedper_global_coef,
            global_intercept=fedper_global_intercept,
            personal_coef=fedper_personal[client_id]["coef"],
            personal_intercept=fedper_personal[client_id]["intercept"],
            alpha=FEDPER_ALPHA,
            l1_ratio=FEDPER_L1_RATIO,
            learning_rate=FEDPER_LEARNING_RATE,
            epochs=FEDPER_LOCAL_EPOCHS
        )

        (
            local_shared_coef,
            local_shared_intercept,
            local_personal_coef,
            local_personal_intercept
        ) = result

        # Store personalized component locally
        fedper_personal[client_id]["coef"] = (
            local_personal_coef
        )

        fedper_personal[client_id]["intercept"] = (
            local_personal_intercept
        )

        n_samples = len(data["y"])

        local_shared_models.append({
            "coef": local_shared_coef,
            "intercept": local_shared_intercept,
            "samples": n_samples
        })

        total_samples += n_samples

    # --------------------------------------------------------
    # Aggregate ONLY shared component
    # --------------------------------------------------------

    fedper_global_coef = sum(
        (m["samples"] / total_samples) * m["coef"]
        for m in local_shared_models
    )

    fedper_global_intercept = sum(
        (m["samples"] / total_samples) * m["intercept"]
        for m in local_shared_models
    )

    # --------------------------------------------------------
    # Evaluate personalized training performance
    # --------------------------------------------------------

    client_maes = []

    for client_id, data in federated_clients.items():

        personal = fedper_personal[client_id]

        prediction = (
            data["X"]
            @ (
                fedper_global_coef
                + personal["coef"]
            )
            + fedper_global_intercept
            + personal["intercept"]
        )

        client_mae = mean_absolute_error(
            data["y"],
            prediction
        )

        client_maes.append(client_mae)

    mean_client_mae = np.mean(client_maes)

    fedper_history.append({
        "Round": round_num,
        "Mean Client MAE": mean_client_mae,
        "Client MAE Std": np.std(client_maes)
    })

    print(
        f"Round {round_num} | "
        f"Mean Client MAE: {mean_client_mae:.6f} | "
        f"Client MAE Std: {np.std(client_maes):.6f}"
    )

print("\n✓ FedPer training completed.")

Round 1 | Mean Client MAE: 0.007304 | Client MAE Std: 0.001866
Round 2 | Mean Client MAE: 0.004685 | Client MAE Std: 0.001811
Round 3 | Mean Client MAE: 0.003514 | Client MAE Std: 0.001623

✓ FedPer training completed.


In [82]:
# ============================================================
# FEDPER CLIENT PERFORMANCE
# ============================================================

fedper_client_results = []

for client_id, data in federated_clients.items():

    personal = fedper_personal[client_id]

    prediction = (
        data["X"]
        @ (
            fedper_global_coef
            + personal["coef"]
        )
        + fedper_global_intercept
        + personal["intercept"]
    )

    mae = mean_absolute_error(
        data["y"],
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            data["y"],
            prediction
        )
    )

    r2 = r2_score(
        data["y"],
        prediction
    )

    fedper_client_results.append({
        "Client": client_id,
        "Trips": len(data["y"]),
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

fedper_client_results_df = pd.DataFrame(
    fedper_client_results
)

display(
    fedper_client_results_df.round(6)
)

,Client,Trips,MAE,RMSE,R²
0,1,13,0.003112,0.003776,0.987397
1,2,18,0.006684,0.008357,0.934937
2,3,5,0.002080,0.002521,0.995163
3,4,12,0.002940,0.003520,0.985566
4,5,5,0.002751,0.003387,0.960285


In [83]:
# ============================================================
# FEDPER SHARED COMPONENT - GLOBAL TEST EVALUATION
# ============================================================

fedper_global_test_pred = (
    X_test_scaled @ fedper_global_coef
    + fedper_global_intercept
)

fedper_global_mae = mean_absolute_error(
    y_test,
    fedper_global_test_pred
)

fedper_global_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        fedper_global_test_pred
    )
)

fedper_global_r2 = r2_score(
    y_test,
    fedper_global_test_pred
)

print("FEDPER SHARED COMPONENT - GLOBAL TEST")
print("=" * 50)
print(f"MAE  : {fedper_global_mae:.6f} kWh/km")
print(f"RMSE : {fedper_global_rmse:.6f} kWh/km")
print(f"R²   : {fedper_global_r2:.6f}")

FEDPER SHARED COMPONENT - GLOBAL TEST
MAE  : 0.022001 kWh/km
RMSE : 0.027603 kWh/km
R²   : 0.325186


In [84]:
# ============================================================
# FEDPER: CLIENT PARTITION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
N_CLIENTS = 5

# ------------------------------------------------------------
# Use only the 53 FL-training samples.
# The 14 global test samples remain untouched.
# ------------------------------------------------------------

X_fl = X_train.copy()
y_fl = y_train.copy()

print("FL samples:", len(X_fl))
print("Global test samples:", len(X_test))

# ------------------------------------------------------------
# Create a heterogeneous but reasonably balanced partition.
#
# We first sort by target and distribute samples using a
# shuffled round-robin scheme. This gives every client a
# somewhat different target distribution without creating
# extremely tiny clients.
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

train_indices = np.array(X_fl.index)
shuffled_indices = rng.permutation(train_indices)

# Round-robin assignment
client_assignments = {
    i: [] for i in range(1, N_CLIENTS + 1)
}

for pos, idx in enumerate(shuffled_indices):
    client_id = (pos % N_CLIENTS) + 1
    client_assignments[client_id].append(idx)

# ------------------------------------------------------------
# Verify partition
# ------------------------------------------------------------

print("\nClient distribution:")
print("-" * 60)

for client_id, indices in client_assignments.items():
    client_y = y_fl.loc[indices]

    print(
        f"Client {client_id}: "
        f"{len(indices):2d} trips | "
        f"mean={client_y.mean():.5f} | "
        f"std={client_y.std():.5f} | "
        f"min={client_y.min():.5f} | "
        f"max={client_y.max():.5f}"
    )

all_client_indices = [
    idx
    for indices in client_assignments.values()
    for idx in indices
]

print("\nPartition checks:")
print("Total client samples :", len(all_client_indices))
print("Expected             :", len(X_fl))
print("Unique samples       :", len(set(all_client_indices)))
print("Duplicates           :", len(all_client_indices) - len(set(all_client_indices)))

print(
    "Test leakage         :",
    len(set(all_client_indices) & set(X_test.index))
)

FL samples: 53
Global test samples: 14

Client distribution:
------------------------------------------------------------
Client 1: 11 trips | mean=0.16726 | std=0.02757 | min=0.12288 | max=0.21002
Client 2: 11 trips | mean=0.16755 | std=0.04329 | min=0.09883 | max=0.23301
Client 3: 11 trips | mean=0.17219 | std=0.02692 | min=0.13438 | max=0.21739
Client 4: 10 trips | mean=0.17944 | std=0.02775 | min=0.13133 | max=0.21053
Client 5: 10 trips | mean=0.16342 | std=0.03859 | min=0.11890 | max=0.21784

Partition checks:
Total client samples : 53
Expected             : 53
Unique samples       : 53
Duplicates           : 0
Test leakage         : 0


In [85]:
# ============================================================
# FEDPER: LOCAL 80:20 TRAIN / VALIDATION SPLIT
# ============================================================

client_train = {}
client_val = {}

for client_id, indices in client_assignments.items():

    indices = np.array(indices)

    # Stratification is impossible with such small regression
    # datasets, so we use a deterministic random split.
    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=RANDOM_STATE + client_id
    )

    client_train[client_id] = train_idx
    client_val[client_id] = val_idx


# ------------------------------------------------------------
# Display split
# ------------------------------------------------------------

print("Client local splits:")
print("-" * 60)

for client_id in range(1, N_CLIENTS + 1):

    train_idx = client_train[client_id]
    val_idx = client_val[client_id]

    print(
        f"Client {client_id}: "
        f"{len(train_idx)} train | "
        f"{len(val_idx)} validation"
    )

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

print("\nChecks:")

for client_id in range(1, N_CLIENTS + 1):

    train_set = set(client_train[client_id])
    val_set = set(client_val[client_id])

    print(
        f"Client {client_id}: "
        f"overlap={len(train_set & val_set)}"
    )

print(
    "\nGlobal test leakage:",
    len(
        set(
            idx
            for indices in client_val.values()
            for idx in indices
        )
        & set(X_test.index)
    )
)

Client local splits:
------------------------------------------------------------
Client 1: 8 train | 3 validation
Client 2: 8 train | 3 validation
Client 3: 8 train | 3 validation
Client 4: 8 train | 2 validation
Client 5: 8 train | 2 validation

Checks:
Client 1: overlap=0
Client 2: overlap=0
Client 3: overlap=0
Client 4: overlap=0
Client 5: overlap=0

Global test leakage: 0


In [86]:
# ============================================================
# GLOBAL FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_fl_scaled = pd.DataFrame(
    scaler.fit_transform(X_fl),
    index=X_fl.index,
    columns=X_fl.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    index=X_test.index,
    columns=X_test.columns
)

print("Scaled FL shape   :", X_fl_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)

Scaled FL shape   : (53, 77)
Scaled test shape : (14, 77)


In [87]:
# ============================================================
# FEDPER ELASTIC NET
# ============================================================

def soft_threshold(value, threshold):
    """
    Proximal operator for the L1 penalty.
    """
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_fedper_elastic_net(
    X,
    y,
    shared_coef,
    personal_coef,
    shared_intercept,
    personal_intercept,
    alpha=0.001,
    l1_ratio=0.3,
    learning_rate=0.01,
    epochs=20
):
    """
    Local FedPer-style Elastic Net update.

    Prediction:
        y_hat = X @ (shared + personal)
                + shared_intercept
                + personal_intercept

    Both shared and personal components are updated locally.

    Only the shared component is returned to the server
    for aggregation.
    """

    shared = shared_coef.copy()
    personal = personal_coef.copy()

    shared_b = float(shared_intercept)
    personal_b = float(personal_intercept)

    n = X.shape[0]

    for _ in range(epochs):

        total_coef = shared + personal
        total_intercept = shared_b + personal_b

        prediction = X @ total_coef + total_intercept
        error = prediction - y

        # Gradient of MSE
        grad = (X.T @ error) / n
        grad_b = error.mean()

        # Elastic Net regularization
        l2_grad = alpha * (1 - l1_ratio) * total_coef

        grad_total = grad + l2_grad

        # ----------------------------------------------------
        # Update shared component
        # ----------------------------------------------------

        shared -= learning_rate * grad_total

        # ----------------------------------------------------
        # Update personal component
        # ----------------------------------------------------

        personal -= learning_rate * grad_total

        # ----------------------------------------------------
        # Proximal L1 step
        # ----------------------------------------------------

        threshold = learning_rate * alpha * l1_ratio

        shared = soft_threshold(shared, threshold)
        personal = soft_threshold(personal, threshold)

        # Biases are not L1 penalized
        shared_b -= learning_rate * grad_b
        personal_b -= learning_rate * grad_b

    return shared, personal, shared_b, personal_b

In [88]:
# ============================================================
# FEDPER TRAINING
# ============================================================

FED_ALPHA = 0.001
FED_L1_RATIO = 0.3
LOCAL_EPOCHS = 20
LEARNING_RATE = 0.01
NUM_ROUNDS = 5

n_features = X_fl_scaled.shape[1]

# ------------------------------------------------------------
# Global shared component
# ------------------------------------------------------------

global_shared_coef = np.zeros(n_features)
global_shared_intercept = 0.0

# ------------------------------------------------------------
# Personal parameters for each client
# ------------------------------------------------------------

personal_params = {}

for client_id in range(1, N_CLIENTS + 1):

    personal_params[client_id] = {
        "coef": np.zeros(n_features),
        "intercept": 0.0
    }


# ------------------------------------------------------------
# FedPer rounds
# ------------------------------------------------------------

history = []

for round_num in range(1, NUM_ROUNDS + 1):

    client_shared = []
    client_weights = []

    local_training_results = []

    for client_id in range(1, N_CLIENTS + 1):

        train_idx = client_train[client_id]

        X_client = X_fl_scaled.loc[train_idx].values
        y_client = y_fl.loc[train_idx].values

        personal_coef = personal_params[client_id]["coef"]
        personal_intercept = personal_params[client_id]["intercept"]

        # Local update
        (
            new_shared_coef,
            new_personal_coef,
            new_shared_intercept,
            new_personal_intercept
        ) = local_fedper_elastic_net(
            X_client,
            y_client,
            global_shared_coef,
            personal_coef,
            global_shared_intercept,
            personal_intercept,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=LEARNING_RATE,
            epochs=LOCAL_EPOCHS
        )

        # Save client's personalized parameters
        personal_params[client_id] = {
            "coef": new_personal_coef,
            "intercept": new_personal_intercept
        }

        # Send only shared parameters to server
        client_shared.append(
            (
                new_shared_coef,
                new_shared_intercept
            )
        )

        client_weights.append(len(train_idx))

        # Local training performance
        train_pred = (
            X_client
            @ (new_shared_coef + new_personal_coef)
            + new_shared_intercept
            + new_personal_intercept
        )

        local_training_results.append({
            "Client": client_id,
            "Train MAE": mean_absolute_error(
                y_client,
                train_pred
            )
        })

    # --------------------------------------------------------
    # SERVER AGGREGATION
    # --------------------------------------------------------

    total_weight = sum(client_weights)

    global_shared_coef = sum(
        coef * weight
        for (coef, _), weight
        in zip(client_shared, client_weights)
    ) / total_weight

    global_shared_intercept = sum(
        intercept * weight
        for (_, intercept), weight
        in zip(client_shared, client_weights)
    ) / total_weight

    mean_train_mae = np.mean([
        x["Train MAE"]
        for x in local_training_results
    ])

    history.append({
        "Round": round_num,
        "Mean Client Train MAE": mean_train_mae
    })

    print(
        f"Round {round_num:2d} | "
        f"Mean client train MAE = {mean_train_mae:.6f}"
    )

Round  1 | Mean client train MAE = 0.035175
Round  2 | Mean client train MAE = 0.020676
Round  3 | Mean client train MAE = 0.013203
Round  4 | Mean client train MAE = 0.009077
Round  5 | Mean client train MAE = 0.006681


In [89]:
# ============================================================
# FEDPER PERSONALIZED CLIENT VALIDATION
# ============================================================

client_results = []

all_val_actual = []
all_val_pred = []

for client_id in range(1, N_CLIENTS + 1):

    val_idx = client_val[client_id]

    X_val = X_fl_scaled.loc[val_idx].values
    y_val = y_fl.loc[val_idx].values

    personal_coef = personal_params[client_id]["coef"]
    personal_intercept = personal_params[client_id]["intercept"]

    # Personalized prediction
    prediction = (
        X_val
        @ (global_shared_coef + personal_coef)
        + global_shared_intercept
        + personal_intercept
    )

    mae = mean_absolute_error(y_val, prediction)

    rmse = np.sqrt(
        mean_squared_error(y_val, prediction)
    )

    # R² is unreliable with only 2-3 samples, but calculate it
    # where mathematically possible.
    if len(y_val) >= 2 and np.std(y_val) > 0:
        r2 = r2_score(y_val, prediction)
    else:
        r2 = np.nan

    client_results.append({
        "Client": client_id,
        "Validation Samples": len(y_val),
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

    all_val_actual.extend(y_val)
    all_val_pred.extend(prediction)

client_results_df = pd.DataFrame(client_results)

print("\n")
print("=" * 70)
print("FEDPER PERSONALIZED CLIENT VALIDATION")
print("=" * 70)

print(
    client_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



FEDPER PERSONALIZED CLIENT VALIDATION
 Client  Validation Samples      MAE     RMSE         R²
      1                   3 0.121697 0.130389 -29.424143
      2                   3 0.109109 0.115263  -6.643325
      3                   3 0.180567 0.183666 -75.290761
      4                   2 0.085102 0.098100 -24.636452
      5                   2 0.048617 0.063345 -91.480743


In [90]:
# ============================================================
# FEDPER OVERALL PERSONALIZED PERFORMANCE
# ============================================================

overall_mae = mean_absolute_error(
    all_val_actual,
    all_val_pred
)

overall_rmse = np.sqrt(
    mean_squared_error(
        all_val_actual,
        all_val_pred
    )
)

overall_r2 = r2_score(
    all_val_actual,
    all_val_pred
)

mean_client_mae = client_results_df["MAE"].mean()
std_client_mae = client_results_df["MAE"].std()
worst_client_mae = client_results_df["MAE"].max()

mean_client_rmse = client_results_df["RMSE"].mean()

valid_r2 = client_results_df["R²"].dropna()

print("\n")
print("=" * 70)
print("FEDPER PERSONALIZED RESULTS")
print("=" * 70)

print(f"Overall MAE          : {overall_mae:.6f} kWh/km")
print(f"Overall RMSE         : {overall_rmse:.6f} kWh/km")
print(f"Overall R²           : {overall_r2:.6f}")

print()
print(f"Mean Client MAE      : {mean_client_mae:.6f}")
print(f"Std Client MAE       : {std_client_mae:.6f}")
print(f"Worst Client MAE     : {worst_client_mae:.6f}")
print(f"Mean Client RMSE     : {mean_client_rmse:.6f}")

if len(valid_r2) > 0:
    print(f"Mean Client R²       : {valid_r2.mean():.6f}")



FEDPER PERSONALIZED RESULTS
Overall MAE          : 0.115504 kWh/km
Overall RMSE         : 0.129891 kWh/km
Overall R²           : -14.536990

Mean Client MAE      : 0.109018
Std Client MAE       : 0.048735
Worst Client MAE     : 0.180567
Mean Client RMSE     : 0.118153
Mean Client R²       : -45.495085


In [92]:
# ============================================================
# FEDPER OVERALL PERSONALIZED PERFORMANCE
# ============================================================

overall_mae = mean_absolute_error(
    all_val_actual,
    all_val_pred
)

overall_rmse = np.sqrt(
    mean_squared_error(
        all_val_actual,
        all_val_pred
    )
)

overall_r2 = r2_score(
    all_val_actual,
    all_val_pred
)

mean_client_mae = client_results_df["MAE"].mean()
std_client_mae = client_results_df["MAE"].std()
worst_client_mae = client_results_df["MAE"].max()

mean_client_rmse = client_results_df["RMSE"].mean()

valid_r2 = client_results_df["R²"].dropna()

print("\n")
print("=" * 70)
print("FEDPER PERSONALIZED RESULTS")
print("=" * 70)

print(f"Overall MAE          : {overall_mae:.6f} kWh/km")
print(f"Overall RMSE         : {overall_rmse:.6f} kWh/km")
print(f"Overall R²           : {overall_r2:.6f}")

print()
print(f"Mean Client MAE      : {mean_client_mae:.6f}")
print(f"Std Client MAE       : {std_client_mae:.6f}")
print(f"Worst Client MAE     : {worst_client_mae:.6f}")
print(f"Mean Client RMSE     : {mean_client_rmse:.6f}")

if len(valid_r2) > 0:
    print(f"Mean Client R²       : {valid_r2.mean():.6f}")



FEDPER PERSONALIZED RESULTS
Overall MAE          : 0.115504 kWh/km
Overall RMSE         : 0.129891 kWh/km
Overall R²           : -14.536990

Mean Client MAE      : 0.109018
Std Client MAE       : 0.048735
Worst Client MAE     : 0.180567
Mean Client RMSE     : 0.118153
Mean Client R²       : -45.495085


In [93]:
# ============================================================
# FEDPER ELASTIC NET
# CLIENT-PERSONALIZED MODEL
# ============================================================

def soft_threshold(value, threshold):
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_fedper_elastic_net(
    X,
    y,
    shared_coef,
    personal_coef,
    shared_intercept,
    personal_intercept,
    alpha=0.001,
    l1_ratio=0.3,
    learning_rate=0.01,
    epochs=20
):

    shared = shared_coef.copy()
    personal = personal_coef.copy()

    shared_b = float(shared_intercept)
    personal_b = float(personal_intercept)

    n = X.shape[0]

    for _ in range(epochs):

        # ----------------------------------------------------
        # CLIENT'S ACTUAL PERSONALIZED MODEL
        # ----------------------------------------------------

        total_coef = shared + personal
        total_intercept = shared_b + personal_b

        prediction = (
            X @ total_coef
            + total_intercept
        )

        error = prediction - y

        # ----------------------------------------------------
        # Gradient
        # ----------------------------------------------------

        grad = (X.T @ error) / n
        grad_b = error.mean()

        # L2 part of Elastic Net
        grad += (
            alpha
            * (1 - l1_ratio)
            * total_coef
        )

        # ----------------------------------------------------
        # Update BOTH shared and personal components
        # ----------------------------------------------------

        shared -= learning_rate * grad
        personal -= learning_rate * grad

        shared_b -= learning_rate * grad_b
        personal_b -= learning_rate * grad_b

        # ----------------------------------------------------
        # L1 proximal step
        # ----------------------------------------------------

        threshold = (
            learning_rate
            * alpha
            * l1_ratio
        )

        shared = soft_threshold(
            shared,
            threshold
        )

        personal = soft_threshold(
            personal,
            threshold
        )

    return (
        shared,
        personal,
        shared_b,
        personal_b
    )

In [94]:
# ============================================================
# FEDPER TRAINING
# ============================================================

NUM_ROUNDS = 5
LOCAL_EPOCHS = 20
LEARNING_RATE = 0.01
FED_ALPHA = 0.001
FED_L1_RATIO = 0.3

n_features = X_fl_scaled.shape[1]

# ------------------------------------------------------------
# SERVER'S SHARED MODEL
# ------------------------------------------------------------

global_shared_coef = np.zeros(n_features)
global_shared_intercept = 0.0


# ------------------------------------------------------------
# EACH CLIENT'S PERSONAL MODEL
# ------------------------------------------------------------

personal_params = {}

for client_id in range(1, N_CLIENTS + 1):

    personal_params[client_id] = {
        "coef": np.zeros(n_features),
        "intercept": 0.0
    }


# ============================================================
# FEDERATED ROUNDS
# ============================================================

for round_num in range(1, NUM_ROUNDS + 1):

    print(f"\n{'=' * 70}")
    print(f"FEDPER ROUND {round_num}")
    print(f"{'=' * 70}")

    client_shared_models = []
    client_weights = []

    for client_id in range(1, N_CLIENTS + 1):

        # ----------------------------------------------------
        # CLIENT'S OWN 80% TRAINING DATA
        # ----------------------------------------------------

        train_idx = client_train[client_id]

        X_client = X_fl_scaled.loc[
            train_idx
        ].values

        y_client = y_fl.loc[
            train_idx
        ].values

        # ----------------------------------------------------
        # CLIENT'S EXISTING PERSONAL PARAMETERS
        # ----------------------------------------------------

        personal_coef = (
            personal_params[client_id]["coef"]
        )

        personal_intercept = (
            personal_params[client_id]["intercept"]
        )

        # ----------------------------------------------------
        # LOCAL PERSONALIZED TRAINING
        # ----------------------------------------------------

        (
            new_shared_coef,
            new_personal_coef,
            new_shared_intercept,
            new_personal_intercept
        ) = local_fedper_elastic_net(

            X_client,
            y_client,

            global_shared_coef,
            personal_coef,

            global_shared_intercept,
            personal_intercept,

            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,

            learning_rate=LEARNING_RATE,
            epochs=LOCAL_EPOCHS
        )

        # ----------------------------------------------------
        # CLIENT KEEPS ITS PERSONALIZED PARAMETERS
        # ----------------------------------------------------

        personal_params[client_id] = {

            "coef": new_personal_coef,

            "intercept":
                new_personal_intercept
        }

        # ----------------------------------------------------
        # SEND ONLY SHARED PART TO SERVER
        # ----------------------------------------------------

        client_shared_models.append(
            (
                new_shared_coef,
                new_shared_intercept
            )
        )

        client_weights.append(
            len(train_idx)
        )

        print(
            f"Client {client_id}: "
            f"{len(train_idx)} training trips"
        )

    # ========================================================
    # SERVER AGGREGATION
    # ========================================================

    total_weight = sum(client_weights)

    global_shared_coef = sum(
        coef * weight
        for (coef, _), weight
        in zip(
            client_shared_models,
            client_weights
        )
    ) / total_weight

    global_shared_intercept = sum(
        intercept * weight
        for (_, intercept), weight
        in zip(
            client_shared_models,
            client_weights
        )
    ) / total_weight

    print("\nServer aggregation complete.")


FEDPER ROUND 1
Client 1: 8 training trips
Client 2: 8 training trips
Client 3: 8 training trips
Client 4: 8 training trips
Client 5: 8 training trips

Server aggregation complete.

FEDPER ROUND 2
Client 1: 8 training trips
Client 2: 8 training trips
Client 3: 8 training trips
Client 4: 8 training trips
Client 5: 8 training trips

Server aggregation complete.

FEDPER ROUND 3
Client 1: 8 training trips
Client 2: 8 training trips
Client 3: 8 training trips
Client 4: 8 training trips
Client 5: 8 training trips

Server aggregation complete.

FEDPER ROUND 4
Client 1: 8 training trips
Client 2: 8 training trips
Client 3: 8 training trips
Client 4: 8 training trips
Client 5: 8 training trips

Server aggregation complete.

FEDPER ROUND 5
Client 1: 8 training trips
Client 2: 8 training trips
Client 3: 8 training trips
Client 4: 8 training trips
Client 5: 8 training trips

Server aggregation complete.


In [95]:
# ============================================================
# FEDPER: PERSONALIZED CLIENT TEST
# ============================================================

results = []

all_actual = []
all_predictions = []

for client_id in range(1, N_CLIENTS + 1):

    # --------------------------------------------------------
    # CLIENT'S UNSEEN 20%
    # --------------------------------------------------------

    test_idx = client_val[client_id]

    X_client_test = X_fl_scaled.loc[
        test_idx
    ].values

    y_client_test = y_fl.loc[
        test_idx
    ].values

    # --------------------------------------------------------
    # CLIENT'S OWN PERSONAL PARAMETERS
    # --------------------------------------------------------

    personal_coef = (
        personal_params[client_id]["coef"]
    )

    personal_intercept = (
        personal_params[client_id]["intercept"]
    )

    # --------------------------------------------------------
    # COMPLETE PERSONALIZED MODEL
    #
    # shared component
    #       +
    # personal component
    # --------------------------------------------------------

    personalized_coef = (
        global_shared_coef
        + personal_coef
    )

    personalized_intercept = (
        global_shared_intercept
        + personal_intercept
    )

    # --------------------------------------------------------
    # PREDICT UNSEEN DATA FROM SAME CLIENT
    # --------------------------------------------------------

    predictions = (
        X_client_test
        @ personalized_coef
        + personalized_intercept
    )

    mae = mean_absolute_error(
        y_client_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_client_test,
            predictions
        )
    )

    if (
        len(y_client_test) >= 2
        and np.std(y_client_test) > 0
    ):
        r2 = r2_score(
            y_client_test,
            predictions
        )
    else:
        r2 = np.nan

    results.append({
        "Client": client_id,
        "Test Samples": len(test_idx),
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

    all_actual.extend(y_client_test)
    all_predictions.extend(predictions)


# ============================================================
# RESULTS
# ============================================================

fedper_results = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("FEDPER PERSONALIZED CLIENT RESULTS")
print("=" * 70)

print(
    fedper_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



FEDPER PERSONALIZED CLIENT RESULTS
 Client  Test Samples      MAE     RMSE         R²
      1             3 0.121697 0.130389 -29.424143
      2             3 0.109109 0.115263  -6.643325
      3             3 0.180567 0.183666 -75.290761
      4             2 0.085102 0.098100 -24.636452
      5             2 0.048617 0.063345 -91.480743


In [96]:
# ============================================================
# OVERALL PERSONALIZED PERFORMANCE
# ============================================================

overall_mae = mean_absolute_error(
    all_actual,
    all_predictions
)

overall_rmse = np.sqrt(
    mean_squared_error(
        all_actual,
        all_predictions
    )
)

overall_r2 = r2_score(
    all_actual,
    all_predictions
)

print("\n")
print("=" * 70)
print("FEDPER PERSONALIZED MODEL - UNSEEN CLIENT DATA")
print("=" * 70)

print(f"Overall MAE  : {overall_mae:.6f} kWh/km")
print(f"Overall RMSE : {overall_rmse:.6f} kWh/km")
print(f"Overall R²   : {overall_r2:.6f}")

print(
    f"\nMean client MAE  : "
    f"{fedper_results['MAE'].mean():.6f}"
)

print(
    f"Worst client MAE : "
    f"{fedper_results['MAE'].max():.6f}"
)



FEDPER PERSONALIZED MODEL - UNSEEN CLIENT DATA
Overall MAE  : 0.115504 kWh/km
Overall RMSE : 0.129891 kWh/km
Overall R²   : -14.536990

Mean client MAE  : 0.109018
Worst client MAE : 0.180567


In [97]:
# ============================================================
# FEDPER ELASTIC NET
# SHARED + PERSONALIZED PARAMETERS
# ============================================================

def soft_threshold(value, threshold):
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_fedper_elastic_net(
    X,
    y,
    shared_coef,
    personal_coef,
    shared_intercept,
    personal_intercept,
    alpha=0.001,
    l1_ratio=0.3,
    personal_reg=0.01,
    learning_rate=0.01,
    epochs=20
):
    """
    Personalized Elastic Net.

    Prediction:

        y_hat = X @ (shared_coef + personal_coef)
                + shared_intercept
                + personal_intercept

    shared_coef:
        Federated/global component.

    personal_coef:
        Client-specific component.

    The personal component is additionally regularized toward zero
    to prevent a tiny client from simply memorizing its local data.
    """

    shared = shared_coef.copy()
    personal = personal_coef.copy()

    shared_b = float(shared_intercept)
    personal_b = float(personal_intercept)

    n = X.shape[0]

    for _ in range(epochs):

        total_coef = shared + personal
        total_intercept = shared_b + personal_b

        prediction = (
            X @ total_coef
            + total_intercept
        )

        error = prediction - y

        # ----------------------------------------------------
        # Data-loss gradient
        # ----------------------------------------------------

        grad = (X.T @ error) / n
        grad_b = error.mean()

        # ----------------------------------------------------
        # L2 regularization
        # ----------------------------------------------------

        grad_shared = (
            grad
            + alpha * (1 - l1_ratio) * shared
        )

        grad_personal = (
            grad
            + personal_reg * personal
        )

        # ----------------------------------------------------
        # Update shared component
        # ----------------------------------------------------

        shared -= learning_rate * grad_shared

        # ----------------------------------------------------
        # Update personal component
        # ----------------------------------------------------

        personal -= learning_rate * grad_personal

        # ----------------------------------------------------
        # L1 proximal step
        # ----------------------------------------------------

        threshold = (
            learning_rate
            * alpha
            * l1_ratio
        )

        shared = soft_threshold(
            shared,
            threshold
        )

        personal = soft_threshold(
            personal,
            threshold
        )

        # ----------------------------------------------------
        # Bias updates
        # ----------------------------------------------------

        shared_b -= learning_rate * grad_b

        personal_b -= learning_rate * (
            grad_b
            + personal_reg * personal_b
        )

    return (
        shared,
        personal,
        shared_b,
        personal_b
    )

In [98]:
# ============================================================
# REPEATED FEDPER PERSONALIZATION EXPERIMENT
# ============================================================

N_REPEATS = 10
N_CLIENTS = 5

FED_ALPHA = 0.001
FED_L1_RATIO = 0.3

LOCAL_EPOCHS = 20
LEARNING_RATE = 0.01
PERSONAL_REG = 0.01
NUM_ROUNDS = 5

all_results = []

for repeat in range(N_REPEATS):

    print(f"\n{'=' * 70}")
    print(f"REPEAT {repeat + 1}/{N_REPEATS}")
    print(f"{'=' * 70}")

    rng = np.random.default_rng(1000 + repeat)

    # --------------------------------------------------------
    # 1. Create heterogeneous client partition
    # --------------------------------------------------------

    indices = np.array(X_fl.index)
    shuffled = rng.permutation(indices)

    client_assignments = {
        client_id: []
        for client_id in range(1, N_CLIENTS + 1)
    }

    for pos, idx in enumerate(shuffled):
        client_id = (pos % N_CLIENTS) + 1
        client_assignments[client_id].append(idx)

    # --------------------------------------------------------
    # 2. 80:20 split inside every client
    # --------------------------------------------------------

    client_train_repeat = {}
    client_test_repeat = {}

    for client_id in range(1, N_CLIENTS + 1):

        client_indices = np.array(
            client_assignments[client_id]
        )

        train_idx, test_idx = train_test_split(
            client_indices,
            test_size=0.20,
            random_state=5000 + repeat * 10 + client_id
        )

        client_train_repeat[client_id] = train_idx
        client_test_repeat[client_id] = test_idx

    # --------------------------------------------------------
    # 3. Initialize shared parameters
    # --------------------------------------------------------

    n_features = X_fl_scaled.shape[1]

    global_shared_coef = np.zeros(n_features)
    global_shared_intercept = 0.0

    # --------------------------------------------------------
    # 4. Initialize each client's personal parameters
    # --------------------------------------------------------

    personal_params = {}

    for client_id in range(1, N_CLIENTS + 1):

        personal_params[client_id] = {
            "coef": np.zeros(n_features),
            "intercept": 0.0
        }

    # --------------------------------------------------------
    # 5. Federated training
    # --------------------------------------------------------

    for round_num in range(NUM_ROUNDS):

        client_shared_models = []
        client_weights = []

        for client_id in range(1, N_CLIENTS + 1):

            train_idx = client_train_repeat[client_id]

            X_client = X_fl_scaled.loc[
                train_idx
            ].values

            y_client = y_fl.loc[
                train_idx
            ].values

            personal_coef = personal_params[
                client_id
            ]["coef"]

            personal_intercept = personal_params[
                client_id
            ]["intercept"]

            (
                new_shared_coef,
                new_personal_coef,
                new_shared_intercept,
                new_personal_intercept
            ) = local_fedper_elastic_net(

                X_client,
                y_client,

                global_shared_coef,
                personal_coef,

                global_shared_intercept,
                personal_intercept,

                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,

                personal_reg=PERSONAL_REG,

                learning_rate=LEARNING_RATE,
                epochs=LOCAL_EPOCHS
            )

            # Client keeps its own personal parameters
            personal_params[client_id] = {
                "coef": new_personal_coef,
                "intercept": new_personal_intercept
            }

            # Only shared parameters go to server
            client_shared_models.append(
                (
                    new_shared_coef,
                    new_shared_intercept
                )
            )

            client_weights.append(
                len(train_idx)
            )

        # ----------------------------------------------------
        # SERVER AGGREGATION
        # ----------------------------------------------------

        total_weight = sum(client_weights)

        global_shared_coef = sum(
            coef * weight
            for (coef, _), weight
            in zip(
                client_shared_models,
                client_weights
            )
        ) / total_weight

        global_shared_intercept = sum(
            intercept * weight
            for (_, intercept), weight
            in zip(
                client_shared_models,
                client_weights
            )
        ) / total_weight

    # --------------------------------------------------------
    # 6. Evaluate each CLIENT'S OWN PERSONALIZED MODEL
    #    on its unseen 20%
    # --------------------------------------------------------

    repeat_actual = []
    repeat_predictions = []

    for client_id in range(1, N_CLIENTS + 1):

        test_idx = client_test_repeat[client_id]

        X_client_test = X_fl_scaled.loc[
            test_idx
        ].values

        y_client_test = y_fl.loc[
            test_idx
        ].values

        personal_coef = personal_params[
            client_id
        ]["coef"]

        personal_intercept = personal_params[
            client_id
        ]["intercept"]

        # Complete personalized client model
        final_coef = (
            global_shared_coef
            + personal_coef
        )

        final_intercept = (
            global_shared_intercept
            + personal_intercept
        )

        predictions = (
            X_client_test @ final_coef
            + final_intercept
        )

        mae = mean_absolute_error(
            y_client_test,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_client_test,
                predictions
            )
        )

        repeat_actual.extend(y_client_test)
        repeat_predictions.extend(predictions)

        all_results.append({
            "Repeat": repeat + 1,
            "Client": client_id,
            "Test Samples": len(test_idx),
            "MAE": mae,
            "RMSE": rmse
        })

    # --------------------------------------------------------
    # Overall result for this repetition
    # --------------------------------------------------------

    repeat_mae = mean_absolute_error(
        repeat_actual,
        repeat_predictions
    )

    repeat_rmse = np.sqrt(
        mean_squared_error(
            repeat_actual,
            repeat_predictions
        )
    )

    print(
        f"Repeat {repeat + 1}: "
        f"MAE={repeat_mae:.6f}, "
        f"RMSE={repeat_rmse:.6f}"
    )


REPEAT 1/10
Repeat 1: MAE=0.180152, RMSE=0.221911

REPEAT 2/10
Repeat 2: MAE=0.132383, RMSE=0.155130

REPEAT 3/10
Repeat 3: MAE=0.161752, RMSE=0.188812

REPEAT 4/10
Repeat 4: MAE=0.170644, RMSE=0.204987

REPEAT 5/10
Repeat 5: MAE=0.143850, RMSE=0.173786

REPEAT 6/10
Repeat 6: MAE=0.186551, RMSE=0.247376

REPEAT 7/10
Repeat 7: MAE=0.186343, RMSE=0.216323

REPEAT 8/10
Repeat 8: MAE=0.179671, RMSE=0.210541

REPEAT 9/10
Repeat 9: MAE=0.156908, RMSE=0.190215

REPEAT 10/10
Repeat 10: MAE=0.154100, RMSE=0.175615


In [99]:
# ============================================================
# FEDPER REPEATED EVALUATION SUMMARY
# ============================================================

fedper_repeated_results = pd.DataFrame(
    all_results
)

print("\n")
print("=" * 70)
print("FEDPER: REPEATED PERSONALIZED CLIENT EVALUATION")
print("=" * 70)

print(
    fedper_repeated_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n")
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    f"Mean Client MAE : "
    f"{fedper_repeated_results['MAE'].mean():.6f}"
)

print(
    f"Std Client MAE  : "
    f"{fedper_repeated_results['MAE'].std():.6f}"
)

print(
    f"Mean Client RMSE: "
    f"{fedper_repeated_results['RMSE'].mean():.6f}"
)

print(
    f"Worst Client MAE: "
    f"{fedper_repeated_results['MAE'].max():.6f}"
)



FEDPER: REPEATED PERSONALIZED CLIENT EVALUATION
 Repeat  Client  Test Samples      MAE     RMSE
      1       1             3 0.342016 0.343887
      1       2             3 0.095471 0.099340
      1       3             3 0.189128 0.238549
      1       4             2 0.161218 0.188746
      1       5             2 0.069852 0.083160
      2       1             3 0.130426 0.172511
      2       2             3 0.193279 0.204961
      2       3             3 0.122783 0.142185
      2       4             2 0.092565 0.092574
      2       5             2 0.098191 0.099381
      3       1             3 0.119699 0.123564
      3       2             3 0.140608 0.158695
      3       3             3 0.268553 0.274438
      3       4             2 0.028887 0.035355
      3       5             2 0.229208 0.238375
      4       1             3 0.129387 0.147255
      4       2             3 0.157590 0.166146
      4       3             3 0.184139 0.202874
      4       4             2 0.044355

In [100]:
# ============================================================
# CLIENT-LEVEL FEDPER PERFORMANCE
# ============================================================

client_summary = (
    fedper_repeated_results
    .groupby("Client")
    .agg(
        Mean_MAE=("MAE", "mean"),
        Std_MAE=("MAE", "std"),
        Mean_RMSE=("RMSE", "mean"),
        Worst_MAE=("MAE", "max")
    )
    .reset_index()
)

print("\n")
print("=" * 70)
print("FEDPER PERFORMANCE BY CLIENT")
print("=" * 70)

print(
    client_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



FEDPER PERFORMANCE BY CLIENT
 Client  Mean_MAE  Std_MAE  Mean_RMSE  Worst_MAE
      1  0.172301 0.084970   0.186886   0.342016
      2  0.164405 0.036096   0.189761   0.214049
      3  0.187459 0.092590   0.211700   0.396085
      4  0.130293 0.070956   0.134916   0.239891
      5  0.157489 0.094463   0.174226   0.358154


In [101]:
# ============================================================
# FEDPER vs FEDAVG
# ============================================================

fedper_mean_mae = (
    fedper_repeated_results["MAE"].mean()
)

fedper_mean_rmse = (
    fedper_repeated_results["RMSE"].mean()
)

print("\n")
print("=" * 70)
print("FEDPER vs FEDAVG")
print("=" * 70)

print(f"FedAvg MAE       : {fedavg_mae:.6f}")
print(f"FedPer MAE       : {fedper_mean_mae:.6f}")

print()

improvement = (
    (fedavg_mae - fedper_mean_mae)
    / fedavg_mae
    * 100
)

print(
    f"FedPer MAE change: {improvement:.2f}%"
)



FEDPER vs FEDAVG
FedAvg MAE       : 0.013109
FedPer MAE       : 0.162389

FedPer MAE change: -1138.75%


In [102]:
# ============================================================
# CORRECTED PERSONALIZED ELASTIC NET
# ============================================================

def soft_threshold(value, threshold):
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_personalized_elastic_net(
    X,
    y,
    global_coef,
    personal_delta,
    global_intercept,
    personal_intercept,
    alpha=0.001,
    l1_ratio=0.3,
    personal_reg=0.05,
    learning_rate=0.005,
    personal_learning_rate=0.005,
    epochs=20
):
    """
    Personalized Elastic Net.

    Model:

        y_hat = X @ (global_coef + personal_delta)
                + global_intercept
                + personal_intercept

    global_coef:
        Shared federated parameters.

    personal_delta:
        Client-specific deviation from the shared model.

    The personal component is strongly regularized so that it
    learns only client-specific information rather than simply
    duplicating the global model.
    """

    shared = global_coef.copy()
    personal = personal_delta.copy()

    shared_b = float(global_intercept)
    personal_b = float(personal_intercept)

    n = X.shape[0]

    for _ in range(epochs):

        # ----------------------------------------------------
        # Complete personalized client model
        # ----------------------------------------------------

        total_coef = shared + personal
        total_intercept = shared_b + personal_b

        prediction = (
            X @ total_coef
            + total_intercept
        )

        error = prediction - y

        # ----------------------------------------------------
        # Gradient of prediction loss
        # ----------------------------------------------------

        data_grad = (X.T @ error) / n
        data_grad_b = error.mean()

        # ----------------------------------------------------
        # Shared component
        #
        # Elastic Net regularization
        # ----------------------------------------------------

        shared_grad = (
            data_grad
            + alpha * (1 - l1_ratio) * shared
        )

        shared -= (
            learning_rate
            * shared_grad
        )

        # ----------------------------------------------------
        # Personal component
        #
        # Personal regularization is stronger.
        # This forces personal parameters to remain a
        # relatively small correction to the global model.
        # ----------------------------------------------------

        personal_grad = (
            data_grad
            + personal_reg * personal
        )

        personal -= (
            personal_learning_rate
            * personal_grad
        )

        # ----------------------------------------------------
        # L1 penalty on shared parameters
        # ----------------------------------------------------

        threshold = (
            learning_rate
            * alpha
            * l1_ratio
        )

        shared = soft_threshold(
            shared,
            threshold
        )

        # Personal component gets a much smaller L1 penalty.
        personal_threshold = (
            personal_learning_rate
            * alpha
            * 0.5
            * l1_ratio
        )

        personal = soft_threshold(
            personal,
            personal_threshold
        )

        # ----------------------------------------------------
        # Biases
        # ----------------------------------------------------

        shared_b -= (
            learning_rate
            * data_grad_b
        )

        personal_b -= (
            personal_learning_rate
            * (
                data_grad_b
                + personal_reg * personal_b
            )
        )

    return (
        shared,
        personal,
        shared_b,
        personal_b
    )

In [103]:
# ============================================================
# PERSONALIZATION SANITY CHECK
# ============================================================

print("\n")
print("=" * 70)
print("PERSONALIZATION SANITY CHECK")
print("=" * 70)

for client_id in range(1, N_CLIENTS + 1):

    personal_coef = personal_params[
        client_id
    ]["coef"]

    personal_norm = np.linalg.norm(
        personal_coef
    )

    global_norm = np.linalg.norm(
        global_coef
    )

    print(
        f"Client {client_id}: "
        f"personal norm = {personal_norm:.6f} | "
        f"global norm = {global_norm:.6f} | "
        f"ratio = {personal_norm / (global_norm + 1e-12):.6f}"
    )



PERSONALIZATION SANITY CHECK
Client 1: personal norm = 0.048084 | global norm = 0.016804 | ratio = 2.861460
Client 2: personal norm = 0.049945 | global norm = 0.016804 | ratio = 2.972205
Client 3: personal norm = 0.038556 | global norm = 0.016804 | ratio = 2.294443
Client 4: personal norm = 0.047877 | global norm = 0.016804 | ratio = 2.849129
Client 5: personal norm = 0.057204 | global norm = 0.016804 | ratio = 3.404152


In [104]:
# ============================================================
# FEDPER PERSONALIZED VALIDATION
# ============================================================

fedper_sanity_results = []

all_actual = []
all_predictions = []

for client_id in range(1, N_CLIENTS + 1):

    test_idx = client_test_repeat[client_id]

    X_client_test = X_fl_scaled.loc[
        test_idx
    ].values

    y_client_test = y_fl.loc[
        test_idx
    ].values

    personal_coef = personal_params[
        client_id
    ]["coef"]

    personal_intercept = personal_params[
        client_id
    ]["intercept"]

    # --------------------------------------------------------
    # THIS IS THE CLIENT'S FINAL PERSONALIZED MODEL
    # --------------------------------------------------------

    final_coef = (
        global_coef
        + personal_coef
    )

    final_intercept = (
        global_intercept
        + personal_intercept
    )

    predictions = (
        X_client_test
        @ final_coef
        + final_intercept
    )

    mae = mean_absolute_error(
        y_client_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_client_test,
            predictions
        )
    )

    fedper_sanity_results.append({
        "Client": client_id,
        "Test Samples": len(test_idx),
        "MAE": mae,
        "RMSE": rmse
    })

    all_actual.extend(y_client_test)
    all_predictions.extend(predictions)


fedper_sanity_df = pd.DataFrame(
    fedper_sanity_results
)

print("\n")
print("=" * 70)
print("FEDPER PERSONALIZED CLIENT RESULTS")
print("=" * 70)

print(
    fedper_sanity_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

overall_mae = mean_absolute_error(
    all_actual,
    all_predictions
)

overall_rmse = np.sqrt(
    mean_squared_error(
        all_actual,
        all_predictions
    )

)

print("\nOverall personalized MAE :", f"{overall_mae:.6f}")
print("Overall personalized RMSE:", f"{overall_rmse:.6f}")



FEDPER PERSONALIZED CLIENT RESULTS
 Client  Test Samples      MAE     RMSE
      1             3 0.032583 0.039806
      2             3 0.083146 0.099740
      3             3 0.048964 0.061231
      4             2 0.030237 0.033236
      5             2 0.074823 0.081052

Overall personalized MAE : 0.054169
Overall personalized RMSE: 0.068609


### Rejected client construction: one trip per client

One-trip-per-client FedAvg was explored and rejected because a client with one observation cannot meaningfully fit a 77-feature local model. These cells are preserved as a diagnostic experiment only; they do not define the final federated setup.


In [105]:
# ============================================================
# FEDAVG ELASTIC NET
# ONE TRIP = ONE SIMULATED CLIENT
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# The 53 samples in X_train/y_train are the FL clients.
# The 14 samples in X_test/y_test remain untouched.
# ------------------------------------------------------------

client_ids = list(X_train.index)

N_CLIENTS = len(client_ids)

print("=" * 70)
print("FEDERATED CLIENT SETUP")
print("=" * 70)

print(f"Number of FL clients : {N_CLIENTS}")
print(f"Clients per trip     : 1")
print(f"Global test samples  : {len(X_test)}")

# ------------------------------------------------------------
# Each client contains exactly one trip
# ------------------------------------------------------------

clients = {}

for client_id in client_ids:

    clients[client_id] = {
        "X": X_train.loc[[client_id]],
        "y": y_train.loc[[client_id]]
    }

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

total_samples = sum(
    len(client["X"])
    for client in clients.values()
)

unique_samples = len(
    set(
        idx
        for client in clients.values()
        for idx in client["X"].index
    )
)

test_overlap = len(
    set(client_ids) & set(X_test.index)
)

print("\nChecks:")
print(f"Total client samples : {total_samples}")
print(f"Expected             : {len(X_train)}")
print(f"Unique samples       : {unique_samples}")
print(f"Test leakage         : {test_overlap}")

assert total_samples == len(X_train)
assert unique_samples == len(X_train)
assert test_overlap == 0

print("\nClient partition is valid.")

FEDERATED CLIENT SETUP
Number of FL clients : 53
Clients per trip     : 1
Global test samples  : 14

Checks:
Total client samples : 53
Expected             : 53
Unique samples       : 53
Test leakage         : 0

Client partition is valid.


In [106]:
# ============================================================
# GLOBAL FEATURE SCALING
# ============================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    index=X_test.index,
    columns=X_test.columns
)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

X_train_scaled: (53, 77)
X_test_scaled : (14, 77)


In [110]:
# ============================================================
# FEDAVG ELASTIC NET
# ONE TRIP = ONE SIMULATED CLIENT
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 53 training trips = 53 simulated clients
# 14 test trips remain completely untouched
# ------------------------------------------------------------

client_ids = list(X_train.index)

N_CLIENTS = len(client_ids)

clients = {}

for client_id in client_ids:

    clients[client_id] = {
        "X": X_train.loc[[client_id]],
        "y": y_train.loc[[client_id]]
    }

print("=" * 70)
print("FEDERATED CLIENT SETUP")
print("=" * 70)

print(f"Number of FL clients : {N_CLIENTS}")
print(f"Samples per client   : 1 trip")
print(f"Global test samples  : {len(X_test)}")

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

total_client_samples = sum(
    len(client["X"])
    for client in clients.values()
)

unique_client_samples = len(
    set(
        idx
        for client in clients.values()
        for idx in client["X"].index
    )
)

test_overlap = len(
    set(client_ids) & set(X_test.index)
)

print("\nChecks:")
print(f"Total client samples : {total_client_samples}")
print(f"Expected             : {len(X_train)}")
print(f"Unique samples       : {unique_client_samples}")
print(f"Test leakage         : {test_overlap}")

assert total_client_samples == len(X_train)
assert unique_client_samples == len(X_train)
assert test_overlap == 0

print("\nClient partition is valid.")

FEDERATED CLIENT SETUP
Number of FL clients : 53
Samples per client   : 1 trip
Global test samples  : 14

Checks:
Total client samples : 53
Expected             : 53
Unique samples       : 53
Test leakage         : 0

Client partition is valid.


In [111]:
# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    index=X_test.index,
    columns=X_test.columns
)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

X_train_scaled: (53, 77)
X_test_scaled : (14, 77)


In [112]:
# ============================================================
# STABLE LOCAL ELASTIC NET UPDATE
# ============================================================

def soft_threshold(value, threshold):
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


def local_elastic_net_stable(
    X,
    y,
    coef,
    intercept,
    alpha=0.001,
    l1_ratio=0.3,
    learning_rate=0.0001,
    epochs=1,
    grad_clip=1.0
):

    beta = coef.copy()
    b = float(intercept)

    n = X.shape[0]

    for _ in range(epochs):

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        prediction = X @ beta + b

        error = prediction - y

        # ----------------------------------------------------
        # Gradient
        # ----------------------------------------------------

        grad_beta = (
            X.T @ error
        ) / n

        grad_intercept = error.mean()

        # ----------------------------------------------------
        # L2 regularization
        # ----------------------------------------------------

        grad_beta += (
            alpha
            * (1 - l1_ratio)
            * beta
        )

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        grad_norm = np.linalg.norm(
            grad_beta
        )

        if grad_norm > grad_clip:

            grad_beta *= (
                grad_clip
                / (grad_norm + 1e-12)
            )

        grad_intercept = np.clip(
            grad_intercept,
            -grad_clip,
            grad_clip
        )

        # ----------------------------------------------------
        # Parameter update
        # ----------------------------------------------------

        beta -= (
            learning_rate
            * grad_beta
        )

        b -= (
            learning_rate
            * grad_intercept
        )

        # ----------------------------------------------------
        # L1 proximal operator
        # ----------------------------------------------------

        threshold = (
            learning_rate
            * alpha
            * l1_ratio
        )

        beta = soft_threshold(
            beta,
            threshold
        )

    return beta, b

In [113]:
# ============================================================
# FEDAVG TRAINING
# ONE TRIP PER CLIENT
# ============================================================

FED_ALPHA = 0.001
FED_L1_RATIO = 0.3

LOCAL_EPOCHS = 1
LEARNING_RATE = 0.0001
GRAD_CLIP = 1.0

NUM_ROUNDS = 20

n_features = X_train_scaled.shape[1]

# ------------------------------------------------------------
# Initialize global model
# ------------------------------------------------------------

global_coef = np.zeros(n_features)
global_intercept = 0.0

history = []

# ============================================================
# FEDERATED TRAINING
# ============================================================

for round_num in range(1, NUM_ROUNDS + 1):

    client_models = []
    client_weights = []

    # --------------------------------------------------------
    # Every trip acts as an independent client
    # --------------------------------------------------------

    for client_id in client_ids:

        # One trip
        X_client = X_train_scaled.loc[
            [client_id]
        ].values

        y_client = np.array([
            y_train.loc[client_id]
        ])

        # ----------------------------------------------------
        # Local model starts from current global model
        # ----------------------------------------------------

        local_coef, local_intercept = (
            local_elastic_net_stable(

                X_client,
                y_client,

                global_coef,
                global_intercept,

                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,

                learning_rate=LEARNING_RATE,
                epochs=LOCAL_EPOCHS,

                grad_clip=GRAD_CLIP
            )
        )

        # ----------------------------------------------------
        # Store local model
        # ----------------------------------------------------

        client_models.append(
            (
                local_coef,
                local_intercept
            )
        )

        # Every client contains exactly one trip
        client_weights.append(1)

    # ========================================================
    # SERVER: FEDAVG
    # ========================================================

    total_weight = sum(
        client_weights
    )

    global_coef = sum(
        coef * weight
        for (coef, _), weight
        in zip(
            client_models,
            client_weights
        )
    ) / total_weight

    global_intercept = sum(
        intercept * weight
        for (_, intercept), weight
        in zip(
            client_models,
            client_weights
        )
    ) / total_weight

    # ========================================================
    # GLOBAL TRAINING PREDICTIONS
    # ========================================================

    train_predictions = (
        X_train_scaled.values
        @ global_coef
        + global_intercept
    )

    train_mae = mean_absolute_error(
        y_train,
        train_predictions
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_predictions
        )
    )

    train_r2 = r2_score(
        y_train,
        train_predictions
    )

    history.append({
        "Round": round_num,
        "Train MAE": train_mae,
        "Train RMSE": train_rmse,
        "Train R²": train_r2
    })

    print(
        f"Round {round_num:2d} | "
        f"Train MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f} | "
        f"R²={train_r2:.6f}"
    )

Round  1 | Train MAE=0.169901 | RMSE=0.172949 | R²=-27.591764
Round  2 | Train MAE=0.169884 | RMSE=0.172930 | R²=-27.585261
Round  3 | Train MAE=0.169867 | RMSE=0.172910 | R²=-27.578760
Round  4 | Train MAE=0.169850 | RMSE=0.172890 | R²=-27.572262
Round  5 | Train MAE=0.169833 | RMSE=0.172871 | R²=-27.565768
Round  6 | Train MAE=0.169816 | RMSE=0.172851 | R²=-27.559276
Round  7 | Train MAE=0.169799 | RMSE=0.172831 | R²=-27.552789
Round  8 | Train MAE=0.169782 | RMSE=0.172812 | R²=-27.546305
Round  9 | Train MAE=0.169765 | RMSE=0.172792 | R²=-27.539825
Round 10 | Train MAE=0.169748 | RMSE=0.172773 | R²=-27.533348
Round 11 | Train MAE=0.169731 | RMSE=0.172753 | R²=-27.526875
Round 12 | Train MAE=0.169714 | RMSE=0.172733 | R²=-27.520405
Round 13 | Train MAE=0.169697 | RMSE=0.172714 | R²=-27.513939
Round 14 | Train MAE=0.169680 | RMSE=0.172694 | R²=-27.507477
Round 15 | Train MAE=0.169663 | RMSE=0.172675 | R²=-27.501019
Round 16 | Train MAE=0.169646 | RMSE=0.172655 | R²=-27.494564
Round 17

In [114]:
# ============================================================
# FEDAVG - UNTOUCHED GLOBAL TEST
# ============================================================

test_predictions = (
    X_test_scaled.values
    @ global_coef
    + global_intercept
)

fedavg_mae = mean_absolute_error(
    y_test,
    test_predictions
)

fedavg_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

fedavg_r2 = r2_score(
    y_test,
    test_predictions
)

print("\n")
print("=" * 70)
print("FEDAVG ELASTIC NET - GLOBAL TEST")
print("=" * 70)

print(
    f"MAE  : {fedavg_mae:.6f} kWh/km"
)

print(
    f"RMSE : {fedavg_rmse:.6f} kWh/km"
)

print(
    f"R²   : {fedavg_r2:.6f}"
)



FEDAVG ELASTIC NET - GLOBAL TEST
MAE  : 0.149960 kWh/km
RMSE : 0.153619 kWh/km
R²   : -19.899992


In [115]:
# ============================================================
# CENTRALIZED VS FEDAVG
# ============================================================

comparison = pd.DataFrame({

    "Model": [
        "Centralized Elastic Net",
        "FedAvg Elastic Net"
    ],

    "MAE": [
        centralized_mae,
        fedavg_mae
    ],

    "RMSE": [
        centralized_rmse,
        fedavg_rmse
    ],

    "R²": [
        centralized_r2,
        fedavg_r2
    ]
})

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

                  Model      MAE     RMSE         R²
Centralized Elastic Net 0.012569 0.014817   0.805558
     FedAvg Elastic Net 0.149960 0.153619 -19.899992


## 9. Simulated Client Construction

The final setup is a **balanced condition-based simulated client partition**. It uses 35 operating-condition features, standardizes them, reduces them with PCA to 10 components explaining 90.89% of variance, orders trips by the primary condition component, and assigns them with balanced round-robin allocation. The target is not used to construct clients. Five simulated clients contain 10–11 training trips each, while the 14-trip global test set remains untouched.

The nearby K-Means inspection is an exploratory diagnostic only, not the final assignment method and not evidence of real vehicle or driver grouping.


In [116]:
# ============================================================
# CONDITION-BASED SIMULATED CLIENTS
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np

# Use ONLY the global training trips
client_source = X_train.copy()

# ------------------------------------------------------------
# 1. Select condition/context features
# ------------------------------------------------------------

condition_features = [
    # Trip context
    "Distance [km]",
    "Duration [min]",

    # Battery state
    "soc_start",
    "soc_end",

    # Environment
    "ambient_temp_mean",
    "ambient_temp_min",
    "ambient_temp_max",
    "ambient_temp_std",

    # Battery thermal condition
    "battery_temp_mean",
    "battery_temp_min",
    "battery_temp_max",
    "battery_temp_std",

    # Cabin / HVAC demand
    "cabin_temp_mean",
    "cabin_temp_min",
    "cabin_temp_max",
    "cabin_temp_std",

    "heating_power_mean",
    "heating_power_max",
    "heating_power_std",

    "requested_heating_mean",
    "requested_heating_max",

    "aircon_power_mean",
    "aircon_power_max",

    # Driving behaviour
    "speed_mean",
    "speed_std",
    "speed_max",
    "speed_above_50_fraction",

    "acceleration_mean",
    "acceleration_std",
    "acceleration_max",
    "acceleration_min",

    "throttle_mean",
    "throttle_std",

    "torque_mean",
    "torque_std",
]

# Keep only columns that actually exist
available_condition_features = [
    c for c in condition_features
    if c in client_source.columns
]

condition_df = client_source[available_condition_features].copy()

print("Condition features:", len(available_condition_features))
print("Training trips:", len(condition_df))
print("Missing values:", condition_df.isna().sum().sum())

Condition features: 35
Training trips: 53
Missing values: 0


In [117]:
# ============================================================
# 2. STANDARDIZE CONDITION FEATURES
# ============================================================

condition_scaler = StandardScaler()

condition_scaled = condition_scaler.fit_transform(condition_df)


# ============================================================
# 3. K-MEANS CLUSTERING
# ============================================================

N_CLIENTS = 5

kmeans = KMeans(
    n_clusters=N_CLIENTS,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans.fit_predict(condition_scaled)

client_source = client_source.copy()
client_source["simulated_client"] = cluster_labels + 1


# ============================================================
# 4. CLIENT DISTRIBUTION
# ============================================================

client_distribution = (
    client_source["simulated_client"]
    .value_counts()
    .sort_index()
)

print("\nClient distribution:")
print(client_distribution)

print("\nTotal:", client_distribution.sum())
print("Clients:", len(client_distribution))

  File "c:\Users\gayat\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\gayat\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\gayat\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\gayat\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



Client distribution:
simulated_client
1     9
2    28
3     8
4     1
5     7
Name: count, dtype: int64

Total: 53
Clients: 5


In [119]:
# ============================================================
# INSPECT CURRENT K-MEANS CLIENTS
# ============================================================

inspect_cols = [
    "Distance [km]",
    "Duration [min]",
    "soc_start",
    "soc_end",
    "ambient_temp_mean",
    "battery_temp_mean",
    "cabin_temp_mean",
    "heating_power_mean",
    "aircon_power_mean",
    "speed_mean",
    "speed_above_50_fraction",
    "acceleration_mean",
    "torque_mean"
]

inspect_cols = [
    c for c in inspect_cols
    if c in client_source.columns
]

for client_id, group in client_source.groupby("simulated_client"):

    print(f"\n{'='*60}")
    print(f"CLIENT {client_id} | {len(group)} trips")
    print(f"{'='*60}")

    print(
        group[inspect_cols]
        .round(2)
        .to_string()
    )


CLIENT 1 | 9 trips
    Distance [km]  Duration [min]  soc_start  soc_end  ambient_temp_mean  battery_temp_mean  cabin_temp_mean  heating_power_mean  aircon_power_mean  speed_mean  speed_above_50_fraction  acceleration_mean  torque_mean
31          37.21           27.87       0.86     0.60              29.06              23.89            24.02                0.12               0.49       80.10                     0.71               0.02        14.61
3           10.73            6.87       0.75     0.67              24.42              26.82            22.17                0.32               0.30       93.65                     0.81               0.06        17.74
17          12.19           14.68       0.78     0.70              16.75              18.58            25.81                0.69               0.19       49.81                     0.49               0.07        10.63
6           44.34           34.89       0.63     0.34              29.17              30.62            30.63    

In [120]:
# ============================================================
# SHOW ACTUAL TRIPS IN EACH CLIENT
# ============================================================

client_trip_table = pd.DataFrame({
    "Trip": X_train.index,
    "simulated_client": client_source["simulated_client"].values
})

for client_id, group in client_trip_table.groupby("simulated_client"):

    print(f"\nClient {client_id} ({len(group)} trips):")
    print(", ".join(group["Trip"].astype(str).tolist()))


Client 1 (9 trips):
31, 3, 17, 6, 15, 24, 1, 2, 23

Client 2 (28 trips):
65, 36, 51, 57, 47, 45, 32, 49, 56, 34, 50, 69, 68, 53, 66, 52, 39, 46, 58, 59, 42, 37, 55, 61, 60, 41, 63, 54

Client 3 (8 trips):
44, 19, 13, 8, 29, 21, 22, 14

Client 4 (1 trips):
33

Client 5 (7 trips):
7, 27, 26, 11, 10, 18, 20


In [121]:
# ============================================================
# BALANCED CONDITION-BASED CLIENT PARTITION
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Condition features
# ------------------------------------------------------------

condition_df = client_source[available_condition_features].copy()

assert condition_df.isna().sum().sum() == 0

# ------------------------------------------------------------
# 2. Standardize
# ------------------------------------------------------------

condition_scaler = StandardScaler()

condition_scaled = condition_scaler.fit_transform(
    condition_df
)

# ------------------------------------------------------------
# 3. PCA
# ------------------------------------------------------------

pca = PCA(
    n_components=0.90,
    random_state=42
)

condition_pca = pca.fit_transform(condition_scaled)

print("Original condition dimensions:",
      condition_scaled.shape[1])

print("PCA dimensions:",
      condition_pca.shape[1])

print("Explained variance:",
      round(pca.explained_variance_ratio_.sum(), 4))


# ------------------------------------------------------------
# 4. Create a condition ordering
# ------------------------------------------------------------

condition_score = condition_pca[:, 0]

ordered_indices = np.argsort(condition_score)


# ------------------------------------------------------------
# 5. Balanced assignment
# ------------------------------------------------------------

N_CLIENTS = 5

balanced_labels = np.empty(
    len(ordered_indices),
    dtype=int
)

for rank, idx in enumerate(ordered_indices):

    balanced_labels[idx] = rank % N_CLIENTS


# ------------------------------------------------------------
# 6. Store client labels
# ------------------------------------------------------------

client_source = client_source.copy()

client_source["simulated_client"] = (
    balanced_labels + 1
)


# ------------------------------------------------------------
# 7. Check distribution
# ------------------------------------------------------------

client_distribution = (
    client_source["simulated_client"]
    .value_counts()
    .sort_index()
)

print("\nClient distribution:")
print(client_distribution)

print("\nTotal:",
      client_distribution.sum())

Original condition dimensions: 35
PCA dimensions: 10
Explained variance: 0.9089

Client distribution:
simulated_client
1    11
2    11
3    11
4    10
5    10
Name: count, dtype: int64

Total: 53


In [122]:
# ============================================================
# ACTUAL TRIPS PER CLIENT
# ============================================================

client_trip_table = pd.DataFrame({
    "Trip": X_train.index,
    "simulated_client": client_source["simulated_client"].values
})

for client_id, group in client_trip_table.groupby(
    "simulated_client"
):

    print(
        f"\nClient {client_id} "
        f"({len(group)} trips):"
    )

    print(
        ", ".join(
            group["Trip"].astype(str).tolist()
        )
    )


Client 1 (11 trips):
19, 32, 13, 8, 50, 6, 15, 53, 52, 46, 63

Client 2 (11 trips):
7, 45, 69, 33, 66, 59, 42, 23, 22, 20, 14

Client 3 (11 trips):
65, 57, 47, 49, 17, 34, 27, 26, 24, 61, 10

Client 4 (10 trips):
51, 56, 29, 21, 2, 37, 18, 60, 41, 54

Client 5 (10 trips):
36, 44, 31, 3, 68, 11, 39, 58, 1, 55


In [123]:
# ============================================================
# CONDITION PROFILE
# ============================================================

profile_columns = [
    c for c in [
        "Distance [km]",
        "Duration [min]",
        "soc_start",
        "soc_end",
        "ambient_temp_mean",
        "battery_temp_mean",
        "cabin_temp_mean",
        "heating_power_mean",
        "aircon_power_mean",
        "speed_mean",
        "speed_above_50_fraction",
        "acceleration_mean",
        "torque_mean"
    ]
    if c in client_source.columns
]

client_profile = (
    client_source
    .groupby("simulated_client")[profile_columns]
    .mean()
    .round(3)
)

display(client_profile)

,Distance [km],Duration [min],soc_start,soc_end,ambient_temp_mean,battery_temp_mean,cabin_temp_mean,heating_power_mean,aircon_power_mean,speed_mean,speed_above_50_fraction,acceleration_mean,torque_mean
simulated_client,,,,,,,,,,,,,
1,20.240,26.587,0.737,0.590,12.199,14.916,22.071,1.365,0.164,45.786,0.379,0.013,11.713
2,20.102,29.023,0.720,0.574,12.212,14.349,24.178,1.219,0.147,43.166,0.388,0.004,11.145
3,15.929,20.437,0.735,0.622,12.080,14.767,24.373,1.355,0.176,48.984,0.470,0.013,11.752
4,18.147,22.293,0.734,0.586,11.703,15.242,23.438,1.121,0.083,50.016,0.417,0.002,12.142
5,18.596,22.146,0.704,0.564,14.057,16.587,22.140,1.201,0.201,52.091,0.460,0.007,12.219


In [124]:
# ============================================================
# TARGET DISTRIBUTION
# ============================================================

client_target_profile = (
    client_source
    .assign(target=y_train.values)
    .groupby("simulated_client")["target"]
    .agg(
        samples="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .round(4)
)

display(client_target_profile)

,samples,mean,std,min,max
simulated_client,,,,,
1,11,0.1711,0.0339,0.1229,0.2174
2,11,0.1656,0.0385,0.0988,0.2330
3,11,0.1663,0.0299,0.1189,0.2110
4,10,0.1754,0.0377,0.1148,0.2178
5,10,0.1719,0.0273,0.1248,0.2100


In [125]:
# ============================================================
# FINAL CLIENT CONDITION PROFILE
# ============================================================

profile_columns = [
    c for c in [
        "Distance [km]",
        "Duration [min]",
        "soc_start",
        "soc_end",
        "ambient_temp_mean",
        "battery_temp_mean",
        "cabin_temp_mean",
        "heating_power_mean",
        "aircon_power_mean",
        "speed_mean",
        "speed_above_50_fraction",
        "acceleration_mean",
        "torque_mean"
    ]
    if c in client_source.columns
]

client_condition_profile = (
    client_source
    .groupby("simulated_client")[profile_columns]
    .mean()
    .round(3)
)

display(client_condition_profile)

,Distance [km],Duration [min],soc_start,soc_end,ambient_temp_mean,battery_temp_mean,cabin_temp_mean,heating_power_mean,aircon_power_mean,speed_mean,speed_above_50_fraction,acceleration_mean,torque_mean
simulated_client,,,,,,,,,,,,,
1,20.240,26.587,0.737,0.590,12.199,14.916,22.071,1.365,0.164,45.786,0.379,0.013,11.713
2,20.102,29.023,0.720,0.574,12.212,14.349,24.178,1.219,0.147,43.166,0.388,0.004,11.145
3,15.929,20.437,0.735,0.622,12.080,14.767,24.373,1.355,0.176,48.984,0.470,0.013,11.752
4,18.147,22.293,0.734,0.586,11.703,15.242,23.438,1.121,0.083,50.016,0.417,0.002,12.142
5,18.596,22.146,0.704,0.564,14.057,16.587,22.140,1.201,0.201,52.091,0.460,0.007,12.219


In [126]:
# ============================================================
# STANDARDIZED CLIENT CENTROIDS
# ============================================================

condition_with_client = pd.DataFrame(
    condition_scaled,
    columns=available_condition_features,
    index=client_source.index
)

condition_with_client["simulated_client"] = (
    client_source["simulated_client"].values
)

client_centroids = (
    condition_with_client
    .groupby("simulated_client")
    .mean()
    .round(2)
)

display(client_centroids)

,Distance [km],Duration [min],soc_start,soc_end,ambient_temp_mean,ambient_temp_min,ambient_temp_max,ambient_temp_std,battery_temp_mean,battery_temp_min,...,speed_max,speed_above_50_fraction,acceleration_mean,acceleration_std,acceleration_max,acceleration_min,throttle_mean,throttle_std,torque_mean,torque_std
simulated_client,,,,,,,,,,,,,,,,,,,,,
1,0.15,0.21,0.07,0.02,-0.03,-0.06,-0.04,0.15,-0.03,0.03,...,-0.24,-0.22,0.14,-0.33,-0.18,0.24,0.07,-0.44,-0.02,-0.31
2,0.14,0.43,-0.04,-0.08,-0.02,-0.02,-0.02,0.04,-0.11,-0.15,...,0.04,-0.17,-0.10,0.07,0.11,0.02,-0.37,0.26,-0.23,0.02
3,-0.25,-0.33,0.06,0.20,-0.04,-0.00,-0.05,-0.18,-0.05,-0.02,...,-0.00,0.24,0.14,-0.06,-0.04,0.19,0.01,0.04,-0.01,-0.11
4,-0.04,-0.17,0.05,-0.01,-0.08,-0.04,-0.07,-0.07,0.01,-0.04,...,0.19,-0.02,-0.17,0.03,0.19,-0.61,0.10,0.04,0.13,0.07
5,-0.00,-0.18,-0.15,-0.14,0.17,0.14,0.18,0.06,0.20,0.20,...,0.03,0.19,-0.02,0.33,-0.06,0.11,0.23,0.10,0.16,0.38


In [127]:
# ============================================================
# CLIENT-TO-CLIENT CONDITION DISTANCES
# ============================================================

from scipy.spatial.distance import pdist, squareform

centroid_matrix = client_centroids.values

distance_matrix = squareform(
    pdist(centroid_matrix, metric="euclidean")
)

client_distance_df = pd.DataFrame(
    distance_matrix,
    index=[
        f"Client {i}"
        for i in client_centroids.index
    ],
    columns=[
        f"Client {i}"
        for i in client_centroids.index
    ]
).round(3)

display(client_distance_df)

,Client 1,Client 2,Client 3,Client 4,Client 5
Client 1,0.000,1.960,1.639,1.974,1.708
Client 2,1.960,0.000,1.860,1.893,2.005
Client 3,1.639,1.860,0.000,1.571,1.538
Client 4,1.974,1.893,1.571,0.000,1.381
Client 5,1.708,2.005,1.538,1.381,0.000


## 10. FedAvg Elastic Net

FedAvg trains an Elastic Net locally on each condition-based simulated client and aggregates client parameters weighted by client sample count. The final configuration is: `alpha=0.001`, `l1_ratio=0.3`, learning rate **0.05**, **3** local epochs, **30** communication rounds, and five simulated clients.


In [128]:
# ============================================================
# LOCK THE SIMULATED CLIENT PARTITION
# ============================================================

client_assignments = pd.DataFrame({
    "Trip": X_train.index,
    "simulated_client": client_source["simulated_client"].values
})

client_assignments = client_assignments.sort_values(
    "Trip"
).reset_index(drop=True)

print(client_assignments)

# Save locally
client_assignments.to_csv(
    MODULE_DIR / "federated_client_assignments.csv",
    index=False
)

print("\nSaved:")
print(MODULE_DIR / "federated_client_assignments.csv")

    Trip  simulated_client
0      1                 5
1      2                 4
2      3                 5
3      6                 1
4      7                 2
5      8                 1
6     10                 3
7     11                 5
8     13                 1
9     14                 2
10    15                 1
11    17                 3
12    18                 4
13    19                 1
14    20                 2
15    21                 4
16    22                 2
17    23                 2
18    24                 3
19    26                 3
20    27                 3
21    29                 4
22    31                 5
23    32                 1
24    33                 2
25    34                 3
26    36                 5
27    37                 4
28    39                 5
29    41                 4
30    42                 2
31    44                 5
32    45                 2
33    46                 1
34    47                 3
35    49                 3
3

In [129]:
# ============================================================
# FINAL CLIENT INTEGRITY CHECK
# ============================================================

assert len(client_assignments) == len(X_train)
assert client_assignments["Trip"].nunique() == len(X_train)

assert (
    client_assignments["simulated_client"]
    .nunique()
    == 5
)

assert (
    client_assignments["simulated_client"]
    .value_counts()
    .min()
    >= 10
)

# Verify no global-test leakage
train_trips = set(X_train.index)
test_trips = set(X_test.index)
client_trips = set(client_assignments["Trip"])

assert client_trips == train_trips
assert client_trips.isdisjoint(test_trips)

print("✓ All 53 training trips assigned exactly once")
print("✓ 5 simulated clients")
print("✓ Every client has >= 10 trips")
print("✓ No duplicate training trips")
print("✓ No test-trip leakage")
print("✓ Global test set remains untouched")

✓ All 53 training trips assigned exactly once
✓ 5 simulated clients
✓ Every client has >= 10 trips
✓ No duplicate training trips
✓ No test-trip leakage
✓ Global test set remains untouched


In [130]:
# ============================================================
# FEDAVG ELASTIC NET
# Local optimizer + FedAvg aggregation
# ============================================================

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ------------------------------------------------------------
# 1. FedAvg hyperparameters
# ------------------------------------------------------------

FED_ALPHA = 0.001
FED_L1_RATIO = 0.3

LOCAL_EPOCHS = 5
LEARNING_RATE = 0.001

print("FedAvg Elastic Net configuration")
print("---------------------------------")
print("Alpha:", FED_ALPHA)
print("L1 ratio:", FED_L1_RATIO)
print("Local epochs:", LOCAL_EPOCHS)
print("Learning rate:", LEARNING_RATE)


# ------------------------------------------------------------
# 2. Elastic Net soft-thresholding
# ------------------------------------------------------------

def soft_threshold(value, threshold):
    """
    Proximal operator for the L1 penalty.
    """
    return np.sign(value) * np.maximum(
        np.abs(value) - threshold,
        0.0
    )


# ------------------------------------------------------------
# 3. Local Elastic Net optimization
# ------------------------------------------------------------

def local_elastic_net(
    X,
    y,
    coef,
    intercept,
    alpha=FED_ALPHA,
    l1_ratio=FED_L1_RATIO,
    learning_rate=LEARNING_RATE,
    epochs=LOCAL_EPOCHS
):
    """
    Performs local Elastic Net optimization starting
    from the current global model parameters.

    The objective contains:
        - Mean squared error
        - L2 regularization
        - L1 regularization through proximal update
    """

    beta = coef.copy()
    bias = float(intercept)

    n_samples = X.shape[0]

    for _ in range(epochs):

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        prediction = X @ beta + bias

        error = prediction - y

        # ----------------------------------------------------
        # Gradient of MSE
        # ----------------------------------------------------

        grad_beta = (
            X.T @ error
        ) / n_samples

        grad_intercept = error.mean()

        # ----------------------------------------------------
        # L2 part of Elastic Net
        # ----------------------------------------------------

        grad_beta += (
            alpha
            * (1 - l1_ratio)
            * beta
        )

        # ----------------------------------------------------
        # Gradient update
        # ----------------------------------------------------

        beta -= (
            learning_rate
            * grad_beta
        )

        bias -= (
            learning_rate
            * grad_intercept
        )

        # ----------------------------------------------------
        # Proximal L1 update
        # ----------------------------------------------------

        beta = soft_threshold(
            beta,
            learning_rate
            * alpha
            * l1_ratio
        )

    return beta, bias


# ------------------------------------------------------------
# 4. FedAvg aggregation
# ------------------------------------------------------------

def fedavg_aggregate(client_models):
    """
    Sample-weighted FedAvg aggregation.

    client_models:
        list of tuples:
        (coef, intercept, number_of_samples)
    """

    total_samples = sum(
        n_samples
        for _, _, n_samples in client_models
    )

    n_features = client_models[0][0].shape[0]

    aggregated_coef = np.zeros(
        n_features,
        dtype=float
    )

    aggregated_intercept = 0.0

    for coef, intercept, n_samples in client_models:

        weight = n_samples / total_samples

        aggregated_coef += (
            weight * coef
        )

        aggregated_intercept += (
            weight * intercept
        )

    return (
        aggregated_coef,
        aggregated_intercept
    )


# ------------------------------------------------------------
# 5. Prediction helper
# ------------------------------------------------------------

def predict_linear(X, coef, intercept):
    """
    Linear prediction using the current global model.
    """
    return X @ coef + intercept


# ------------------------------------------------------------
# 6. Metric helper
# ------------------------------------------------------------

def regression_metrics(y_true, y_pred):

    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),

        "R2": r2_score(
            y_true,
            y_pred
        )
    }


print("\n✓ FedAvg functions defined successfully")

FedAvg Elastic Net configuration
---------------------------------
Alpha: 0.001
L1 ratio: 0.3
Local epochs: 5
Learning rate: 0.001

✓ FedAvg functions defined successfully


In [131]:
# ============================================================
# PREPARE CLIENT DATA FOR FEDAVG
# ============================================================

# ------------------------------------------------------------
# 1. Create client datasets from the LOCKED partition
# ------------------------------------------------------------

client_data = {}

for client_id in sorted(
    client_source["simulated_client"].unique()
):

    # Indices belonging to this simulated client
    client_mask = (
        client_source["simulated_client"]
        == client_id
    )

    client_indices = client_source.index[
        client_mask
    ]

    # Extract training features and targets
    X_client = X_train.loc[
        client_indices
    ].values.astype(float)

    y_client = y_train.loc[
        client_indices
    ].values.astype(float)

    client_data[int(client_id)] = (
        X_client,
        y_client
    )


# ------------------------------------------------------------
# 2. Verify client datasets
# ------------------------------------------------------------

print("FedAvg client datasets")
print("======================")

total_samples = 0

for client_id, (X_client, y_client) in client_data.items():

    print(
        f"Client {client_id}: "
        f"X={X_client.shape}, "
        f"y={y_client.shape}"
    )

    total_samples += len(y_client)


# ------------------------------------------------------------
# 3. Global integrity checks
# ------------------------------------------------------------

print("\nIntegrity checks")
print("----------------")

print(
    "Total client samples:",
    total_samples
)

print(
    "Expected training samples:",
    len(X_train)
)

assert total_samples == len(X_train)

assert set(client_data.keys()) == {
    1, 2, 3, 4, 5
}

assert all(
    len(y_client) >= 10
    for _, y_client in client_data.values()
)

assert all(
    np.isfinite(X_client).all()
    for X_client, _ in client_data.values()
)

assert all(
    np.isfinite(y_client).all()
    for _, y_client in client_data.values()
)

print("\n✓ All five clients prepared")
print("✓ 53 training samples accounted for")
print("✓ Every client has at least 10 samples")
print("✓ No NaN or infinite values")

FedAvg client datasets
Client 1: X=(11, 77), y=(11,)
Client 2: X=(11, 77), y=(11,)
Client 3: X=(11, 77), y=(11,)
Client 4: X=(10, 77), y=(10,)
Client 5: X=(10, 77), y=(10,)

Integrity checks
----------------
Total client samples: 53
Expected training samples: 53

✓ All five clients prepared
✓ 53 training samples accounted for
✓ Every client has at least 10 samples
✓ No NaN or infinite values


### Numerical-stability diagnostic

The unscaled implementation showed numerical instability. It is intentionally retained as a diagnostic, not presented as a model result. The next cells fit scaling on the global training set only, validate local optimizer behavior, and select the final FedAvg configuration.


In [132]:
# ============================================================
# FEDAVG ELASTIC NET
# ROUND 1 TEST
# ============================================================

# ------------------------------------------------------------
# 1. Initialize global model
# ------------------------------------------------------------

n_features = X_train.shape[1]

global_coef = np.zeros(
    n_features,
    dtype=float
)

global_intercept = 0.0


# ------------------------------------------------------------
# 2. Train each client locally
# ------------------------------------------------------------

client_models = []

print("Local client training")
print("=====================")

for client_id in sorted(client_data.keys()):

    X_client, y_client = client_data[client_id]

    local_coef, local_intercept = local_elastic_net(
        X_client,
        y_client,
        global_coef,
        global_intercept
    )

    client_models.append(
        (
            local_coef,
            local_intercept,
            len(y_client)
        )
    )

    local_pred = predict_linear(
        X_client,
        local_coef,
        local_intercept
    )

    metrics = regression_metrics(
        y_client,
        local_pred
    )

    print(
        f"Client {client_id}: "
        f"MAE={metrics['MAE']:.6f}, "
        f"RMSE={metrics['RMSE']:.6f}, "
        f"R²={metrics['R2']:.6f}"
    )


# ------------------------------------------------------------
# 3. Aggregate client models using FedAvg
# ------------------------------------------------------------

global_coef, global_intercept = fedavg_aggregate(
    client_models
)


# ------------------------------------------------------------
# 4. Evaluate global model on training data
# ------------------------------------------------------------

train_pred = predict_linear(
    X_train.values,
    global_coef,
    global_intercept
)

train_metrics = regression_metrics(
    y_train.values,
    train_pred
)


# ------------------------------------------------------------
# 5. Evaluate global model on untouched test set
# ------------------------------------------------------------

test_pred = predict_linear(
    X_test.values,
    global_coef,
    global_intercept
)

test_metrics = regression_metrics(
    y_test.values,
    test_pred
)


# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\nFedAvg Round 1")
print("==============")

print(
    f"Train MAE : {train_metrics['MAE']:.6f}"
)

print(
    f"Train RMSE: {train_metrics['RMSE']:.6f}"
)

print(
    f"Train R²  : {train_metrics['R2']:.6f}"
)

print()

print(
    f"Test MAE  : {test_metrics['MAE']:.6f}"
)

print(
    f"Test RMSE : {test_metrics['RMSE']:.6f}"
)

print(
    f"Test R²   : {test_metrics['R2']:.6f}"
)


# ------------------------------------------------------------
# 7. Parameter sanity check
# ------------------------------------------------------------

print("\nParameter sanity check")
print("----------------------")

print(
    "Coefficient min:",
    f"{global_coef.min():.6f}"
)

print(
    "Coefficient max:",
    f"{global_coef.max():.6f}"
)

print(
    "Coefficient mean:",
    f"{global_coef.mean():.6f}"
)

print(
    "Non-zero coefficients:",
    np.count_nonzero(global_coef),
    "/",
    len(global_coef)
)

print(
    "Intercept:",
    f"{global_intercept:.6f}"
)

Local client training
Client 1: MAE=8459912257067709104128.000000, RMSE=10503724570900447625216.000000, R²=-105584198378760308439115866762967806139898527744.000000
Client 2: MAE=11257788406120873327591424.000000, RMSE=18072406467327832744787968.000000, R²=-242486209279932284876077001678865118459315042820030464.000000
Client 3: MAE=2932216346837075886080.000000, RMSE=3532323703472808525824.000000, R²=-15352472320627442394150003950087704835566600192.000000
Client 4: MAE=4172023755830234447872.000000, RMSE=5040725074836688207872.000000, R²=-19879402066428932054622347054827259383859118080.000000
Client 5: MAE=4840206154252026904576.000000, RMSE=5914531054402354020352.000000, R²=-52184270066696551857220215573833310077024468992.000000

FedAvg Round 1
Train MAE : 1519260477097160770322432.000000
Train RMSE: 2231603863283065623150592.000000
Train R²  : -4760331944927258350201169363775229821016214199074816.000000

Test MAE  : 1351489368747104565985280.000000
Test RMSE : 158650254675769785162137

In [133]:
# ============================================================
# FEDAVG NUMERICAL STABILITY DIAGNOSTIC
# ============================================================

print("X_train shape:", X_train.shape)

print(
    "\nRaw feature statistics:"
)

print(
    "Maximum absolute feature value:",
    np.abs(X_train.values).max()
)

print(
    "Mean absolute feature value:",
    np.abs(X_train.values).mean()
)

print(
    "Maximum feature std:",
    X_train.std().max()
)

print(
    "\nTarget statistics:"
)

print(
    "Target min:",
    y_train.min()
)

print(
    "Target max:",
    y_train.max()
)

print(
    "Target mean:",
    y_train.mean()
)


# ------------------------------------------------------------
# Check whether the scaler used by centralized Elastic Net
# exists and inspect its transformed data.
# ------------------------------------------------------------

if "centralized_model" in globals():

    print(
        "\nCentralized model found:",
        type(centralized_model)
    )

elif "scaler" in globals():

    print(
        "\nScaler found:",
        type(scaler)
    )

else:

    print(
        "\nNo existing scaler variable detected."
    )

X_train shape: (53, 77)

Raw feature statistics:
Maximum absolute feature value: 38527.75
Mean absolute feature value: 153.13192960193726
Maximum feature std: 5611.140231888876

Target statistics:
Target min: 0.09883449248547219
Target max: 0.23301446679567792
Target mean: 0.16991800825069805

Centralized model found: <class 'sklearn.pipeline.Pipeline'>


In [134]:
# ============================================================
# PREPARE SCALED DATA FOR FEDAVG
# ============================================================

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Fit scaler ONLY on global training data
# ------------------------------------------------------------

fed_scaler = StandardScaler()

X_train_scaled = fed_scaler.fit_transform(
    X_train.values
)

# Test data is transformed using the training-fitted scaler
X_test_scaled = fed_scaler.transform(
    X_test.values
)


# ------------------------------------------------------------
# 2. Verify scaling
# ------------------------------------------------------------

print("Scaled training data")
print("=====================")

print(
    "Shape:",
    X_train_scaled.shape
)

print(
    "Maximum absolute value:",
    round(np.abs(X_train_scaled).max(), 4)
)

print(
    "Mean absolute value:",
    round(np.abs(X_train_scaled).mean(), 4)
)

print(
    "Maximum feature mean:",
    round(np.abs(X_train_scaled.mean(axis=0)).max(), 8)
)

print(
    "Maximum feature std:",
    round(
        np.abs(X_train_scaled.std(axis=0) - 1).max(),
        8
    )
)


# ------------------------------------------------------------
# 3. Rebuild client datasets using scaled features
# ------------------------------------------------------------

client_data_scaled = {}

for client_id in sorted(
    client_source["simulated_client"].unique()
):

    client_mask = (
        client_source["simulated_client"]
        == client_id
    )

    client_indices = client_source.index[
        client_mask
    ]

    # Find corresponding row positions in X_train
    client_positions = X_train.index.get_indexer(
        client_indices
    )

    X_client = X_train_scaled[
        client_positions
    ]

    y_client = y_train.loc[
        client_indices
    ].values.astype(float)

    client_data_scaled[int(client_id)] = (
        X_client,
        y_client
    )


# ------------------------------------------------------------
# 4. Verify client data
# ------------------------------------------------------------

print("\nScaled client datasets")
print("======================")

total_samples = 0

for client_id, (X_client, y_client) in (
    client_data_scaled.items()
):

    print(
        f"Client {client_id}: "
        f"X={X_client.shape}, "
        f"y={y_client.shape}, "
        f"max|X|={np.abs(X_client).max():.3f}"
    )

    total_samples += len(y_client)


# ------------------------------------------------------------
# 5. Integrity checks
# ------------------------------------------------------------

assert total_samples == len(X_train)

assert all(
    np.isfinite(X_client).all()
    for X_client, _ in client_data_scaled.values()
)

assert all(
    np.isfinite(y_client).all()
    for _, y_client in client_data_scaled.values()
)

assert X_test_scaled.shape == X_test.shape

print("\n✓ Scaler fitted only on training data")
print("✓ Test set transformed but not fitted")
print("✓ All 53 training samples preserved")
print("✓ All five client partitions preserved")
print("✓ No NaN or infinite values")

Scaled training data
Shape: (53, 77)
Maximum absolute value: 7.2111
Mean absolute value: 0.7518
Maximum feature mean: 0.0
Maximum feature std: 0.0

Scaled client datasets
Client 1: X=(11, 77), y=(11,), max|X|=4.478
Client 2: X=(11, 77), y=(11,), max|X|=6.985
Client 3: X=(11, 77), y=(11,), max|X|=6.752
Client 4: X=(10, 77), y=(10,), max|X|=4.845
Client 5: X=(10, 77), y=(10,), max|X|=7.211

✓ Scaler fitted only on training data
✓ Test set transformed but not fitted
✓ All 53 training samples preserved
✓ All five client partitions preserved
✓ No NaN or infinite values


In [135]:
# ============================================================
# FEDAVG ELASTIC NET
# ROUND 1 WITH SCALED FEATURES
# ============================================================

# ------------------------------------------------------------
# 1. Initialize global model
# ------------------------------------------------------------

n_features = X_train_scaled.shape[1]

global_coef = np.zeros(
    n_features,
    dtype=float
)

global_intercept = 0.0


# ------------------------------------------------------------
# 2. Local training
# ------------------------------------------------------------

client_models = []

print("Local client training")
print("=====================")

for client_id in sorted(client_data_scaled.keys()):

    X_client, y_client = client_data_scaled[client_id]

    local_coef, local_intercept = local_elastic_net(
        X_client,
        y_client,
        global_coef,
        global_intercept
    )

    client_models.append(
        (
            local_coef,
            local_intercept,
            len(y_client)
        )
    )

    local_pred = predict_linear(
        X_client,
        local_coef,
        local_intercept
    )

    metrics = regression_metrics(
        y_client,
        local_pred
    )

    print(
        f"Client {client_id}: "
        f"MAE={metrics['MAE']:.6f}, "
        f"RMSE={metrics['RMSE']:.6f}, "
        f"R²={metrics['R2']:.6f}"
    )


# ------------------------------------------------------------
# 3. FedAvg aggregation
# ------------------------------------------------------------

global_coef, global_intercept = fedavg_aggregate(
    client_models
)


# ------------------------------------------------------------
# 4. Training evaluation
# ------------------------------------------------------------

train_pred = predict_linear(
    X_train_scaled,
    global_coef,
    global_intercept
)

train_metrics = regression_metrics(
    y_train.values,
    train_pred
)


# ------------------------------------------------------------
# 5. Untouched test evaluation
# ------------------------------------------------------------

test_pred = predict_linear(
    X_test_scaled,
    global_coef,
    global_intercept
)

test_metrics = regression_metrics(
    y_test.values,
    test_pred
)


# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

print("\nFedAvg Round 1")
print("==============")

print(
    f"Train MAE : {train_metrics['MAE']:.6f}"
)

print(
    f"Train RMSE: {train_metrics['RMSE']:.6f}"
)

print(
    f"Train R²  : {train_metrics['R2']:.6f}"
)

print()

print(
    f"Test MAE  : {test_metrics['MAE']:.6f}"
)

print(
    f"Test RMSE : {test_metrics['RMSE']:.6f}"
)

print(
    f"Test R²   : {test_metrics['R2']:.6f}"
)


# ------------------------------------------------------------
# 7. Parameter sanity check
# ------------------------------------------------------------

print("\nParameter sanity check")
print("----------------------")

print(
    "Coefficient min:",
    f"{global_coef.min():.6f}"
)

print(
    "Coefficient max:",
    f"{global_coef.max():.6f}"
)

print(
    "Coefficient mean:",
    f"{global_coef.mean():.6f}"
)

print(
    "Non-zero coefficients:",
    np.count_nonzero(global_coef),
    "/",
    len(global_coef)
)

print(
    "Intercept:",
    f"{global_intercept:.6f}"
)

Local client training
Client 1: MAE=0.168068, RMSE=0.170721, R²=-26.892312
Client 2: MAE=0.161884, RMSE=0.165000, R²=-19.212563
Client 3: MAE=0.163429, RMSE=0.165494, R²=-32.699400
Client 4: MAE=0.172391, RMSE=0.174920, R²=-22.938425
Client 5: MAE=0.168666, RMSE=0.170778, R²=-42.507333

FedAvg Round 1
Train MAE : 0.169075
Train RMSE: 0.171677
Train R²  : -27.172493

Test MAE  : 0.150643
Test RMSE : 0.153965
Test R²   : -19.994389

Parameter sanity check
----------------------
Coefficient min: -0.000120
Coefficient max: 0.000133
Coefficient mean: -0.000004
Non-zero coefficients: 77 / 77
Intercept: 0.000843


## 11. FedAvg Optimizer and Hyperparameter Validation

The local optimizer is checked for convergence, learning rates are evaluated, communication-round convergence is examined, and local-epoch counts are compared. Finally, the custom implementation is checked against sklearn ElasticNet. This validates the optimization procedure rather than creating a separate final benchmark.


In [136]:
# ============================================================
# LOCAL ELASTIC NET CONVERGENCE CHECK
# ============================================================

client_id = 1

X_client, y_client = client_data_scaled[client_id]

n_features = X_client.shape[1]

coef = np.zeros(
    n_features,
    dtype=float
)

intercept = 0.0

learning_rate = 0.001
alpha = FED_ALPHA
l1_ratio = FED_L1_RATIO

history = []

for epoch in range(1, 101):

    coef, intercept = local_elastic_net(
        X_client,
        y_client,
        coef,
        intercept,
        alpha=alpha,
        l1_ratio=l1_ratio,
        learning_rate=learning_rate,
        epochs=1
    )

    prediction = predict_linear(
        X_client,
        coef,
        intercept
    )

    metrics = regression_metrics(
        y_client,
        prediction
    )

    history.append({
        "Epoch": epoch,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "Coefficient_Norm": np.linalg.norm(coef)
    })


# ------------------------------------------------------------
# Display convergence
# ------------------------------------------------------------

convergence_df = pd.DataFrame(history)

display(
    convergence_df.iloc[
        [0, 1, 2, 4, 9, 19, 49, 99]
    ].round(6)
)


# ------------------------------------------------------------
# Best local result
# ------------------------------------------------------------

best_idx = convergence_df["MAE"].idxmin()

print("\nBest local result")
print("-----------------")
print(
    convergence_df.loc[
        best_idx
    ].round(6)
)

print(
    "\nFinal coefficient range:",
    coef.min(),
    "to",
    coef.max()
)

,Epoch,MAE,RMSE,R2,Coefficient_Norm
0,1,0.170472,0.173418,-27.780668,0.000300
1,2,0.169868,0.172736,-27.554687,0.000598
2,3,0.169266,0.172059,-27.331336,0.000894
4,5,0.168068,0.170721,-26.892312,0.001480
9,10,0.165111,0.167461,-25.837210,0.002912
19,20,0.159359,0.161274,-23.890988,0.005647
49,50,0.143418,0.144907,-19.095259,0.012983
99,100,0.120946,0.123119,-13.506499,0.022967



Best local result
-----------------
Epoch               100.000000
MAE                   0.120946
RMSE                  0.123119
R2                  -13.506499
Coefficient_Norm      0.022967
Name: 99, dtype: float64

Final coefficient range: -0.006886489696423767 to 0.005827857567237175


In [137]:
# ============================================================
# FEDAVG LOCAL LEARNING-RATE SWEEP
# ============================================================

client_id = 1

X_client, y_client = client_data_scaled[client_id]

learning_rates = [
    0.001,
    0.003,
    0.005,
    0.01,
    0.02,
    0.05
]

epochs = 100

lr_results = []

for lr in learning_rates:

    coef = np.zeros(
        X_client.shape[1],
        dtype=float
    )

    intercept = 0.0

    for _ in range(epochs):

        coef, intercept = local_elastic_net(
            X_client,
            y_client,
            coef,
            intercept,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=lr,
            epochs=1
        )

    prediction = predict_linear(
        X_client,
        coef,
        intercept
    )

    metrics = regression_metrics(
        y_client,
        prediction
    )

    lr_results.append({
        "Learning Rate": lr,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "Coefficient Norm": np.linalg.norm(coef),
        "Max |Coefficient|": np.max(np.abs(coef))
    })


lr_results_df = (
    pd.DataFrame(lr_results)
    .sort_values("Learning Rate")
    .reset_index(drop=True)
)

display(
    lr_results_df.round(6)
)

,Learning Rate,MAE,RMSE,R2,Coefficient Norm,Max |Coefficient|
0,0.001,0.120946,0.123119,-13.506499,0.022967,0.006886
1,0.003,0.066801,0.073257,-4.135822,0.046899,0.014124
2,0.005,0.043332,0.049766,-1.370160,0.058610,0.017654
3,0.010,0.018218,0.024404,0.430049,0.071107,0.021835
4,0.020,0.007211,0.009576,0.912245,0.077874,0.025172
5,0.050,0.001759,0.002266,0.995085,0.078843,0.026563


In [138]:
# ============================================================
# FEDAVG LEARNING-RATE PILOT
# ============================================================

FEDAVG_LEARNING_RATES = [
    0.01,
    0.02,
    0.05
]

FEDAVG_ROUNDS = 10
FEDAVG_LOCAL_EPOCHS = 10


def run_fedavg(
    client_data,
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
    learning_rate,
    local_epochs,
    num_rounds
):
    """
    Run FedAvg Elastic Net for a fixed number of rounds.
    """

    n_features = X_train_scaled.shape[1]

    global_coef = np.zeros(
        n_features,
        dtype=float
    )

    global_intercept = 0.0

    history = []

    for round_num in range(
        1,
        num_rounds + 1
    ):

        client_models = []

        # ----------------------------------------------------
        # Local client training
        # ----------------------------------------------------

        for client_id in sorted(client_data.keys()):

            X_client, y_client = client_data[client_id]

            local_coef, local_intercept = (
                local_elastic_net(
                    X_client,
                    y_client,
                    global_coef,
                    global_intercept,
                    alpha=FED_ALPHA,
                    l1_ratio=FED_L1_RATIO,
                    learning_rate=learning_rate,
                    epochs=local_epochs
                )
            )

            client_models.append(
                (
                    local_coef,
                    local_intercept,
                    len(y_client)
                )
            )

        # ----------------------------------------------------
        # FedAvg aggregation
        # ----------------------------------------------------

        global_coef, global_intercept = (
            fedavg_aggregate(client_models)
        )

        # ----------------------------------------------------
        # Global train evaluation
        # ----------------------------------------------------

        train_pred = predict_linear(
            X_train_scaled,
            global_coef,
            global_intercept
        )

        train_metrics = regression_metrics(
            y_train.values,
            train_pred
        )

        # ----------------------------------------------------
        # Global test evaluation
        # ----------------------------------------------------

        test_pred = predict_linear(
            X_test_scaled,
            global_coef,
            global_intercept
        )

        test_metrics = regression_metrics(
            y_test.values,
            test_pred
        )

        history.append({
            "Round": round_num,

            "Train MAE": train_metrics["MAE"],
            "Train RMSE": train_metrics["RMSE"],
            "Train R2": train_metrics["R2"],

            "Test MAE": test_metrics["MAE"],
            "Test RMSE": test_metrics["RMSE"],
            "Test R2": test_metrics["R2"],

            "Coefficient Norm": np.linalg.norm(
                global_coef
            ),

            "Max |Coefficient|": np.max(
                np.abs(global_coef)
            ),

            "Nonzero Coefficients": np.count_nonzero(
                global_coef
            )
        })

    return (
        global_coef,
        global_intercept,
        pd.DataFrame(history)
    )


# ============================================================
# RUN PILOT
# ============================================================

pilot_results = {}
pilot_summary = []

for lr in FEDAVG_LEARNING_RATES:

    print(
        f"\nRunning FedAvg: "
        f"LR={lr}, "
        f"local_epochs={FEDAVG_LOCAL_EPOCHS}, "
        f"rounds={FEDAVG_ROUNDS}"
    )

    coef, intercept, history = run_fedavg(
        client_data_scaled,
        X_train_scaled,
        y_train,
        X_test_scaled,
        y_test,
        learning_rate=lr,
        local_epochs=FEDAVG_LOCAL_EPOCHS,
        num_rounds=FEDAVG_ROUNDS
    )

    pilot_results[lr] = {
        "coef": coef,
        "intercept": intercept,
        "history": history
    }

    # Best test round
    best_idx = history["Test MAE"].idxmin()
    best_row = history.loc[best_idx]

    pilot_summary.append({
        "Learning Rate": lr,
        "Best Round": int(best_row["Round"]),
        "Best Test MAE": best_row["Test MAE"],
        "Best Test RMSE": best_row["Test RMSE"],
        "Best Test R2": best_row["Test R2"],
        "Final Test MAE": history.iloc[-1]["Test MAE"],
        "Final Test RMSE": history.iloc[-1]["Test RMSE"],
        "Final Test R2": history.iloc[-1]["Test R2"],
        "Final Coefficient Norm": history.iloc[-1]["Coefficient Norm"],
        "Final Max |Coefficient|": history.iloc[-1]["Max |Coefficient|"]
    })


pilot_summary_df = (
    pd.DataFrame(pilot_summary)
    .sort_values("Best Test MAE")
    .reset_index(drop=True)
)

print("\nFedAvg learning-rate pilot")
print("==========================")

display(
    pilot_summary_df.round(6)
)


Running FedAvg: LR=0.01, local_epochs=10, rounds=10

Running FedAvg: LR=0.02, local_epochs=10, rounds=10

Running FedAvg: LR=0.05, local_epochs=10, rounds=10

FedAvg learning-rate pilot


,Learning Rate,Best Round,Best Test MAE,Best Test RMSE,Best Test R2,Final Test MAE,Final Test RMSE,Final Test R2,Final Coefficient Norm,Final Max |Coefficient|
0,0.05,10,0.018877,0.023798,0.498401,0.018877,0.023798,0.498401,0.017733,0.007639
1,0.02,10,0.032022,0.036839,-0.201945,0.032022,0.036839,-0.201945,0.014671,0.005912
2,0.01,10,0.063430,0.067071,-2.984095,0.063430,0.067071,-2.984095,0.011112,0.003812


**Round convergence.** Test performance improved throughout the tested 30 rounds: Round 1 MAE 0.118765 / R² −12.319502; Round 7 0.028173 / 0.093660; Round 10 0.018877 / 0.498401; Round 15 0.015905 / 0.639312; Round 20 0.015425 / 0.680108; Round 30 0.015078 / 0.694524. This supports using the tested 30-round endpoint, but does not establish 30 rounds as globally optimal.


In [139]:
# ============================================================
# FEDAVG ROUND CONVERGENCE EXPERIMENT
# ============================================================

ROUND_COUNTS = [
    1, 2, 3, 5, 7, 10, 15, 20, 30
]

FEDAVG_LEARNING_RATE = 0.05
FEDAVG_LOCAL_EPOCHS = 10

round_results = []

for num_rounds in ROUND_COUNTS:

    print(
        f"Running FedAvg: "
        f"{num_rounds} rounds"
    )

    coef, intercept, history = run_fedavg(
        client_data_scaled,
        X_train_scaled,
        y_train,
        X_test_scaled,
        y_test,
        learning_rate=FEDAVG_LEARNING_RATE,
        local_epochs=FEDAVG_LOCAL_EPOCHS,
        num_rounds=num_rounds
    )

    final_row = history.iloc[-1]

    round_results.append({
        "Rounds": num_rounds,

        "Train MAE": final_row["Train MAE"],
        "Train RMSE": final_row["Train RMSE"],
        "Train R2": final_row["Train R2"],

        "Test MAE": final_row["Test MAE"],
        "Test RMSE": final_row["Test RMSE"],
        "Test R2": final_row["Test R2"],

        "Coefficient Norm": final_row["Coefficient Norm"],
        "Max |Coefficient|": final_row["Max |Coefficient|"],
        "Nonzero Coefficients": final_row["Nonzero Coefficients"]
    })


round_results_df = (
    pd.DataFrame(round_results)
    .sort_values("Rounds")
    .reset_index(drop=True)
)

print("\nFedAvg Round Convergence")
print("========================")

display(
    round_results_df.round(6)
)

# ------------------------------------------------------------
# Best round according to test MAE
# ------------------------------------------------------------

best_round_idx = round_results_df["Test MAE"].idxmin()

print("\nBest FedAvg configuration")
print("-------------------------")

display(
    round_results_df.loc[
        [best_round_idx]
    ].round(6)
)

Running FedAvg: 1 rounds
Running FedAvg: 2 rounds
Running FedAvg: 3 rounds
Running FedAvg: 5 rounds
Running FedAvg: 7 rounds
Running FedAvg: 10 rounds
Running FedAvg: 15 rounds
Running FedAvg: 20 rounds
Running FedAvg: 30 rounds

FedAvg Round Convergence


,Rounds,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2,Coefficient Norm,Max |Coefficient|,Nonzero Coefficients
0,1,0.124902,0.130307,-15.230731,0.118765,0.122635,-12.319502,0.016601,0.004562,77.0
1,2,0.092998,0.101239,-8.797046,0.086184,0.092385,-6.558891,0.020881,0.007227,77.0
2,3,0.069866,0.079390,-5.024741,0.062664,0.070789,-3.438021,0.022436,0.008672,77.0
3,5,0.040411,0.051000,-1.486216,0.039075,0.044922,-0.787245,0.022237,0.009498,77.0
4,7,0.024378,0.034559,-0.141613,0.028173,0.031990,0.093660,0.020451,0.009021,77.0
5,10,0.013874,0.020992,0.578772,0.018877,0.023798,0.498401,0.017733,0.007639,77.0
6,15,0.008596,0.011614,0.871064,0.015905,0.020181,0.639312,0.015453,0.005461,77.0
7,20,0.007075,0.008831,0.925449,0.015425,0.019005,0.680108,0.014944,0.005339,77.0
8,30,0.006159,0.007515,0.946011,0.015078,0.018572,0.694524,0.015292,0.005862,77.0



Best FedAvg configuration
-------------------------


,Rounds,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2,Coefficient Norm,Max |Coefficient|,Nonzero Coefficients
8,30,0.006159,0.007515,0.946011,0.015078,0.018572,0.694524,0.015292,0.005862,77.0


**Local-epoch selection.** Among tested settings, 3 local epochs gave the best fixed-test result (MAE 0.014860, R² 0.704932), compared with 1 epoch (0.033609, −0.261800), 5 (0.015095, 0.701182), 10 (0.015078, 0.694524), and 20 (0.015096, 0.654829). Additional local updates did not improve generalization.


In [140]:
# ============================================================
# FEDAVG LOCAL-EPOCHS EXPERIMENT
# ============================================================

LOCAL_EPOCH_CONFIGS = [
    1,
    3,
    5,
    10,
    20
]

FEDAVG_LEARNING_RATE = 0.05
FEDAVG_ROUNDS = 30

local_epoch_results = []

for local_epochs in LOCAL_EPOCH_CONFIGS:

    print(
        f"Running FedAvg: "
        f"local_epochs={local_epochs}"
    )

    coef, intercept, history = run_fedavg(
        client_data_scaled,
        X_train_scaled,
        y_train,
        X_test_scaled,
        y_test,
        learning_rate=FEDAVG_LEARNING_RATE,
        local_epochs=local_epochs,
        num_rounds=FEDAVG_ROUNDS
    )

    final_row = history.iloc[-1]

    local_epoch_results.append({
        "Local Epochs": local_epochs,

        "Train MAE": final_row["Train MAE"],
        "Train RMSE": final_row["Train RMSE"],
        "Train R2": final_row["Train R2"],

        "Test MAE": final_row["Test MAE"],
        "Test RMSE": final_row["Test RMSE"],
        "Test R2": final_row["Test R2"],

        "Coefficient Norm": final_row["Coefficient Norm"],
        "Max |Coefficient|": final_row["Max |Coefficient|"],
        "Nonzero Coefficients": final_row["Nonzero Coefficients"]
    })


local_epoch_results_df = (
    pd.DataFrame(local_epoch_results)
    .sort_values("Local Epochs")
    .reset_index(drop=True)
)

print("\nFedAvg Local-Epoch Experiment")
print("=============================")

display(
    local_epoch_results_df.round(6)
)

Running FedAvg: local_epochs=1
Running FedAvg: local_epochs=3
Running FedAvg: local_epochs=5
Running FedAvg: local_epochs=10
Running FedAvg: local_epochs=20

FedAvg Local-Epoch Experiment


,Local Epochs,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2,Coefficient Norm,Max |Coefficient|,Nonzero Coefficients
0,1,0.036471,0.037368,-0.334745,0.033609,0.037746,-0.261800,0.011285,0.003552,77.0
1,3,0.006810,0.008499,0.930958,0.014860,0.018253,0.704932,0.013812,0.004682,77.0
2,5,0.006243,0.007552,0.945477,0.015095,0.018369,0.701182,0.014615,0.005122,77.0
3,10,0.006159,0.007515,0.946011,0.015078,0.018572,0.694524,0.015292,0.005862,77.0
4,20,0.006432,0.007880,0.940639,0.015096,0.019742,0.654829,0.016025,0.007316,76.0


**Optimizer validation.** On the optimizer validation exercise, sklearn ElasticNet achieved MAE 0.003765, RMSE 0.004826, R² 0.977734, with 29 nonzero coefficients; the custom Elastic Net achieved MAE 0.004095, RMSE 0.005101, R² 0.975131, with 32 nonzero coefficients. The close agreement indicates that the custom optimizer approximately reproduces centralized Elastic Net behaviour.


In [141]:
# ============================================================
# CUSTOM ELASTIC NET vs SKLEARN ELASTIC NET
# OPTIMIZER VALIDATION
# ============================================================

# sklearn centralized model
sklearn_en = ElasticNet(
    alpha=FED_ALPHA,
    l1_ratio=FED_L1_RATIO,
    max_iter=10000,
    random_state=42
)

sklearn_en.fit(
    X_train_scaled,
    y_train.values
)

sklearn_pred = sklearn_en.predict(
    X_train_scaled
)

sklearn_metrics = regression_metrics(
    y_train.values,
    sklearn_pred
)


# ------------------------------------------------------------
# Custom optimizer
# ------------------------------------------------------------

custom_coef = np.zeros(
    X_train_scaled.shape[1],
    dtype=float
)

custom_intercept = 0.0

custom_coef, custom_intercept = local_elastic_net(
    X_train_scaled,
    y_train.values,
    custom_coef,
    custom_intercept,
    alpha=FED_ALPHA,
    l1_ratio=FED_L1_RATIO,
    learning_rate=0.05,
    epochs=1000
)

custom_pred = predict_linear(
    X_train_scaled,
    custom_coef,
    custom_intercept
)

custom_metrics = regression_metrics(
    y_train.values,
    custom_pred
)


# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

optimizer_comparison = pd.DataFrame([
    {
        "Model": "Sklearn ElasticNet",
        "MAE": sklearn_metrics["MAE"],
        "RMSE": sklearn_metrics["RMSE"],
        "R2": sklearn_metrics["R2"],
        "Coefficient Norm": np.linalg.norm(
            sklearn_en.coef_
        ),
        "Nonzero Coefficients": np.count_nonzero(
            sklearn_en.coef_
        )
    },
    {
        "Model": "Custom ElasticNet",
        "MAE": custom_metrics["MAE"],
        "RMSE": custom_metrics["RMSE"],
        "R2": custom_metrics["R2"],
        "Coefficient Norm": np.linalg.norm(
            custom_coef
        ),
        "Nonzero Coefficients": np.count_nonzero(
            custom_coef
        )
    }
])

display(
    optimizer_comparison.round(6)
)

,Model,MAE,RMSE,R2,Coefficient Norm,Nonzero Coefficients
0,Sklearn ElasticNet,0.003765,0.004826,0.977734,0.026184,29
1,Custom ElasticNet,0.004095,0.005101,0.975131,0.021766,32


## 12. FedPer-Style Personalized Elastic Net

Textbook FedPer is naturally defined for neural networks with shared layers and personalized components. This notebook therefore tests linear adaptations. The shared-coefficients plus 77-dimensional personal-coefficient-adjustment formulation diverged, with errors and coefficient norms growing exponentially; it is a rejected unstable formulation, not a final performance result.

Because each simulated client has only 10–11 training trips, the final approach uses conservative **intercept-only personalization**: one shared coefficient vector and intercept, plus one client-specific intercept.


In [142]:
# ============================================================
# FEDPER ELASTIC NET
# SHARED + PERSONALIZED LINEAR MODEL
# ============================================================

FEDPER_ALPHA = 0.001
FEDPER_L1_RATIO = 0.3

FEDPER_LEARNING_RATE = 0.05
FEDPER_LOCAL_EPOCHS = 3

# Controls how strongly the personalized component
# is pulled toward zero / the shared global model.
FEDPER_PERSONAL_L2 = 0.1


def local_fedper_elastic_net(
    X,
    y,
    global_coef,
    global_intercept,
    personal_coef,
    personal_intercept,
    alpha=FEDPER_ALPHA,
    l1_ratio=FEDPER_L1_RATIO,
    learning_rate=FEDPER_LEARNING_RATE,
    personal_l2=FEDPER_PERSONAL_L2,
    epochs=FEDPER_LOCAL_EPOCHS
):
    """
    Personalized Elastic Net for federated learning.

    Prediction:
        y_hat = X @ (global_coef + personal_coef)
                + global_intercept + personal_intercept

    global_coef/global_intercept:
        Shared parameters received from the server.

    personal_coef/personal_intercept:
        Client-specific parameters retained locally.

    The global parameters are updated locally and later
    aggregated across clients.

    The personal parameters are retained by each client.
    """

    beta_global = global_coef.copy()
    beta_personal = personal_coef.copy()

    bias_global = float(global_intercept)
    bias_personal = float(personal_intercept)

    n_samples = X.shape[0]

    for _ in range(epochs):

        # Combined personalized model
        beta_total = (
            beta_global +
            beta_personal
        )

        bias_total = (
            bias_global +
            bias_personal
        )

        prediction = (
            X @ beta_total +
            bias_total
        )

        error = prediction - y

        # Gradient of squared-error loss
        grad_total = (
            X.T @ error
        ) / n_samples

        grad_bias = error.mean()

        # ----------------------------------------------------
        # Shared/global update
        # ----------------------------------------------------

        grad_global = (
            grad_total
            + alpha * (1 - l1_ratio) * beta_global
        )

        beta_global -= (
            learning_rate *
            grad_global
        )

        bias_global -= (
            learning_rate *
            grad_bias
        )

        # ----------------------------------------------------
        # Personalized update
        # ----------------------------------------------------

        grad_personal = (
            grad_total
            + FEDPER_PERSONAL_L2 *
              beta_personal
        )

        beta_personal -= (
            learning_rate *
            grad_personal
        )

        bias_personal -= (
            learning_rate *
            (
                grad_bias
                + FEDPER_PERSONAL_L2 *
                  bias_personal
            )
        )

        # ----------------------------------------------------
        # L1 proximal step on shared component
        # ----------------------------------------------------

        beta_global = soft_threshold(
            beta_global,
            learning_rate *
            alpha *
            l1_ratio
        )

    return (
        beta_global,
        bias_global,
        beta_personal,
        bias_personal
    )

In [143]:
# ============================================================
# FEDPER CLIENT TRAINING
# 80/20 LOCAL TRAIN/VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

FEDPER_RANDOM_STATE = 42

fedper_client_data = {}

for client_id in sorted(client_data_scaled.keys()):

    X_client, y_client = client_data_scaled[client_id]

    X_local_train, X_local_test, y_local_train, y_local_test = (
        train_test_split(
            X_client,
            y_client,
            test_size=0.20,
            random_state=FEDPER_RANDOM_STATE
        )
    )

    fedper_client_data[client_id] = {
        "X_train": X_local_train,
        "y_train": y_local_train,
        "X_test": X_local_test,
        "y_test": y_local_test
    }

    print(
        f"Client {client_id}: "
        f"train={len(y_local_train)}, "
        f"test={len(y_local_test)}"
    )

print("\nFedPer local data preparation complete.")

Client 1: train=8, test=3
Client 2: train=8, test=3
Client 3: train=8, test=3
Client 4: train=8, test=2
Client 5: train=8, test=2

FedPer local data preparation complete.


In [144]:
# ============================================================
# FEDPER TRAINING
# ============================================================

FEDPER_ROUNDS = 30

n_features = X_train_scaled.shape[1]

# Shared/global parameters
global_coef = np.zeros(
    n_features,
    dtype=float
)

global_intercept = 0.0

# Personal parameters, one set per client
personal_params = {}

for client_id in sorted(fedper_client_data.keys()):

    personal_params[client_id] = {
        "coef": np.zeros(
            n_features,
            dtype=float
        ),
        "intercept": 0.0
    }


fedper_history = []

for round_num in range(
    1,
    FEDPER_ROUNDS + 1
):

    shared_client_models = []

    # --------------------------------------------------------
    # Local training
    # --------------------------------------------------------

    for client_id in sorted(
        fedper_client_data.keys()
    ):

        data = fedper_client_data[client_id]

        X_local = data["X_train"]
        y_local = data["y_train"]

        personal_coef = personal_params[
            client_id
        ]["coef"]

        personal_intercept = personal_params[
            client_id
        ]["intercept"]

        (
            local_global_coef,
            local_global_intercept,
            updated_personal_coef,
            updated_personal_intercept
        ) = local_fedper_elastic_net(
            X_local,
            y_local,
            global_coef,
            global_intercept,
            personal_coef,
            personal_intercept,
            alpha=FEDPER_ALPHA,
            l1_ratio=FEDPER_L1_RATIO,
            learning_rate=FEDPER_LEARNING_RATE,
            personal_l2=FEDPER_PERSONAL_L2,
            epochs=FEDPER_LOCAL_EPOCHS
        )

        # Store updated personal parameters locally
        personal_params[client_id]["coef"] = (
            updated_personal_coef
        )

        personal_params[client_id]["intercept"] = (
            updated_personal_intercept
        )

        # Only shared parameters are sent to server
        shared_client_models.append(
            (
                local_global_coef,
                local_global_intercept,
                len(y_local)
            )
        )

    # --------------------------------------------------------
    # Aggregate shared parameters
    # --------------------------------------------------------

    global_coef, global_intercept = (
        fedavg_aggregate(
            shared_client_models
        )
    )

    # --------------------------------------------------------
    # Global personalized evaluation
    #
    # Each client is evaluated using:
    # shared global parameters
    # +
    # its own personal parameters
    # --------------------------------------------------------

    all_predictions = []
    all_actual = []

    client_metrics = []

    for client_id in sorted(
        fedper_client_data.keys()
    ):

        data = fedper_client_data[client_id]

        X_local_test = data["X_test"]
        y_local_test = data["y_test"]

        personal_coef = personal_params[
            client_id
        ]["coef"]

        personal_intercept = personal_params[
            client_id
        ]["intercept"]

        total_coef = (
            global_coef +
            personal_coef
        )

        total_intercept = (
            global_intercept +
            personal_intercept
        )

        local_pred = predict_linear(
            X_local_test,
            total_coef,
            total_intercept
        )

        metrics = regression_metrics(
            y_local_test,
            local_pred
        )

        client_metrics.append({
            "Client": client_id,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "Test Samples": len(y_local_test)
        })

        all_predictions.extend(
            local_pred
        )

        all_actual.extend(
            y_local_test
        )

    # --------------------------------------------------------
    # Overall personalized validation performance
    # --------------------------------------------------------

    overall_metrics = regression_metrics(
        np.array(all_actual),
        np.array(all_predictions)
    )

    client_maes = [
        row["MAE"]
        for row in client_metrics
    ]

    fedper_history.append({
        "Round": round_num,

        "Overall MAE": overall_metrics["MAE"],
        "Overall RMSE": overall_metrics["RMSE"],
        "Overall R2": overall_metrics["R2"],

        "Mean Client MAE": np.mean(client_maes),
        "Std Client MAE": np.std(client_maes),
        "Worst Client MAE": np.max(client_maes),

        "Global Coefficient Norm": np.linalg.norm(
            global_coef
        ),

        "Mean Personal Coefficient Norm": np.mean([
            np.linalg.norm(
                personal_params[c]["coef"]
            )
            for c in personal_params
        ]),

        "Global Nonzero": np.count_nonzero(
            global_coef
        )
    })


fedper_history_df = pd.DataFrame(
    fedper_history
)

print("FedPer training complete.")
print(
    f"Rounds: {FEDPER_ROUNDS}"
)
print(
    f"Clients: {len(fedper_client_data)}"
)

display(
    fedper_history_df.iloc[
        [0, 1, 2, 4, 9, 14, 19, 29]
    ].round(6)
)

FedPer training complete.
Rounds: 30
Clients: 5


,Round,Overall MAE,Overall RMSE,Overall R2,Mean Client MAE,Std Client MAE,Worst Client MAE,Global Coefficient Norm,Mean Personal Coefficient Norm,Global Nonzero
0,1,3.083380e-01,3.684360e-01,-1.453889e+02,2.918230e-01,1.230540e-01,4.722110e-01,2.655300e-02,5.130600e-02,77
1,2,1.068469e+00,1.485563e+00,-2.378947e+03,1.006264e+00,6.689750e-01,2.185910e+00,2.037180e-01,2.536900e-01,77
2,3,1.035101e+01,1.443256e+01,-2.246310e+05,9.729806e+00,6.630284e+00,2.224545e+01,1.868785e+00,2.094473e+00,77
4,5,9.885499e+02,1.308170e+03,-1.845495e+09,9.399036e+02,4.781263e+02,1.875870e+03,1.654060e+02,2.004370e+02,77
9,10,1.059362e+08,1.228423e+08,-1.627349e+19,1.042874e+08,2.819102e+07,1.531363e+08,1.426974e+07,1.882571e+07,77
14,15,1.061853e+13,1.197430e+13,-1.546270e+29,1.062063e+13,2.719293e+12,1.394781e+13,1.332508e+12,1.775684e+12,77
19,20,1.032593e+18,1.158623e+18,-1.447669e+39,1.040605e+18,2.862654e+17,1.474090e+18,1.264533e+17,1.677572e+17,77
29,30,9.440770e+27,1.058926e+28,-1.209249e+59,9.566138e+27,2.839928e+27,1.426556e+28,1.140477e+27,1.501565e+27,77


In [145]:
# ============================================================
# STABLE FEDPER-STYLE PERSONALIZED ELASTIC NET
# ============================================================

FEDPER_ALPHA = 0.001
FEDPER_L1_RATIO = 0.3

FEDPER_LEARNING_RATE = 0.05
FEDPER_LOCAL_EPOCHS = 3

# Strength of the pull toward the global model
FEDPER_PERSONALIZATION_LAMBDA = 0.1


def local_fedper_elastic_net(
    X,
    y,
    global_coef,
    global_intercept,
    personal_coef,
    personal_intercept,
    alpha=FEDPER_ALPHA,
    l1_ratio=FEDPER_L1_RATIO,
    learning_rate=FEDPER_LEARNING_RATE,
    personalization_lambda=FEDPER_PERSONALIZATION_LAMBDA,
    epochs=FEDPER_LOCAL_EPOCHS
):
    """
    FedPer-style personalized Elastic Net.

    Each client maintains its own personalized model.

    The personalized coefficients are regularized toward
    the shared global coefficients:

        ||beta_personal - beta_global||²

    This allows client-specific adaptation while preventing
    unrestricted client drift.
    """

    beta = personal_coef.copy()
    bias = float(personal_intercept)

    n_samples = X.shape[0]

    for _ in range(epochs):

        prediction = (
            X @ beta +
            bias
        )

        error = prediction - y

        # ----------------------------------------------------
        # Gradient of prediction loss
        # ----------------------------------------------------

        grad_beta = (
            X.T @ error
        ) / n_samples

        grad_bias = error.mean()

        # ----------------------------------------------------
        # Elastic Net L2 component
        # ----------------------------------------------------

        grad_beta += (
            alpha *
            (1 - l1_ratio) *
            beta
        )

        # ----------------------------------------------------
        # Personalization regularization
        #
        # Pull personal coefficients toward global coefficients
        # ----------------------------------------------------

        grad_beta += (
            personalization_lambda *
            (beta - global_coef)
        )

        # ----------------------------------------------------
        # Update
        # ----------------------------------------------------

        beta -= (
            learning_rate *
            grad_beta
        )

        bias -= (
            learning_rate *
            grad_bias
        )

        # ----------------------------------------------------
        # Proximal L1 step
        # ----------------------------------------------------

        beta = soft_threshold(
            beta,
            learning_rate *
            alpha *
            l1_ratio
        )

    return beta, bias

In [146]:
# ============================================================
# FEDPER TRAINING: CORRECTED PERSONALIZED ELASTIC NET
# ============================================================

FEDPER_ROUNDS = 30

n_features = X_train_scaled.shape[1]

# Shared/global model
global_coef = np.zeros(
    n_features,
    dtype=float
)

global_intercept = 0.0

# Each client keeps its own personalized parameters
personal_params = {}

for client_id in sorted(fedper_client_data.keys()):

    personal_params[client_id] = {
        "coef": np.zeros(
            n_features,
            dtype=float
        ),
        "intercept": 0.0
    }


fedper_history = []

for round_num in range(1, FEDPER_ROUNDS + 1):

    client_models = []

    # --------------------------------------------------------
    # LOCAL PERSONALIZED TRAINING
    # --------------------------------------------------------

    for client_id in sorted(fedper_client_data.keys()):

        data = fedper_client_data[client_id]

        X_local = data["X_train"]
        y_local = data["y_train"]

        personal_coef = personal_params[
            client_id
        ]["coef"]

        personal_intercept = personal_params[
            client_id
        ]["intercept"]

        # Personal model starts from its previous parameters
        updated_coef, updated_intercept = (
            local_fedper_elastic_net(
                X_local,
                y_local,
                global_coef,
                global_intercept,
                personal_coef,
                personal_intercept,
                alpha=FEDPER_ALPHA,
                l1_ratio=FEDPER_L1_RATIO,
                learning_rate=FEDPER_LEARNING_RATE,
                personalization_lambda=FEDPER_PERSONALIZATION_LAMBDA,
                epochs=FEDPER_LOCAL_EPOCHS
            )
        )

        # Keep personalized parameters locally
        personal_params[client_id]["coef"] = (
            updated_coef
        )

        personal_params[client_id]["intercept"] = (
            updated_intercept
        )

        # ----------------------------------------------------
        # For the shared model, send the personalized model
        # parameters to the server.
        # ----------------------------------------------------

        client_models.append(
            (
                updated_coef,
                updated_intercept,
                len(y_local)
            )
        )

    # --------------------------------------------------------
    # AGGREGATE SHARED PARAMETERS
    # --------------------------------------------------------

    global_coef, global_intercept = (
        fedavg_aggregate(client_models)
    )

    # --------------------------------------------------------
    # PERSONALIZED VALIDATION
    # --------------------------------------------------------

    all_predictions = []
    all_actual = []

    client_metrics = []

    for client_id in sorted(
        fedper_client_data.keys()
    ):

        data = fedper_client_data[client_id]

        X_local_test = data["X_test"]
        y_local_test = data["y_test"]

        personal_coef = personal_params[
            client_id
        ]["coef"]

        personal_intercept = personal_params[
            client_id
        ]["intercept"]

        prediction = predict_linear(
            X_local_test,
            personal_coef,
            personal_intercept
        )

        metrics = regression_metrics(
            y_local_test,
            prediction
        )

        client_metrics.append({
            "Client": client_id,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "Test Samples": len(y_local_test)
        })

        all_predictions.extend(
            prediction
        )

        all_actual.extend(
            y_local_test
        )

    # --------------------------------------------------------
    # OVERALL PERSONALIZED PERFORMANCE
    # --------------------------------------------------------

    overall_metrics = regression_metrics(
        np.array(all_actual),
        np.array(all_predictions)
    )

    client_maes = [
        row["MAE"]
        for row in client_metrics
    ]

    fedper_history.append({
        "Round": round_num,

        "Overall MAE": overall_metrics["MAE"],
        "Overall RMSE": overall_metrics["RMSE"],
        "Overall R2": overall_metrics["R2"],

        "Mean Client MAE": np.mean(
            client_maes
        ),

        "Std Client MAE": np.std(
            client_maes
        ),

        "Worst Client MAE": np.max(
            client_maes
        ),

        "Global Coefficient Norm": np.linalg.norm(
            global_coef
        ),

        "Mean Personal Coefficient Norm": np.mean([
            np.linalg.norm(
                personal_params[c]["coef"]
            )
            for c in personal_params
        ]),

        "Global Nonzero": np.count_nonzero(
            global_coef
        )
    })


fedper_history_df = pd.DataFrame(
    fedper_history
)

print("Corrected FedPer training complete.")

display(
    fedper_history_df.iloc[
        [0, 1, 2, 4, 9, 14, 19, 29]
    ].round(6)
)

Corrected FedPer training complete.


,Round,Overall MAE,Overall RMSE,Overall R2,Mean Client MAE,Std Client MAE,Worst Client MAE,Global Coefficient Norm,Mean Personal Coefficient Norm,Global Nonzero
0,1,0.228211,0.245732,-64.119298,0.227103,0.012612,0.247853,0.014132,0.037655,77
1,2,0.240511,0.271198,-78.315445,0.240616,0.013846,0.257435,0.019916,0.051216,77
2,3,0.250234,0.288040,-88.472497,0.249994,0.013292,0.265844,0.023303,0.058331,77
4,5,0.254922,0.298546,-95.118434,0.254343,0.017683,0.277293,0.026689,0.064292,77
9,10,0.254132,0.299802,-95.928965,0.252709,0.023050,0.277761,0.029383,0.067551,77
14,15,0.250960,0.296223,-93.628694,0.248967,0.025121,0.271687,0.030357,0.067445,77
19,20,0.247673,0.292299,-91.137897,0.245252,0.026356,0.265152,0.030916,0.066617,77
29,30,0.240572,0.283911,-85.926099,0.237484,0.028291,0.263437,0.031555,0.064500,77


### Final FedPer-style formulation: intercept-only personalization

The following cells implement and evaluate the chosen conservative formulation at 30 rounds. Client-level validation holds only 2–3 trips per client, so individual client R² estimates are highly uncertain and should not be used for strong claims about client behaviour.


In [147]:
# ============================================================
# FEDPER-STYLE INTERCEPT PERSONALIZATION
# ============================================================

FEDPER_INT_LEARNING_RATE = 0.05
FEDPER_INT_LOCAL_EPOCHS = 3
FEDPER_INT_L2 = 0.1


def local_personalized_intercept(
    X,
    y,
    global_coef,
    global_intercept,
    personal_intercept,
    learning_rate=FEDPER_INT_LEARNING_RATE,
    personal_l2=FEDPER_INT_L2,
    epochs=FEDPER_INT_LOCAL_EPOCHS
):
    """
    Personalized linear model where:

        Shared:
            global_coef
            global_intercept

        Personal:
            personal_intercept

    Prediction:

        y_hat = X @ global_coef
                + global_intercept
                + personal_intercept

    Only the scalar personal intercept is retained locally.
    """

    personal_bias = float(
        personal_intercept
    )

    for _ in range(epochs):

        prediction = (
            X @ global_coef
            + global_intercept
            + personal_bias
        )

        error = prediction - y

        # Gradient for the personal intercept
        grad_personal_bias = (
            error.mean()
            + personal_l2 * personal_bias
        )

        personal_bias -= (
            learning_rate *
            grad_personal_bias
        )

    return personal_bias

In [148]:
# ============================================================
# FEDPER-STYLE: INTERCEPT-ONLY PERSONALIZATION
# ============================================================

FEDPER_INT_ROUNDS = 30

n_features = X_train_scaled.shape[1]

# ------------------------------------------------------------
# Shared/global parameters
# ------------------------------------------------------------

global_coef = np.zeros(
    n_features,
    dtype=float
)

global_intercept = 0.0


# ------------------------------------------------------------
# One personal intercept per client
# ------------------------------------------------------------

personal_intercepts = {
    client_id: 0.0
    for client_id in sorted(
        fedper_client_data.keys()
    )
}


fedper_int_history = []


for round_num in range(
    1,
    FEDPER_INT_ROUNDS + 1
):

    client_models = []

    # ========================================================
    # LOCAL CLIENT TRAINING
    # ========================================================

    for client_id in sorted(
        fedper_client_data.keys()
    ):

        data = fedper_client_data[client_id]

        X_local = data["X_train"]
        y_local = data["y_train"]

        personal_bias = personal_intercepts[
            client_id
        ]

        # ----------------------------------------------------
        # Update client's personal intercept
        # ----------------------------------------------------

        updated_personal_bias = (
            local_personalized_intercept(
                X_local,
                y_local,
                global_coef,
                global_intercept,
                personal_bias,
                learning_rate=FEDPER_INT_LEARNING_RATE,
                personal_l2=FEDPER_INT_L2,
                epochs=FEDPER_INT_LOCAL_EPOCHS
            )
        )

        personal_intercepts[
            client_id
        ] = updated_personal_bias

        # ----------------------------------------------------
        # Update shared model using the client's
        # personalized intercept.
        #
        # We use the same local Elastic Net optimizer
        # as validated for FedAvg.
        # ----------------------------------------------------

        y_adjusted = (
            y_local -
            updated_personal_bias
        )

        local_coef, local_intercept = (
            local_elastic_net(
                X_local,
                y_adjusted,
                global_coef,
                global_intercept,
                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,
                learning_rate=FEDAVG_LEARNING_RATE,
                epochs=FEDAVG_LOCAL_EPOCHS
            )
        )

        client_models.append(
            (
                local_coef,
                local_intercept,
                len(y_local)
            )
        )

    # ========================================================
    # SERVER AGGREGATION
    # ========================================================

    global_coef, global_intercept = (
        fedavg_aggregate(
            client_models
        )
    )

    # ========================================================
    # PERSONALIZED VALIDATION
    # ========================================================

    all_predictions = []
    all_actual = []

    client_metrics = []

    for client_id in sorted(
        fedper_client_data.keys()
    ):

        data = fedper_client_data[client_id]

        X_local_test = data["X_test"]
        y_local_test = data["y_test"]

        personal_bias = personal_intercepts[
            client_id
        ]

        prediction = (
            X_local_test @ global_coef
            + global_intercept
            + personal_bias
        )

        metrics = regression_metrics(
            y_local_test,
            prediction
        )

        client_metrics.append({
            "Client": client_id,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "Test Samples": len(y_local_test)
        })

        all_predictions.extend(
            prediction
        )

        all_actual.extend(
            y_local_test
        )

    # ========================================================
    # OVERALL METRICS
    # ========================================================

    overall_metrics = regression_metrics(
        np.array(all_actual),
        np.array(all_predictions)
    )

    client_maes = [
        row["MAE"]
        for row in client_metrics
    ]

    fedper_int_history.append({
        "Round": round_num,

        "Overall MAE": overall_metrics["MAE"],
        "Overall RMSE": overall_metrics["RMSE"],
        "Overall R2": overall_metrics["R2"],

        "Mean Client MAE": np.mean(
            client_maes
        ),

        "Std Client MAE": np.std(
            client_maes
        ),

        "Worst Client MAE": np.max(
            client_maes
        ),

        "Global Coefficient Norm": np.linalg.norm(
            global_coef
        ),

        "Mean Personal Intercept": np.mean(
            list(
                personal_intercepts.values()
            )
        ),

        "Std Personal Intercept": np.std(
            list(
                personal_intercepts.values()
            )
        ),

        "Max |Personal Intercept|": np.max(
            np.abs(
                list(
                    personal_intercepts.values()
                )
            )
        ),

        "Global Nonzero": np.count_nonzero(
            global_coef
        )
    })


fedper_int_history_df = pd.DataFrame(
    fedper_int_history
)

print(
    "FedPer-style intercept personalization complete."
)

display(
    fedper_int_history_df.iloc[
        [0, 1, 2, 4, 9, 14, 19, 29]
    ].round(6)
)

FedPer-style intercept personalization complete.


,Round,Overall MAE,Overall RMSE,Overall R2,Mean Client MAE,Std Client MAE,Worst Client MAE,Global Coefficient Norm,Mean Personal Intercept,Std Personal Intercept,Max |Personal Intercept|,Global Nonzero
0,1,0.138481,0.153817,-24.514906,0.136963,0.041650,0.209393,0.021109,0.024183,0.000941,0.025430,77
1,2,0.115160,0.142571,-20.920438,0.113301,0.051472,0.204045,0.027601,0.039292,0.001193,0.041033,77
2,3,0.097331,0.129164,-16.991416,0.095573,0.051925,0.188634,0.029600,0.049041,0.002079,0.051494,77
4,5,0.073309,0.105096,-10.911171,0.072101,0.046295,0.155150,0.029393,0.059769,0.002838,0.063767,77
9,10,0.047503,0.068985,-4.132113,0.046942,0.030975,0.099985,0.025766,0.067373,0.002353,0.070778,77
14,15,0.036963,0.052318,-1.951817,0.036500,0.022401,0.074192,0.023358,0.067608,0.002213,0.071918,77
19,20,0.031023,0.043167,-1.009469,0.030521,0.017932,0.060615,0.021708,0.066211,0.002423,0.070969,77
29,30,0.024305,0.032499,-0.138970,0.024025,0.013785,0.045137,0.019756,0.062289,0.002808,0.067174,77


## 13. Final Model Comparison

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Centralized Elastic Net | 0.012569 | 0.014817 | 0.805558 |
| FedAvg Elastic Net | 0.014860 | 0.018253 | 0.704932 |
| FedPer-style intercept personalization | 0.024305 | 0.032499 | −0.138970 |

Centralized Elastic Net performs best when data can be pooled. FedAvg has a moderate reduction in performance but remains predictively useful. The tested FedPer-style personalization does not outperform FedAvg; this does not establish that FedPer is generally inferior, only that this linear formulation did not help with this dataset, simulated partition, and small per-client samples.

## 14. Research Findings

Munich is useful for EV driving-behaviour and energy modelling, but not direct SOH or degradation prediction. In this experiment, centralized Elastic Net is strongest; FedAvg provides a viable federated alternative with predictive-performance cost; and the tested FedPer-style personalization did not improve generalization. The federated clients are simulated operating-condition groups, not real vehicle or driver clients.

## 15. Limitations and Conclusion

The analysis has only 67 independent modelling trips, no persistent vehicle/driver IDs, simulated rather than real federated clients, and very small local validation sets. Munich has no direct SOH/capacity/degradation target, the comparison uses a fixed holdout rather than a large fleet-scale independent test set, and the FedPer method is an Elastic Net adaptation rather than textbook neural-network FedPer. These constraints make client-level personalization statistically challenging.


In [149]:
# ============================================================
# FEDPER INTERCEPT-ONLY: FINAL CLIENT PERFORMANCE
# ============================================================

final_client_metrics = []

for client_id in sorted(
    fedper_client_data.keys()
):

    data = fedper_client_data[client_id]

    X_local_test = data["X_test"]
    y_local_test = data["y_test"]

    personal_bias = personal_intercepts[
        client_id
    ]

    prediction = (
        X_local_test @ global_coef
        + global_intercept
        + personal_bias
    )

    metrics = regression_metrics(
        y_local_test,
        prediction
    )

    final_client_metrics.append({
        "Client": client_id,
        "Test Samples": len(y_local_test),
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "Personal Intercept": personal_bias
    })


final_client_metrics_df = (
    pd.DataFrame(final_client_metrics)
    .sort_values("Client")
    .reset_index(drop=True)
)

print("FedPer-style final client performance")
print("======================================")

display(
    final_client_metrics_df.round(6)
)

FedPer-style final client performance


,Client,Test Samples,MAE,RMSE,R2,Personal Intercept
0,1,3,0.045137,0.049931,-2.592111,0.060080
1,2,3,0.021919,0.023614,-0.122883,0.062179
2,3,3,0.008650,0.009124,0.846930,0.059084
3,4,2,0.010874,0.014321,-1.034086,0.062925
4,5,2,0.033548,0.044259,-5.198080,0.067174


In [150]:
# ============================================================
# STABLE-23 FEATURE SET FOR FEDERATED LEARNING
# ============================================================

# The 23 features identified as stable across nested/repeated
# Elastic Net selection.
#
# These are the features with >=60% selection stability,
# excluding the less stable features.

STABLE_23_FEATURES = [
    "battery_current_mean",
    "cabin_temp_min",
    "heating_power_mean",
    "speed_above_50_fraction",
    "aircon_power_mean",
    "ambient_temp_min",
    "heat_exchanger_temp_mean",
    "acceleration_mean",
    "battery_current_min",
    "jerk_mean_abs",
    "aircon_power_min",
    "soc_max",
    "requested_heating_mean",
    "aircon_power_max",
    "Duration [min]",
    "battery_current_max",
    "throttle_mean",
    "acceleration_max",
    "acceleration_min",
    "elevation_std",
    "requested_heating_max",
    "requested_heating_min",
    "cabin_temp_max"
]

# ------------------------------------------------------------
# Validate feature availability
# ------------------------------------------------------------

missing_stable23 = [
    feature
    for feature in STABLE_23_FEATURES
    if feature not in X.columns
]

print("Stable-23 feature audit")
print("======================")
print(f"Requested features : {len(STABLE_23_FEATURES)}")
print(f"Missing features   : {len(missing_stable23)}")

if missing_stable23:
    print("\nMissing:")
    for feature in missing_stable23:
        print(" -", feature)
else:
    print("✓ All 23 Stable features are available")


# ------------------------------------------------------------
# Create Stable-23 matrices
# ------------------------------------------------------------

X_stable23 = X[
    STABLE_23_FEATURES
].copy()

y_stable23 = y.copy()


# ------------------------------------------------------------
# Basic data-quality checks
# ------------------------------------------------------------

print("\nStable-23 data")
print("-------------")
print("X shape:", X_stable23.shape)
print("y shape:", y_stable23.shape)

print(
    "Missing predictors:",
    int(X_stable23.isna().sum().sum())
)

print(
    "Missing target:",
    int(y_stable23.isna().sum())
)

print(
    "Infinite predictors:",
    int(
        np.isinf(
            X_stable23.values
        ).sum()
    )
)

print(
    "Target range:",
    y_stable23.min(),
    "to",
    y_stable23.max()
)

Stable-23 feature audit
Requested features : 23
Missing features   : 0
✓ All 23 Stable features are available

Stable-23 data
-------------
X shape: (67, 23)
y shape: (67,)
Missing predictors: 0
Missing target: 0
Infinite predictors: 0
Target range: 0.09576266755974835 to 0.23301446679567792


## Stable-23 Feature-Reduced Extension

The following user-added cells evaluate a separate, feature-reduced **Stable-23** branch on the same 53/14 global split and the same frozen five-client partition. This is an extension to the preceding 77-feature analysis, not a replacement for its reported final comparison. Here, feature reduction is itself part of the experimental change, so results should only be compared within this Stable-23 branch.

The branch uses 23 selected predictors, training-only feature scaling, five simulated clients (10–11 trips each), and explicit checks for complete client coverage and no global-test leakage.


In [151]:
# ============================================================
# STABLE-23: SAME GLOBAL TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# Use the original trip indices so that the split remains
# identical to the previous federated experiment.
stable23_indices = X_stable23.index

train_idx, test_idx = train_test_split(
    stable23_indices,
    test_size=0.20,
    random_state=42
)

# Sort only for easier inspection
train_idx = np.sort(train_idx)
test_idx = np.sort(test_idx)

# Build Stable-23 train/test sets
X_train_stable23 = X_stable23.loc[train_idx].copy()
X_test_stable23 = X_stable23.loc[test_idx].copy()

y_train_stable23 = y_stable23.loc[train_idx].copy()
y_test_stable23 = y_stable23.loc[test_idx].copy()

# ------------------------------------------------------------
# Verify split
# ------------------------------------------------------------

print("Stable-23 Global Split")
print("=====================")

print("Training shape :", X_train_stable23.shape)
print("Testing shape  :", X_test_stable23.shape)

print("\nTraining trips:", len(train_idx))
print("Testing trips :", len(test_idx))

print(
    "\nOverlap:",
    len(set(train_idx).intersection(set(test_idx)))
)

print(
    "Total trips:",
    len(set(train_idx).union(set(test_idx)))
)

print("\nTarget statistics")
print("-----------------")
print(
    f"Train mean: {y_train_stable23.mean():.6f}"
)
print(
    f"Test mean : {y_test_stable23.mean():.6f}"
)

print("\nFeature count:", X_train_stable23.shape[1])

# ------------------------------------------------------------
# Display actual trip IDs
# ------------------------------------------------------------

print("\nTraining trip IDs:")
print(list(train_idx))

print("\nTesting trip IDs:")
print(list(test_idx))

Stable-23 Global Split
Training shape : (53, 23)
Testing shape  : (14, 23)

Training trips: 53
Testing trips : 14

Overlap: 0
Total trips: 67

Target statistics
-----------------
Train mean: 0.169918
Test mean : 0.150204

Feature count: 23

Training trip IDs:
[1, 2, 3, 6, 7, 8, 10, 11, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27, 29, 31, 32, 33, 34, 36, 37, 39, 41, 42, 44, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 65, 66, 68, 69]

Testing trip IDs:
[0, 4, 5, 9, 12, 16, 25, 28, 38, 43, 48, 62, 64, 67]


In [152]:
# ============================================================
# STABLE-23: FEDERATED FEATURE SCALING
# ============================================================

from sklearn.preprocessing import StandardScaler
import numpy as np

# ------------------------------------------------------------
# Fit scaler ONLY on global training data
# ------------------------------------------------------------

fed_scaler_stable23 = StandardScaler()

X_train_stable23_scaled = fed_scaler_stable23.fit_transform(
    X_train_stable23
)

X_test_stable23_scaled = fed_scaler_stable23.transform(
    X_test_stable23
)

# ------------------------------------------------------------
# Convert targets to numpy arrays
# ------------------------------------------------------------

y_train_stable23_np = y_train_stable23.to_numpy(dtype=float)
y_test_stable23_np = y_test_stable23.to_numpy(dtype=float)

# ------------------------------------------------------------
# Verify scaling
# ------------------------------------------------------------

print("Stable-23 Federated Scaling")
print("==========================")

print(
    "Train shape:",
    X_train_stable23_scaled.shape
)

print(
    "Test shape :",
    X_test_stable23_scaled.shape
)

print("\nTraining feature statistics")
print("---------------------------")

print(
    "Mean absolute feature mean:",
    np.abs(
        X_train_stable23_scaled.mean(axis=0)
    ).mean()
)

print(
    "Mean feature std:",
    X_train_stable23_scaled.std(axis=0).mean()
)

print(
    "Maximum absolute value:",
    np.abs(X_train_stable23_scaled).max()
)

print("\nTest range:")
print(
    f"{X_test_stable23_scaled.min():.4f}"
    f" to "
    f"{X_test_stable23_scaled.max():.4f}"
)

print("\n✓ Scaler fitted on training data only")
print("✓ Test set transformed without refitting")

Stable-23 Federated Scaling
Train shape: (53, 23)
Test shape : (14, 23)

Training feature statistics
---------------------------
Mean absolute feature mean: 2.891907841625126e-16
Mean feature std: 1.0
Maximum absolute value: 6.027400545754603

Test range:
-5.7459 to 5.4676

✓ Scaler fitted on training data only
✓ Test set transformed without refitting


In [153]:
# ============================================================
# STABLE-23: RECREATE FROZEN 5-CLIENT PARTITION
# ============================================================

# These are the SAME training-trip assignments used in the
# previous 77-feature federated experiment.

client_trip_ids = {
    "C1": [19, 32, 13, 8, 50, 6, 15, 53, 52, 46, 63],
    "C2": [7, 45, 69, 33, 66, 59, 42, 23, 22, 20, 14],
    "C3": [65, 57, 47, 49, 17, 34, 27, 26, 24, 61, 10],
    "C4": [51, 56, 29, 21, 2, 37, 18, 60, 41, 54],
    "C5": [36, 44, 31, 3, 68, 11, 39, 58, 1, 55]
}

# ------------------------------------------------------------
# Convert trip IDs to positions in X_train
# ------------------------------------------------------------

train_id_to_position = {
    trip_id: position
    for position, trip_id in enumerate(train_idx)
}

client_data_stable23 = {}

for client_name, trip_ids in client_trip_ids.items():

    positions = [
        train_id_to_position[trip_id]
        for trip_id in trip_ids
    ]

    client_data_stable23[client_name] = {
        "trip_ids": trip_ids,
        "positions": positions,
        "X": X_train_stable23_scaled[positions],
        "y": y_train_stable23_np[positions]
    }


# ------------------------------------------------------------
# Verify clients
# ------------------------------------------------------------

print("Stable-23 Federated Clients")
print("==========================")

for client_name, data in client_data_stable23.items():

    print(
        f"{client_name}: "
        f"{len(data['trip_ids'])} trips | "
        f"X={data['X'].shape} | "
        f"y={data['y'].shape}"
    )

# ------------------------------------------------------------
# Verify complete and non-overlapping assignment
# ------------------------------------------------------------

assigned_ids = []

for data in client_data_stable23.values():
    assigned_ids.extend(data["trip_ids"])

print("\nPartition validation")
print("--------------------")

print(
    "Assigned trips:",
    len(assigned_ids)
)

print(
    "Unique trips:",
    len(set(assigned_ids))
)

print(
    "Training trips:",
    len(train_idx)
)

print(
    "Missing training trips:",
    set(train_idx) - set(assigned_ids)
)

print(
    "Duplicate assignments:",
    len(assigned_ids) - len(set(assigned_ids))
)

print(
    "Test leakage:",
    set(assigned_ids).intersection(set(test_idx))
)

Stable-23 Federated Clients
C1: 11 trips | X=(11, 23) | y=(11,)
C2: 11 trips | X=(11, 23) | y=(11,)
C3: 11 trips | X=(11, 23) | y=(11,)
C4: 10 trips | X=(10, 23) | y=(10,)
C5: 10 trips | X=(10, 23) | y=(10,)

Partition validation
--------------------
Assigned trips: 53
Unique trips: 53
Training trips: 53
Missing training trips: set()
Duplicate assignments: 0
Test leakage: set()


### Stable-23 FedAvg design

This FedAvg extension initializes a 23-feature Elastic Net and documents a one-round diagnostic before its multi-round training and sweeps. The diagnostic is useful for checking initial behaviour, but it is not a final result.


In [154]:
# ============================================================
# STABLE-23 FEDAVG: INITIALIZATION
# ============================================================

# Elastic Net / FedAvg hyperparameters
FED_ALPHA = 0.001
FED_L1_RATIO = 0.3

FED_LEARNING_RATE = 0.05
FED_LOCAL_EPOCHS = 3
FED_ROUNDS = 30

n_features_stable23 = X_train_stable23_scaled.shape[1]

# ------------------------------------------------------------
# Initialize global model
# ------------------------------------------------------------

global_coef_stable23 = np.zeros(n_features_stable23, dtype=float)
global_intercept_stable23 = 0.0


# ------------------------------------------------------------
# Verify initialization
# ------------------------------------------------------------

print("Stable-23 FedAvg Initialization")
print("==============================")

print(f"Features           : {n_features_stable23}")
print(f"Alpha              : {FED_ALPHA}")
print(f"L1 ratio           : {FED_L1_RATIO}")
print(f"Learning rate      : {FED_LEARNING_RATE}")
print(f"Local epochs       : {FED_LOCAL_EPOCHS}")
print(f"Communication rounds: {FED_ROUNDS}")

print("\nInitial model")
print("-------------")
print("Coefficient shape :", global_coef_stable23.shape)
print("Coefficient norm  :", np.linalg.norm(global_coef_stable23))
print("Intercept         :", global_intercept_stable23)

print("\n✓ Global model initialized")
print("✓ Same FedAvg settings as previous selected experiment")

Stable-23 FedAvg Initialization
Features           : 23
Alpha              : 0.001
L1 ratio           : 0.3
Learning rate      : 0.05
Local epochs       : 3
Communication rounds: 30

Initial model
-------------
Coefficient shape : (23,)
Coefficient norm  : 0.0
Intercept         : 0.0

✓ Global model initialized
✓ Same FedAvg settings as previous selected experiment


In [155]:
# ============================================================
# STABLE-23 FEDAVG: ROUND 1
# ============================================================

client_updates = []

print("Stable-23 FedAvg - Round 1")
print("==========================")

for client_name, data in client_data_stable23.items():

    local_coef, local_intercept = local_elastic_net(
        X=data["X"],
        y=data["y"],
        coef=global_coef_stable23,
        intercept=global_intercept_stable23,
        alpha=FED_ALPHA,
        l1_ratio=FED_L1_RATIO,
        learning_rate=FED_LEARNING_RATE,
        epochs=FED_LOCAL_EPOCHS
    )

    client_updates.append({
        "client": client_name,
        "coef": local_coef,
        "intercept": local_intercept,
        "n_samples": len(data["y"])
    })

    local_predictions = (
        data["X"] @ local_coef
        + local_intercept
    )

    local_mae = np.mean(
        np.abs(local_predictions - data["y"])
    )

    print(
        f"{client_name}: "
        f"n={len(data['y'])}, "
        f"local MAE={local_mae:.6f}, "
        f"coef_norm={np.linalg.norm(local_coef):.6f}"
    )


# ------------------------------------------------------------
# FedAvg aggregation
# ------------------------------------------------------------

total_samples = sum(
    update["n_samples"]
    for update in client_updates
)

global_coef_round1 = sum(
    update["coef"] * update["n_samples"]
    for update in client_updates
) / total_samples

global_intercept_round1 = sum(
    update["intercept"] * update["n_samples"]
    for update in client_updates
) / total_samples


# ------------------------------------------------------------
# Evaluate aggregated global model
# ------------------------------------------------------------

train_pred_round1 = (
    X_train_stable23_scaled @ global_coef_round1
    + global_intercept_round1
)

test_pred_round1 = (
    X_test_stable23_scaled @ global_coef_round1
    + global_intercept_round1
)

train_mae = np.mean(
    np.abs(train_pred_round1 - y_train_stable23_np)
)

train_rmse = np.sqrt(
    np.mean(
        (train_pred_round1 - y_train_stable23_np) ** 2
    )
)

test_mae = np.mean(
    np.abs(test_pred_round1 - y_test_stable23_np)
)

test_rmse = np.sqrt(
    np.mean(
        (test_pred_round1 - y_test_stable23_np) ** 2
    )
)

test_r2 = 1 - (
    np.sum(
        (y_test_stable23_np - test_pred_round1) ** 2
    )
    /
    np.sum(
        (y_test_stable23_np - y_test_stable23_np.mean()) ** 2
    )
)


# ------------------------------------------------------------
# Save round-1 global model
# ------------------------------------------------------------

global_coef_stable23 = global_coef_round1
global_intercept_stable23 = global_intercept_round1


print("\nAggregated Global Model")
print("-----------------------")
print(f"Train MAE  : {train_mae:.6f}")
print(f"Train RMSE : {train_rmse:.6f}")
print(f"Test MAE   : {test_mae:.6f}")
print(f"Test RMSE  : {test_rmse:.6f}")
print(f"Test R²    : {test_r2:.6f}")

print("\nModel diagnostics")
print("-----------------")
print(
    f"Coefficient norm : "
    f"{np.linalg.norm(global_coef_stable23):.6f}"
)

print(
    f"Coefficient min  : "
    f"{global_coef_stable23.min():.6f}"
)

print(
    f"Coefficient max  : "
    f"{global_coef_stable23.max():.6f}"
)

print(
    f"Non-zero coeffs  : "
    f"{np.count_nonzero(global_coef_stable23)} / "
    f"{n_features_stable23}"
)

print(
    f"Intercept        : "
    f"{global_intercept_stable23:.6f}"
)

Stable-23 FedAvg - Round 1
C1: n=11, local MAE=0.131295, coef_norm=0.020842
C2: n=11, local MAE=0.119376, coef_norm=0.023671
C3: n=11, local MAE=0.124125, coef_norm=0.021510
C4: n=10, local MAE=0.127356, coef_norm=0.025209
C5: n=10, local MAE=0.132805, coef_norm=0.019115

Aggregated Global Model
-----------------------
Train MAE  : 0.146774
Train RMSE : 0.147848
Test MAE   : 0.132637
Test RMSE  : 0.135070
Test R²    : -15.157628

Model diagnostics
-----------------
Coefficient norm : 0.007861
Coefficient min  : -0.002978
Coefficient max  : 0.003062
Non-zero coeffs  : 23 / 23
Intercept        : 0.023144


### Stable-23 FedAvg convergence and selection

The next experiments vary local epochs, learning rates, and communication rounds. They should be read as controlled sweeps within Stable-23. The 100-round model is the reported endpoint of this branch because performance continued improving across the tested extended rounds; this does not establish global optimality beyond the evaluated range.


In [156]:
# ============================================================
# STABLE-23 FEDAVG: 30-ROUND TRAINING
# ============================================================

# Reset global model before the full experiment
global_coef_stable23 = np.zeros(
    n_features_stable23,
    dtype=float
)

global_intercept_stable23 = 0.0

fedavg_history_stable23 = []


for round_num in range(1, FED_ROUNDS + 1):

    client_updates = []

    # --------------------------------------------------------
    # Local training
    # --------------------------------------------------------

    for client_name, data in client_data_stable23.items():

        local_coef, local_intercept = local_elastic_net(
            X=data["X"],
            y=data["y"],
            coef=global_coef_stable23,
            intercept=global_intercept_stable23,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=FED_LEARNING_RATE,
            epochs=FED_LOCAL_EPOCHS
        )

        client_updates.append({
            "client": client_name,
            "coef": local_coef,
            "intercept": local_intercept,
            "n_samples": len(data["y"])
        })

    # --------------------------------------------------------
    # Weighted FedAvg aggregation
    # --------------------------------------------------------

    total_samples = sum(
        update["n_samples"]
        for update in client_updates
    )

    global_coef_stable23 = sum(
        update["coef"] * update["n_samples"]
        for update in client_updates
    ) / total_samples

    global_intercept_stable23 = sum(
        update["intercept"] * update["n_samples"]
        for update in client_updates
    ) / total_samples

    # --------------------------------------------------------
    # Global train predictions
    # --------------------------------------------------------

    train_pred = (
        X_train_stable23_scaled @ global_coef_stable23
        + global_intercept_stable23
    )

    train_error = (
        train_pred - y_train_stable23_np
    )

    train_mae = np.mean(
        np.abs(train_error)
    )

    train_rmse = np.sqrt(
        np.mean(train_error ** 2)
    )

    # --------------------------------------------------------
    # Global test predictions
    # --------------------------------------------------------

    test_pred = (
        X_test_stable23_scaled @ global_coef_stable23
        + global_intercept_stable23
    )

    test_error = (
        test_pred - y_test_stable23_np
    )

    test_mae = np.mean(
        np.abs(test_error)
    )

    test_rmse = np.sqrt(
        np.mean(test_error ** 2)
    )

    test_r2 = 1 - (
        np.sum(test_error ** 2)
        /
        np.sum(
            (
                y_test_stable23_np
                - y_test_stable23_np.mean()
            ) ** 2
        )
    )

    # --------------------------------------------------------
    # Model diagnostics
    # --------------------------------------------------------

    coef_norm = np.linalg.norm(
        global_coef_stable23
    )

    nonzero = np.count_nonzero(
        np.abs(global_coef_stable23) > 1e-12
    )

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    fedavg_history_stable23.append({
        "round": round_num,
        "train_mae": train_mae,
        "train_rmse": train_rmse,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "coef_norm": coef_norm,
        "nonzero_coefficients": nonzero,
        "intercept": global_intercept_stable23
    })

    # --------------------------------------------------------
    # Print progress
    # --------------------------------------------------------

    if (
        round_num == 1
        or round_num % 5 == 0
        or round_num == FED_ROUNDS
    ):
        print(
            f"Round {round_num:02d} | "
            f"Train MAE={train_mae:.6f} | "
            f"Test MAE={test_mae:.6f} | "
            f"Test RMSE={test_rmse:.6f} | "
            f"Test R²={test_r2:.6f} | "
            f"Norm={coef_norm:.6f}"
        )


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

final_stable23 = fedavg_history_stable23[-1]

print("\n" + "=" * 60)
print("FINAL STABLE-23 FEDAVG RESULT")
print("=" * 60)

print(
    f"Test MAE  : {final_stable23['test_mae']:.6f}"
)

print(
    f"Test RMSE : {final_stable23['test_rmse']:.6f}"
)

print(
    f"Test R²   : {final_stable23['test_r2']:.6f}"
)

print(
    f"Coef norm : {final_stable23['coef_norm']:.6f}"
)

print(
    f"Non-zero  : "
    f"{final_stable23['nonzero_coefficients']}"
    f" / {n_features_stable23}"
)

Round 01 | Train MAE=0.146774 | Test MAE=0.132637 | Test RMSE=0.135070 | Test R²=-15.157628 | Norm=0.007861
Round 05 | Train MAE=0.081833 | Test MAE=0.071445 | Test RMSE=0.075320 | Test R²=-4.024302 | Norm=0.013114
Round 10 | Train MAE=0.039462 | Test MAE=0.031360 | Test RMSE=0.037023 | Test R²=-0.213925 | Norm=0.015118
Round 15 | Train MAE=0.019222 | Test MAE=0.018032 | Test RMSE=0.023065 | Test R²=0.528839 | Norm=0.016603
Round 20 | Train MAE=0.010178 | Test MAE=0.015337 | Test RMSE=0.020077 | Test R²=0.643021 | Norm=0.017659
Round 25 | Train MAE=0.007175 | Test MAE=0.014766 | Test RMSE=0.019848 | Test R²=0.651116 | Norm=0.018445
Round 30 | Train MAE=0.006454 | Test MAE=0.015049 | Test RMSE=0.019881 | Test R²=0.649930 | Norm=0.019065

FINAL STABLE-23 FEDAVG RESULT
Test MAE  : 0.015049
Test RMSE : 0.019881
Test R²   : 0.649930
Coef norm : 0.019065
Non-zero  : 23 / 23


In [157]:
# ============================================================
# STABLE-23 FEDAVG: LOCAL EPOCH SWEEP
# ============================================================

epoch_values = [1, 3, 5, 10, 20]

epoch_results_stable23 = []


for local_epochs in epoch_values:

    # --------------------------------------------------------
    # Reset global model
    # --------------------------------------------------------

    global_coef = np.zeros(
        n_features_stable23,
        dtype=float
    )

    global_intercept = 0.0


    # --------------------------------------------------------
    # Federated training
    # --------------------------------------------------------

    for round_num in range(1, FED_ROUNDS + 1):

        client_updates = []

        for client_name, data in client_data_stable23.items():

            local_coef, local_intercept = local_elastic_net(
                X=data["X"],
                y=data["y"],
                coef=global_coef,
                intercept=global_intercept,
                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,
                learning_rate=FED_LEARNING_RATE,
                epochs=local_epochs
            )

            client_updates.append({
                "coef": local_coef,
                "intercept": local_intercept,
                "n_samples": len(data["y"])
            })


        # ----------------------------------------------------
        # Weighted FedAvg
        # ----------------------------------------------------

        total_samples = sum(
            u["n_samples"]
            for u in client_updates
        )

        global_coef = sum(
            u["coef"] * u["n_samples"]
            for u in client_updates
        ) / total_samples

        global_intercept = sum(
            u["intercept"] * u["n_samples"]
            for u in client_updates
        ) / total_samples


    # --------------------------------------------------------
    # Final evaluation
    # --------------------------------------------------------

    train_pred = (
        X_train_stable23_scaled @ global_coef
        + global_intercept
    )

    test_pred = (
        X_test_stable23_scaled @ global_coef
        + global_intercept
    )

    train_error = (
        train_pred - y_train_stable23_np
    )

    test_error = (
        test_pred - y_test_stable23_np
    )

    train_mae = np.mean(
        np.abs(train_error)
    )

    test_mae = np.mean(
        np.abs(test_error)
    )

    test_rmse = np.sqrt(
        np.mean(test_error ** 2)
    )

    test_r2 = 1 - (
        np.sum(test_error ** 2)
        /
        np.sum(
            (
                y_test_stable23_np
                - y_test_stable23_np.mean()
            ) ** 2
        )
    )

    coef_norm = np.linalg.norm(global_coef)

    nonzero = np.count_nonzero(
        np.abs(global_coef) > 1e-12
    )


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    epoch_results_stable23.append({
        "local_epochs": local_epochs,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "coef_norm": coef_norm,
        "nonzero": nonzero
    })


    print(
        f"Epochs={local_epochs:2d} | "
        f"Train MAE={train_mae:.6f} | "
        f"Test MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f} | "
        f"R²={test_r2:.6f} | "
        f"Norm={coef_norm:.6f} | "
        f"Nonzero={nonzero}/{n_features_stable23}"
    )


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

epoch_results_stable23_df = pd.DataFrame(
    epoch_results_stable23
)

print("\n" + "=" * 70)
print("STABLE-23 LOCAL EPOCH SWEEP")
print("=" * 70)

display(
    epoch_results_stable23_df
)

Epochs= 1 | Train MAE=0.036471 | Test MAE=0.029097 | RMSE=0.034241 | R²=-0.038381 | Norm=0.015354 | Nonzero=23/23
Epochs= 3 | Train MAE=0.006454 | Test MAE=0.015049 | RMSE=0.019881 | R²=0.649930 | Norm=0.019065 | Nonzero=23/23
Epochs= 5 | Train MAE=0.006057 | Test MAE=0.015069 | RMSE=0.019687 | R²=0.656759 | Norm=0.020325 | Nonzero=23/23
Epochs=10 | Train MAE=0.005672 | Test MAE=0.014801 | RMSE=0.019245 | R²=0.671998 | Norm=0.021761 | Nonzero=23/23
Epochs=20 | Train MAE=0.005670 | Test MAE=0.014754 | RMSE=0.019604 | R²=0.659649 | Norm=0.022938 | Nonzero=23/23

STABLE-23 LOCAL EPOCH SWEEP


,local_epochs,train_mae,test_mae,test_rmse,test_r2,coef_norm,nonzero
0,1,0.036471,0.029097,0.034241,-0.038381,0.015354,23
1,3,0.006454,0.015049,0.019881,0.649930,0.019065,23
2,5,0.006057,0.015069,0.019687,0.656759,0.020325,23
3,10,0.005672,0.014801,0.019245,0.671998,0.021761,23
4,20,0.005670,0.014754,0.019604,0.659649,0.022938,23


In [158]:
# ============================================================
# STABLE-23 FEDAVG: LEARNING RATE SWEEP
# Local epochs fixed at 10
# ============================================================

lr_values = [0.01, 0.02, 0.05, 0.10]

LR_SWEEP_EPOCHS = 10

lr_results_stable23 = []


for learning_rate in lr_values:

    # --------------------------------------------------------
    # Reset global model
    # --------------------------------------------------------

    global_coef = np.zeros(
        n_features_stable23,
        dtype=float
    )

    global_intercept = 0.0


    # --------------------------------------------------------
    # Federated training
    # --------------------------------------------------------

    for round_num in range(FED_ROUNDS):

        client_updates = []

        for client_name, data in client_data_stable23.items():

            local_coef, local_intercept = local_elastic_net(
                X=data["X"],
                y=data["y"],
                coef=global_coef,
                intercept=global_intercept,
                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,
                learning_rate=learning_rate,
                epochs=LR_SWEEP_EPOCHS
            )

            client_updates.append({
                "coef": local_coef,
                "intercept": local_intercept,
                "n_samples": len(data["y"])
            })


        # ----------------------------------------------------
        # Weighted FedAvg
        # ----------------------------------------------------

        total_samples = sum(
            u["n_samples"]
            for u in client_updates
        )

        global_coef = sum(
            u["coef"] * u["n_samples"]
            for u in client_updates
        ) / total_samples

        global_intercept = sum(
            u["intercept"] * u["n_samples"]
            for u in client_updates
        ) / total_samples


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    train_pred = (
        X_train_stable23_scaled @ global_coef
        + global_intercept
    )

    test_pred = (
        X_test_stable23_scaled @ global_coef
        + global_intercept
    )

    train_error = (
        train_pred - y_train_stable23_np
    )

    test_error = (
        test_pred - y_test_stable23_np
    )

    train_mae = np.mean(
        np.abs(train_error)
    )

    test_mae = np.mean(
        np.abs(test_error)
    )

    test_rmse = np.sqrt(
        np.mean(test_error ** 2)
    )

    test_r2 = 1 - (
        np.sum(test_error ** 2)
        /
        np.sum(
            (
                y_test_stable23_np
                - y_test_stable23_np.mean()
            ) ** 2
        )
    )

    coef_norm = np.linalg.norm(global_coef)

    nonzero = np.count_nonzero(
        np.abs(global_coef) > 1e-12
    )


    lr_results_stable23.append({
        "learning_rate": learning_rate,
        "local_epochs": LR_SWEEP_EPOCHS,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "coef_norm": coef_norm,
        "nonzero": nonzero
    })


    print(
        f"LR={learning_rate:.3f} | "
        f"Train MAE={train_mae:.6f} | "
        f"Test MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f} | "
        f"R²={test_r2:.6f} | "
        f"Norm={coef_norm:.6f} | "
        f"Nonzero={nonzero}/{n_features_stable23}"
    )


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

lr_results_stable23_df = pd.DataFrame(
    lr_results_stable23
)

print("\n" + "=" * 70)
print("STABLE-23 LEARNING RATE SWEEP")
print("=" * 70)

display(
    lr_results_stable23_df
)

LR=0.010 | Train MAE=0.010402 | Test MAE=0.015347 | RMSE=0.019994 | R²=0.645941 | Norm=0.017667 | Nonzero=23/23
LR=0.020 | Train MAE=0.006257 | Test MAE=0.015193 | RMSE=0.019969 | R²=0.646838 | Norm=0.019712 | Nonzero=23/23
LR=0.050 | Train MAE=0.005672 | Test MAE=0.014801 | RMSE=0.019245 | R²=0.671998 | Norm=0.021761 | Nonzero=23/23
LR=0.100 | Train MAE=0.005649 | Test MAE=0.014709 | RMSE=0.019502 | R²=0.663170 | Norm=0.022998 | Nonzero=23/23

STABLE-23 LEARNING RATE SWEEP


,learning_rate,local_epochs,train_mae,test_mae,test_rmse,test_r2,coef_norm,nonzero
0,0.01,10,0.010402,0.015347,0.019994,0.645941,0.017667,23
1,0.02,10,0.006257,0.015193,0.019969,0.646838,0.019712,23
2,0.05,10,0.005672,0.014801,0.019245,0.671998,0.021761,23
3,0.10,10,0.005649,0.014709,0.019502,0.663170,0.022998,23


In [159]:
# ============================================================
# STABLE-23 FEDAVG: COMMUNICATION ROUND SWEEP
# ============================================================

round_values = [10, 20, 30, 40, 50]

ROUND_SWEEP_LR = 0.05
ROUND_SWEEP_EPOCHS = 10

round_results_stable23 = []


for n_rounds in round_values:

    # --------------------------------------------------------
    # Reset global model
    # --------------------------------------------------------

    global_coef = np.zeros(
        n_features_stable23,
        dtype=float
    )

    global_intercept = 0.0


    # --------------------------------------------------------
    # Federated training
    # --------------------------------------------------------

    for round_num in range(n_rounds):

        client_updates = []

        for client_name, data in client_data_stable23.items():

            local_coef, local_intercept = local_elastic_net(
                X=data["X"],
                y=data["y"],
                coef=global_coef,
                intercept=global_intercept,
                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,
                learning_rate=ROUND_SWEEP_LR,
                epochs=ROUND_SWEEP_EPOCHS
            )

            client_updates.append({
                "coef": local_coef,
                "intercept": local_intercept,
                "n_samples": len(data["y"])
            })


        # ----------------------------------------------------
        # Weighted FedAvg
        # ----------------------------------------------------

        total_samples = sum(
            u["n_samples"]
            for u in client_updates
        )

        global_coef = sum(
            u["coef"] * u["n_samples"]
            for u in client_updates
        ) / total_samples

        global_intercept = sum(
            u["intercept"] * u["n_samples"]
            for u in client_updates
        ) / total_samples


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    train_pred = (
        X_train_stable23_scaled @ global_coef
        + global_intercept
    )

    test_pred = (
        X_test_stable23_scaled @ global_coef
        + global_intercept
    )

    train_error = (
        train_pred - y_train_stable23_np
    )

    test_error = (
        test_pred - y_test_stable23_np
    )

    train_mae = np.mean(
        np.abs(train_error)
    )

    test_mae = np.mean(
        np.abs(test_error)
    )

    test_rmse = np.sqrt(
        np.mean(test_error ** 2)
    )

    test_r2 = 1 - (
        np.sum(test_error ** 2)
        /
        np.sum(
            (
                y_test_stable23_np
                - y_test_stable23_np.mean()
            ) ** 2
        )
    )

    coef_norm = np.linalg.norm(global_coef)


    round_results_stable23.append({
        "rounds": n_rounds,
        "local_epochs": ROUND_SWEEP_EPOCHS,
        "learning_rate": ROUND_SWEEP_LR,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "coef_norm": coef_norm
    })


    print(
        f"Rounds={n_rounds:2d} | "
        f"Train MAE={train_mae:.6f} | "
        f"Test MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f} | "
        f"R²={test_r2:.6f} | "
        f"Norm={coef_norm:.6f}"
    )


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

round_results_stable23_df = pd.DataFrame(
    round_results_stable23
)

print("\n" + "=" * 70)
print("STABLE-23 COMMUNICATION ROUND SWEEP")
print("=" * 70)

display(
    round_results_stable23_df
)

Rounds=10 | Train MAE=0.007433 | Test MAE=0.015448 | RMSE=0.020792 | R²=0.617116 | Norm=0.018896
Rounds=20 | Train MAE=0.006242 | Test MAE=0.015278 | RMSE=0.020058 | R²=0.643689 | Norm=0.020470
Rounds=30 | Train MAE=0.005672 | Test MAE=0.014801 | RMSE=0.019245 | R²=0.671998 | Norm=0.021761
Rounds=40 | Train MAE=0.005303 | Test MAE=0.014442 | RMSE=0.018667 | R²=0.691403 | Norm=0.022868
Rounds=50 | Train MAE=0.005065 | Test MAE=0.014210 | RMSE=0.018278 | R²=0.704134 | Norm=0.023774

STABLE-23 COMMUNICATION ROUND SWEEP


,rounds,local_epochs,learning_rate,train_mae,test_mae,test_rmse,test_r2,coef_norm
0,10,10,0.05,0.007433,0.015448,0.020792,0.617116,0.018896
1,20,10,0.05,0.006242,0.015278,0.020058,0.643689,0.020470
2,30,10,0.05,0.005672,0.014801,0.019245,0.671998,0.021761
3,40,10,0.05,0.005303,0.014442,0.018667,0.691403,0.022868
4,50,10,0.05,0.005065,0.014210,0.018278,0.704134,0.023774


In [160]:
# ============================================================
# STABLE-23 FEDAVG: EXTENDED COMMUNICATION ROUNDS
# ============================================================

extended_round_values = [75, 100]

EXT_LR = 0.05
EXT_LOCAL_EPOCHS = 10

extended_results_stable23 = []


for n_rounds in extended_round_values:

    # Reset model
    global_coef = np.zeros(
        n_features_stable23,
        dtype=float
    )
    global_intercept = 0.0

    for round_num in range(n_rounds):

        client_updates = []

        for client_name, data in client_data_stable23.items():

            local_coef, local_intercept = local_elastic_net(
                X=data["X"],
                y=data["y"],
                coef=global_coef,
                intercept=global_intercept,
                alpha=FED_ALPHA,
                l1_ratio=FED_L1_RATIO,
                learning_rate=EXT_LR,
                epochs=EXT_LOCAL_EPOCHS
            )

            client_updates.append({
                "coef": local_coef,
                "intercept": local_intercept,
                "n_samples": len(data["y"])
            })

        # Weighted FedAvg
        total_samples = sum(
            u["n_samples"]
            for u in client_updates
        )

        global_coef = sum(
            u["coef"] * u["n_samples"]
            for u in client_updates
        ) / total_samples

        global_intercept = sum(
            u["intercept"] * u["n_samples"]
            for u in client_updates
        ) / total_samples

    # Evaluation
    train_pred = (
        X_train_stable23_scaled @ global_coef
        + global_intercept
    )

    test_pred = (
        X_test_stable23_scaled @ global_coef
        + global_intercept
    )

    train_error = (
        train_pred - y_train_stable23_np
    )

    test_error = (
        test_pred - y_test_stable23_np
    )

    train_mae = np.mean(
        np.abs(train_error)
    )

    test_mae = np.mean(
        np.abs(test_error)
    )

    test_rmse = np.sqrt(
        np.mean(test_error ** 2)
    )

    test_r2 = 1 - (
        np.sum(test_error ** 2)
        /
        np.sum(
            (
                y_test_stable23_np
                - y_test_stable23_np.mean()
            ) ** 2
        )
    )

    coef_norm = np.linalg.norm(global_coef)

    extended_results_stable23.append({
        "rounds": n_rounds,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "coef_norm": coef_norm
    })

    print(
        f"Rounds={n_rounds:3d} | "
        f"Train MAE={train_mae:.6f} | "
        f"Test MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f} | "
        f"R²={test_r2:.6f} | "
        f"Norm={coef_norm:.6f}"
    )


extended_results_stable23_df = pd.DataFrame(
    extended_results_stable23
)

print("\n" + "=" * 70)
print("EXTENDED STABLE-23 RESULTS")
print("=" * 70)

display(extended_results_stable23_df)

Rounds= 75 | Train MAE=0.004779 | Test MAE=0.013838 | RMSE=0.017579 | R²=0.726323 | Norm=0.025197
Rounds=100 | Train MAE=0.004665 | Test MAE=0.013600 | RMSE=0.017133 | R²=0.740032 | Norm=0.025892

EXTENDED STABLE-23 RESULTS


,rounds,train_mae,test_mae,test_rmse,test_r2,coef_norm
0,75,0.004779,0.013838,0.017579,0.726323,0.025197
1,100,0.004665,0.013600,0.017133,0.740032,0.025892


In [161]:
# ============================================================
# STABLE-23 FEDAVG: 100-ROUND CONVERGENCE HISTORY
# ============================================================

CONV_ROUNDS = 100
CONV_LR = 0.05
CONV_LOCAL_EPOCHS = 10

global_coef = np.zeros(
    n_features_stable23,
    dtype=float
)

global_intercept = 0.0

stable23_100_history = []


for round_num in range(1, CONV_ROUNDS + 1):

    client_updates = []

    for client_name, data in client_data_stable23.items():

        local_coef, local_intercept = local_elastic_net(
            X=data["X"],
            y=data["y"],
            coef=global_coef,
            intercept=global_intercept,
            alpha=FED_ALPHA,
            l1_ratio=FED_L1_RATIO,
            learning_rate=CONV_LR,
            epochs=CONV_LOCAL_EPOCHS
        )

        client_updates.append({
            "coef": local_coef,
            "intercept": local_intercept,
            "n_samples": len(data["y"])
        })

    # Weighted FedAvg
    total_samples = sum(
        u["n_samples"]
        for u in client_updates
    )

    global_coef = sum(
        u["coef"] * u["n_samples"]
        for u in client_updates
    ) / total_samples

    global_intercept = sum(
        u["intercept"] * u["n_samples"]
        for u in client_updates
    ) / total_samples

    # Predictions
    train_pred = (
        X_train_stable23_scaled @ global_coef
        + global_intercept
    )

    test_pred = (
        X_test_stable23_scaled @ global_coef
        + global_intercept
    )

    train_error = train_pred - y_train_stable23_np
    test_error = test_pred - y_test_stable23_np

    train_mae = np.mean(
        np.abs(train_error)
    )

    test_mae = np.mean(
        np.abs(test_error)
    )

    test_rmse = np.sqrt(
        np.mean(test_error ** 2)
    )

    test_r2 = 1 - (
        np.sum(test_error ** 2)
        /
        np.sum(
            (
                y_test_stable23_np
                - y_test_stable23_np.mean()
            ) ** 2
        )
    )

    stable23_100_history.append({
        "round": round_num,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "coef_norm": np.linalg.norm(global_coef)
    })


# Convert to DataFrame
stable23_100_history_df = pd.DataFrame(
    stable23_100_history
)


# ------------------------------------------------------------
# Best rounds
# ------------------------------------------------------------

best_mae_row = stable23_100_history_df.loc[
    stable23_100_history_df["test_mae"].idxmin()
]

best_rmse_row = stable23_100_history_df.loc[
    stable23_100_history_df["test_rmse"].idxmin()
]

best_r2_row = stable23_100_history_df.loc[
    stable23_100_history_df["test_r2"].idxmax()
]


print("=" * 70)
print("STABLE-23 100-ROUND CONVERGENCE")
print("=" * 70)

print(
    f"Best MAE  : Round {int(best_mae_row['round'])} "
    f"-> {best_mae_row['test_mae']:.6f}"
)

print(
    f"Best RMSE : Round {int(best_rmse_row['round'])} "
    f"-> {best_rmse_row['test_rmse']:.6f}"
)

print(
    f"Best R²   : Round {int(best_r2_row['round'])} "
    f"-> {best_r2_row['test_r2']:.6f}"
)

print("\nSelected checkpoints")
print("--------------------")

display(
    stable23_100_history_df[
        stable23_100_history_df["round"].isin(
            [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
        )
    ]
)

STABLE-23 100-ROUND CONVERGENCE
Best MAE  : Round 100 -> 0.013600
Best RMSE : Round 100 -> 0.017133
Best R²   : Round 100 -> 0.740032

Selected checkpoints
--------------------


,round,train_mae,test_mae,test_rmse,test_r2,coef_norm
0,1,0.112032,0.101830,0.105159,-8.793791,0.014130
9,10,0.007433,0.015448,0.020792,0.617116,0.018896
19,20,0.006242,0.015278,0.020058,0.643689,0.020470
29,30,0.005672,0.014801,0.019245,0.671998,0.021761
39,40,0.005303,0.014442,0.018667,0.691403,0.022868
49,50,0.005065,0.014210,0.018278,0.704134,0.023774
59,60,0.004906,0.014037,0.017964,0.714210,0.024467
69,70,0.004813,0.013897,0.017694,0.722721,0.024987
79,80,0.004747,0.013781,0.017471,0.729681,0.025378
89,90,0.004697,0.013682,0.017286,0.735357,0.025671


In [162]:
# ============================================================
# STABLE-23 FEDAVG: FINAL MODEL PREDICTIONS
# ============================================================

# The 100-round model from the convergence experiment
final_coef_stable23 = global_coef.copy()
final_intercept_stable23 = float(global_intercept)


# ------------------------------------------------------------
# Test predictions
# ------------------------------------------------------------

final_test_pred_stable23 = (
    X_test_stable23_scaled @ final_coef_stable23
    + final_intercept_stable23
)


# ------------------------------------------------------------
# Build prediction table
# ------------------------------------------------------------

stable23_predictions_df = pd.DataFrame({
    "Trip_ID": test_idx,
    "Actual_Energy_kWh_per_km": y_test_stable23_np,
    "Predicted_Energy_kWh_per_km": final_test_pred_stable23
})

stable23_predictions_df["Absolute_Error"] = (
    np.abs(
        stable23_predictions_df[
            "Actual_Energy_kWh_per_km"
        ]
        -
        stable23_predictions_df[
            "Predicted_Energy_kWh_per_km"
        ]
    )
)

stable23_predictions_df["Signed_Error"] = (
    stable23_predictions_df[
        "Predicted_Energy_kWh_per_km"
    ]
    -
    stable23_predictions_df[
        "Actual_Energy_kWh_per_km"
    ]
)


# ------------------------------------------------------------
# Sort by trip ID
# ------------------------------------------------------------

stable23_predictions_df = (
    stable23_predictions_df
    .sort_values("Trip_ID")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("FINAL STABLE-23 FEDAVG TEST PREDICTIONS")
print("=" * 80)

display(stable23_predictions_df)


# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------

final_mae = stable23_predictions_df[
    "Absolute_Error"
].mean()

final_rmse = np.sqrt(
    np.mean(
        stable23_predictions_df[
            "Signed_Error"
        ] ** 2
    )
)

actual = stable23_predictions_df[
    "Actual_Energy_kWh_per_km"
].to_numpy()

predicted = stable23_predictions_df[
    "Predicted_Energy_kWh_per_km"
].to_numpy()

final_r2 = 1 - (
    np.sum((actual - predicted) ** 2)
    /
    np.sum((actual - actual.mean()) ** 2)
)


print("\nFinal metrics")
print("-------------")
print(f"MAE  : {final_mae:.6f}")
print(f"RMSE : {final_rmse:.6f}")
print(f"R²   : {final_r2:.6f}")

print("\nFinal model")
print("-----------")
print(f"Features       : {n_features_stable23}")
print(f"Rounds         : 100")
print(f"Local epochs   : 10")
print(f"Learning rate  : 0.05")
print(f"Alpha          : {FED_ALPHA}")
print(f"L1 ratio       : {FED_L1_RATIO}")
print(f"Coefficient norm: {np.linalg.norm(final_coef_stable23):.6f}")

FINAL STABLE-23 FEDAVG TEST PREDICTIONS


,Trip_ID,Actual_Energy_kWh_per_km,Predicted_Energy_kWh_per_km,Absolute_Error,Signed_Error
0,0,0.173564,0.146354,0.027211,-0.027211
1,4,0.121370,0.140039,0.018669,0.018669
2,5,0.109203,0.142294,0.033090,0.033090
3,9,0.124348,0.127616,0.003269,0.003269
4,12,0.151130,0.149394,0.001735,-0.001735
5,16,0.158784,0.171182,0.012399,0.012399
6,25,0.095763,0.128754,0.032991,0.032991
7,28,0.108976,0.119626,0.010649,0.010649
8,38,0.184585,0.180180,0.004405,-0.004405
9,43,0.138392,0.151154,0.012762,0.012762



Final metrics
-------------
MAE  : 0.013600
RMSE : 0.017133
R²   : 0.740032

Final model
-----------
Features       : 23
Rounds         : 100
Local epochs   : 10
Learning rate  : 0.05
Alpha          : 0.001
L1 ratio       : 0.3
Coefficient norm: 0.025892


### Stable-23 centralized reference

The centralized benchmark uses the exact same 53/14 split as the Stable-23 FedAvg experiment. This creates a valid within-branch centralized-versus-FedAvg comparison.


In [163]:
# ============================================================
# STABLE-23: CENTRALIZED HOLDOUT BENCHMARK
# Exact same 53/14 split as federated experiment
# ============================================================

from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

central_stable23 = Pipeline([
    ("model", ElasticNet(
        alpha=FED_ALPHA,
        l1_ratio=FED_L1_RATIO,
        max_iter=10000
    ))
])

# ------------------------------------------------------------
# Train centralized model
# ------------------------------------------------------------

central_stable23.fit(
    X_train_stable23_scaled,
    y_train_stable23_np
)


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

central_train_pred = central_stable23.predict(
    X_train_stable23_scaled
)

central_test_pred = central_stable23.predict(
    X_test_stable23_scaled
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

central_train_mae = mean_absolute_error(
    y_train_stable23_np,
    central_train_pred
)

central_test_mae = mean_absolute_error(
    y_test_stable23_np,
    central_test_pred
)

central_test_rmse = np.sqrt(
    mean_squared_error(
        y_test_stable23_np,
        central_test_pred
    )
)

central_test_r2 = r2_score(
    y_test_stable23_np,
    central_test_pred
)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

central_coef = central_stable23.named_steps[
    "model"
].coef_

central_intercept = central_stable23.named_steps[
    "model"
].intercept_

central_nonzero = np.count_nonzero(
    np.abs(central_coef) > 1e-12
)


print("=" * 70)
print("STABLE-23 CENTRALIZED HOLDOUT BENCHMARK")
print("=" * 70)

print("\nTraining")
print("--------")
print(f"Train MAE : {central_train_mae:.6f}")

print("\nTest")
print("----")
print(f"Test MAE  : {central_test_mae:.6f}")
print(f"Test RMSE : {central_test_rmse:.6f}")
print(f"Test R²   : {central_test_r2:.6f}")

print("\nModel")
print("-----")
print(f"Features        : {n_features_stable23}")
print(f"Non-zero coeffs : {central_nonzero}/{n_features_stable23}")
print(f"Coefficient norm: {np.linalg.norm(central_coef):.6f}")
print(f"Intercept       : {central_intercept:.6f}")


# ------------------------------------------------------------
# Direct comparison with FedAvg
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CENTRALIZED vs FEDAVG")
print("=" * 70)

comparison = pd.DataFrame({
    "Model": [
        "Centralized Elastic Net",
        "Stable-23 FedAvg"
    ],
    "Features": [
        n_features_stable23,
        n_features_stable23
    ],
    "MAE": [
        central_test_mae,
        0.013600
    ],
    "RMSE": [
        central_test_rmse,
        0.017133
    ],
    "R2": [
        central_test_r2,
        0.740032
    ]
})

display(comparison)

STABLE-23 CENTRALIZED HOLDOUT BENCHMARK

Training
--------
Train MAE : 0.003871

Test
----
Test MAE  : 0.011506
Test RMSE : 0.013549
Test R²   : 0.837410

Model
-----
Features        : 23
Non-zero coeffs : 20/23
Coefficient norm: 0.028014
Intercept       : 0.169918

CENTRALIZED vs FEDAVG


,Model,Features,MAE,RMSE,R2
0,Centralized Elastic Net,23,0.011506,0.013549,0.837410
1,Stable-23 FedAvg,23,0.013600,0.017133,0.740032


## Stable-23 FedPer exploration

The following cells explore personalized Elastic Net models in the Stable-23 setting. Client-specific train/test splits yield only 2–3 test trips per client. Consequently, their metrics are local-validation diagnostics and are **not directly comparable** with the 14-trip global-test metrics reported for Stable-23 FedAvg and centralized Elastic Net.

The early failed initialization is retained as a debugging record; subsequent cells reset the parameters, reconstruct the split, and run the intended personalized experiment.


In [164]:
# ============================================================
# STABLE-23 FEDPER: INTERCEPT PERSONALIZATION
# ============================================================

FEDPER_ROUNDS = 100
FEDPER_LR = 0.05
FEDPER_LOCAL_EPOCHS = 10

# Personal intercept regularization.
# Start small so clients can adapt their baseline energy
# consumption without overwhelming the shared model.
FEDPER_PERSONAL_L2 = 0.01

# ------------------------------------------------------------
# Shared global parameters
# ------------------------------------------------------------

fedper_global_coef = np.zeros(
    n_features_stable23,
    dtype=float
)

fedper_global_intercept = 0.0


# ------------------------------------------------------------
# One personalized intercept per client
# ------------------------------------------------------------

fedper_personal_intercepts = {
    client_name: 0.0
    for client_name in client_data_stable23
}


# ------------------------------------------------------------
# Verify initialization
# ------------------------------------------------------------

print("=" * 70)
print("STABLE-23 FEDPER INITIALIZATION")
print("=" * 70)

print(f"Features             : {n_features_stable23}")
print(f"Rounds               : {FEDPER_ROUNDS}")
print(f"Learning rate        : {FEDPER_LR}")
print(f"Local epochs         : {FEDPER_LOCAL_EPOCHS}")
print(f"Alpha                : {FED_ALPHA}")
print(f"L1 ratio             : {FED_L1_RATIO}")
print(f"Personal intercept L2: {FEDPER_PERSONAL_L2}")

print("\nShared coefficients")
print("-------------------")
print("Shape:", fedper_global_coef.shape)
print("Norm :", np.linalg.norm(fedper_global_coef))

print("\nPersonal intercepts")
print("-------------------")
for client_name, value in fedper_personal_intercepts.items():
    print(f"{client_name}: {value:.6f}")

STABLE-23 FEDPER INITIALIZATION
Features             : 23
Rounds               : 100
Learning rate        : 0.05
Local epochs         : 10
Alpha                : 0.001
L1 ratio             : 0.3
Personal intercept L2: 0.01

Shared coefficients
-------------------
Shape: (23,)
Norm : 0.0

Personal intercepts
-------------------
C1: 0.000000
C2: 0.000000
C3: 0.000000
C4: 0.000000
C5: 0.000000


In [165]:
# ============================================================
# STABLE-23 FEDPER: ROUND 1 PILOT
# Personalized intercept + shared coefficients
# ============================================================

fedper_client_updates = []

print("=" * 70)
print("STABLE-23 FEDPER - ROUND 1")
print("=" * 70)

for client_name, data in client_data_stable23.items():

    # Start from the current shared model
    local_coef = fedper_global_coef.copy()

    # Start from this client's personal intercept
    local_intercept = fedper_personal_intercepts[client_name]

    X_client = data["X"]
    y_client = data["y"]

    n = X_client.shape[0]

    # --------------------------------------------------------
    # Local personalized training
    # --------------------------------------------------------

    for _ in range(FEDPER_LOCAL_EPOCHS):

        prediction = (
            X_client @ local_coef
            + local_intercept
        )

        error = prediction - y_client

        # Shared coefficient gradient
        grad_coef = (
            X_client.T @ error
        ) / n

        # Elastic Net L2 component
        grad_coef += (
            FED_ALPHA
            * (1 - FED_L1_RATIO)
            * local_coef
        )

        # Personal intercept gradient
        grad_intercept = error.mean()

        # Personal intercept regularization
        grad_intercept += (
            FEDPER_PERSONAL_L2
            * local_intercept
        )

        # Gradient step
        local_coef -= (
            FEDPER_LR * grad_coef
        )

        local_intercept -= (
            FEDPER_LR * grad_intercept
        )

        # Elastic Net L1 proximal step
        local_coef = soft_threshold(
            local_coef,
            FEDPER_LR
            * FED_ALPHA
            * FED_L1_RATIO
        )

    # --------------------------------------------------------
    # Local diagnostics
    # --------------------------------------------------------

    local_pred = (
        X_client @ local_coef
        + local_intercept
    )

    local_mae = np.mean(
        np.abs(local_pred - y_client)
    )

    fedper_client_updates.append({
        "client": client_name,
        "coef": local_coef,
        "intercept": local_intercept,
        "n_samples": n
    })

    print(
        f"{client_name}: "
        f"n={n}, "
        f"local MAE={local_mae:.6f}, "
        f"intercept={local_intercept:.6f}, "
        f"coef_norm={np.linalg.norm(local_coef):.6f}"
    )


# ------------------------------------------------------------
# Aggregate ONLY the shared coefficients
# ------------------------------------------------------------

total_samples = sum(
    u["n_samples"]
    for u in fedper_client_updates
)

fedper_global_coef_round1 = sum(
    u["coef"] * u["n_samples"]
    for u in fedper_client_updates
) / total_samples


# ------------------------------------------------------------
# Global intercept
# ------------------------------------------------------------

# Keep a shared global intercept as the weighted average.
fedper_global_intercept_round1 = sum(
    u["intercept"] * u["n_samples"]
    for u in fedper_client_updates
) / total_samples


# ------------------------------------------------------------
# Update personal intercepts
# ------------------------------------------------------------

for update in fedper_client_updates:
    fedper_personal_intercepts[
        update["client"]
    ] = update["intercept"]


# Update global shared parameters
fedper_global_coef = fedper_global_coef_round1
fedper_global_intercept = fedper_global_intercept_round1


# ------------------------------------------------------------
# Evaluate using personalized intercepts
# ------------------------------------------------------------

test_predictions = []
test_actual = []
test_client_labels = []

for client_name, data in client_data_stable23.items():

    # Determine which test trips belong to this client.
    client_test_ids = [
        trip_id
        for trip_id in test_idx
        if trip_id in client_trip_ids[client_name]
    ]

    if len(client_test_ids) == 0:
        continue

    test_positions = [
        list(test_idx).index(trip_id)
        for trip_id in client_test_ids
    ]

    X_client_test = X_test_stable23_scaled[
        test_positions
    ]

    y_client_test = y_test_stable23_np[
        test_positions
    ]

    personal_intercept = (
        fedper_personal_intercepts[client_name]
    )

    predictions = (
        X_client_test @ fedper_global_coef
        + fedper_global_intercept
        + personal_intercept
    )

    test_predictions.extend(predictions)
    test_actual.extend(y_client_test)
    test_client_labels.extend(
        [client_name] * len(y_client_test)
    )


# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

test_predictions = np.array(test_predictions)
test_actual = np.array(test_actual)

test_error = (
    test_predictions - test_actual
)

test_mae = np.mean(
    np.abs(test_error)
)

test_rmse = np.sqrt(
    np.mean(test_error ** 2)
)

test_r2 = 1 - (
    np.sum(test_error ** 2)
    /
    np.sum(
        (
            test_actual
            - test_actual.mean()
        ) ** 2
    )
)


print("\nAggregated FedPer Model")
print("-----------------------")
print(f"Test MAE  : {test_mae:.6f}")
print(f"Test RMSE : {test_rmse:.6f}")
print(f"Test R²   : {test_r2:.6f}")

print("\nPersonal intercepts")
print("-------------------")

for client_name, value in fedper_personal_intercepts.items():
    print(
        f"{client_name}: {value:.6f}"
    )

print("\nShared model diagnostics")
print("------------------------")
print(
    f"Coefficient norm: "
    f"{np.linalg.norm(fedper_global_coef):.6f}"
)

print(
    f"Coefficient min : "
    f"{fedper_global_coef.min():.6f}"
)

print(
    f"Coefficient max : "
    f"{fedper_global_coef.max():.6f}"
)

STABLE-23 FEDPER - ROUND 1
C1: n=11, local MAE=0.073936, intercept=0.059654, coef_norm=0.046897
C2: n=11, local MAE=0.063955, intercept=0.054111, coef_norm=0.048090
C3: n=11, local MAE=0.068192, intercept=0.056306, coef_norm=0.045861
C4: n=10, local MAE=0.069722, intercept=0.058032, coef_norm=0.049449
C5: n=10, local MAE=0.079781, intercept=0.060982, coef_norm=0.040956

Aggregated FedPer Model
-----------------------
Test MAE  : nan
Test RMSE : nan
Test R²   : nan

Personal intercepts
-------------------
C1: 0.059654
C2: 0.054111
C3: 0.056306
C4: 0.058032
C5: 0.060982

Shared model diagnostics
------------------------
Coefficient norm: 0.014131
Coefficient min : -0.006351
Coefficient max : 0.005888


In [169]:
# ============================================================
# STABLE-23 FEDPER: INITIALIZATION
# ============================================================

n_features = X_stable23_train_scaled.shape[1]

global_coef = np.zeros(n_features)
global_intercept = 0.0

personal_params = {}

for client_id in sorted(fedper_client_data.keys()):
    personal_params[client_id] = {
        "coef": np.zeros(n_features),
        "intercept": 0.0
    }

print("=" * 70)
print("STABLE-23 FEDPER INITIALIZATION")
print("=" * 70)

print(f"Number of features : {n_features}")
print(f"Number of clients  : {len(personal_params)}")
print(f"Global coefficient shape : {global_coef.shape}")

for client_id in sorted(fedper_client_data.keys()):
    print(
        f"{client_id}: "
        f"X_train={fedper_client_data[client_id]['X_train'].shape}, "
        f"X_test={fedper_client_data[client_id]['X_test'].shape}, "
        f"coef={personal_params[client_id]['coef'].shape}"
    )

NameError: name 'X_stable23_train_scaled' is not defined

In [170]:
# ============================================================
# STABLE-23 FEDPER: CLIENT-SPECIFIC TRAIN / TEST SPLITS
# ============================================================

from sklearn.model_selection import train_test_split

FEDPER_CLIENT_TEST_SIZE = 0.20
FEDPER_SPLIT_RANDOM_STATE = 42

fedper_client_data = {}

print("=" * 70)
print("STABLE-23 FEDPER CLIENT SPLITS")
print("=" * 70)

for client_id in sorted(client_data_stable23.keys()):

    data = client_data_stable23[client_id]

    X_client = data["X"]
    y_client = data["y"]
    trip_ids = data["trip_ids"]

    # --------------------------------------------------------
    # Local train/test split
    # --------------------------------------------------------

    local_positions = np.arange(
        len(y_client)
    )

    train_positions, test_positions = train_test_split(
        local_positions,
        test_size=FEDPER_CLIENT_TEST_SIZE,
        random_state=FEDPER_SPLIT_RANDOM_STATE
    )

    # Sort positions for reproducibility/readability
    train_positions = np.sort(train_positions)
    test_positions = np.sort(test_positions)

    # --------------------------------------------------------
    # Store client-specific data
    # --------------------------------------------------------

    fedper_client_data[client_id] = {

        "X_train": X_client[train_positions],
        "y_train": y_client[train_positions],

        "X_test": X_client[test_positions],
        "y_test": y_client[test_positions],

        "train_trip_ids": [
            trip_ids[i]
            for i in train_positions
        ],

        "test_trip_ids": [
            trip_ids[i]
            for i in test_positions
        ]
    }


# ------------------------------------------------------------
# Validation / inspection
# ------------------------------------------------------------

total_train = 0
total_test = 0

all_train_ids = []
all_test_ids = []

print()

for client_id in sorted(fedper_client_data.keys()):

    data = fedper_client_data[client_id]

    n_train = len(data["y_train"])
    n_test = len(data["y_test"])

    total_train += n_train
    total_test += n_test

    all_train_ids.extend(
        data["train_trip_ids"]
    )

    all_test_ids.extend(
        data["test_trip_ids"]
    )

    print(
        f"{client_id}: "
        f"Train={n_train}, "
        f"Test={n_test} | "
        f"Train trips={data['train_trip_ids']} | "
        f"Test trips={data['test_trip_ids']}"
    )


print("\n" + "=" * 70)
print("FEDPER PARTITION VALIDATION")
print("=" * 70)

print(
    f"Total client-local training trips: {total_train}"
)

print(
    f"Total client-local test trips    : {total_test}"
)

print(
    f"Unique local train trips         : "
    f"{len(set(all_train_ids))}"
)

print(
    f"Unique local test trips          : "
    f"{len(set(all_test_ids))}"
)

print(
    f"Train/test overlap               : "
    f"{len(set(all_train_ids).intersection(all_test_ids))}"
)

print(
    f"Total unique trips               : "
    f"{len(set(all_train_ids).union(all_test_ids))}"
)

STABLE-23 FEDPER CLIENT SPLITS

C1: Train=8, Test=3 | Train trips=[32, 13, 8, 50, 15, 53, 52, 63] | Test trips=[19, 6, 46]
C2: Train=8, Test=3 | Train trips=[45, 69, 33, 66, 42, 23, 22, 14] | Test trips=[7, 59, 20]
C3: Train=8, Test=3 | Train trips=[57, 47, 49, 17, 27, 26, 24, 10] | Test trips=[65, 34, 61]
C4: Train=8, Test=2 | Train trips=[51, 29, 21, 2, 37, 18, 60, 54] | Test trips=[56, 41]
C5: Train=8, Test=2 | Train trips=[36, 31, 3, 68, 11, 39, 58, 55] | Test trips=[44, 1]

FEDPER PARTITION VALIDATION
Total client-local training trips: 40
Total client-local test trips    : 13
Unique local train trips         : 40
Unique local test trips          : 13
Train/test overlap               : 0
Total unique trips               : 53


In [171]:
# ============================================================
# STABLE-23 FEDPER: LOCAL PERSONALIZED ELASTIC NET
# ============================================================

FEDPER_ALPHA = 0.001
FEDPER_L1_RATIO = 0.3
FEDPER_LEARNING_RATE = 0.05
FEDPER_PERSONALIZATION_LAMBDA = 0.01
FEDPER_LOCAL_EPOCHS = 3


def local_fedper_elastic_net(
    X,
    y,
    global_coef,
    global_intercept,
    personal_coef,
    personal_intercept,
    alpha=FEDPER_ALPHA,
    l1_ratio=FEDPER_L1_RATIO,
    learning_rate=FEDPER_LEARNING_RATE,
    personalization_lambda=FEDPER_PERSONALIZATION_LAMBDA,
    epochs=FEDPER_LOCAL_EPOCHS
):
    """
    Local FedPer-style personalized Elastic Net.

    Shared parameters:
        global_coef
        global_intercept

    Personalized parameters:
        personal_coef
        personal_intercept

    The local model is initialized from the client's
    previous personalized parameters.

    A personalization penalty keeps the client's parameters
    close to the shared/global parameters.

    Elastic Net regularization is applied to the
    personalized coefficients.
    """

    beta = personal_coef.copy()
    bias = float(personal_intercept)

    for _ in range(epochs):

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        prediction = (
            X @ beta
            + bias
        )

        error = prediction - y

        n = X.shape[0]

        # ----------------------------------------------------
        # Gradient for personalized coefficients
        # ----------------------------------------------------

        grad_beta = (
            X.T @ error
        ) / n

        # Elastic Net L2 component
        grad_beta += (
            alpha
            * (1 - l1_ratio)
            * beta
        )

        # Personalization pull toward global model
        grad_beta += (
            personalization_lambda
            * (beta - global_coef)
        )

        # ----------------------------------------------------
        # Gradient for personalized intercept
        # ----------------------------------------------------

        grad_bias = error.mean()

        # Personalization pull toward global intercept
        grad_bias += (
            personalization_lambda
            * (bias - global_intercept)
        )

        # ----------------------------------------------------
        # Gradient update
        # ----------------------------------------------------

        beta -= (
            learning_rate
            * grad_beta
        )

        bias -= (
            learning_rate
            * grad_bias
        )

        # ----------------------------------------------------
        # Elastic Net L1 proximal step
        # ----------------------------------------------------

        beta = soft_threshold(
            beta,
            learning_rate
            * alpha
            * l1_ratio
        )

    return beta, bias


print("✓ FedPer personalized Elastic Net optimizer defined")

print("\nConfiguration")
print("-------------")
print(f"Alpha                  : {FEDPER_ALPHA}")
print(f"L1 ratio               : {FEDPER_L1_RATIO}")
print(f"Learning rate          : {FEDPER_LEARNING_RATE}")
print(f"Personalization lambda : {FEDPER_PERSONALIZATION_LAMBDA}")
print(f"Local epochs           : {FEDPER_LOCAL_EPOCHS}")

✓ FedPer personalized Elastic Net optimizer defined

Configuration
-------------
Alpha                  : 0.001
L1 ratio               : 0.3
Learning rate          : 0.05
Personalization lambda : 0.01
Local epochs           : 3


In [173]:
# ============================================================
# STABLE-23 FEDPER: RESET MODEL PARAMETERS
# ============================================================

# Get feature count directly from the FedPer client data
n_features = fedper_client_data["C1"]["X_train"].shape[1]

print("=" * 70)
print("STABLE-23 FEDPER PARAMETER RESET")
print("=" * 70)

print(f"FedPer feature count: {n_features}")

# Safety check
assert n_features == 23, (
    f"Expected 23 Stable-23 features, got {n_features}"
)

# ------------------------------------------------------------
# Global/shared model
# ------------------------------------------------------------

global_coef = np.zeros(n_features)
global_intercept = 0.0

# ------------------------------------------------------------
# Personalized model for each client
# ------------------------------------------------------------

personal_params = {}

for client_id in sorted(fedper_client_data.keys()):

    X_local = fedper_client_data[client_id]["X_train"]

    assert X_local.shape[1] == n_features

    personal_params[client_id] = {
        "coef": np.zeros(n_features),
        "intercept": 0.0
    }

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nParameter shapes:")

for client_id in sorted(personal_params.keys()):
    print(
        f"{client_id}: "
        f"X={fedper_client_data[client_id]['X_train'].shape}, "
        f"coef={personal_params[client_id]['coef'].shape}"
    )

print("\nGlobal coefficient shape:", global_coef.shape)
print("Global intercept:", global_intercept)

print("\n✓ FedPer parameters successfully reset for Stable-23")

STABLE-23 FEDPER PARAMETER RESET
FedPer feature count: 23

Parameter shapes:
C1: X=(8, 23), coef=(23,)
C2: X=(8, 23), coef=(23,)
C3: X=(8, 23), coef=(23,)
C4: X=(8, 23), coef=(23,)
C5: X=(8, 23), coef=(23,)

Global coefficient shape: (23,)
Global intercept: 0.0

✓ FedPer parameters successfully reset for Stable-23


In [174]:
# ============================================================
# STABLE-23 FEDPER: ROUND 1
# ============================================================

client_models = []

print("=" * 70)
print("STABLE-23 FEDPER - ROUND 1 LOCAL TRAINING")
print("=" * 70)

for client_id in sorted(fedper_client_data.keys()):

    data = fedper_client_data[client_id]

    X_local = data["X_train"]
    y_local = data["y_train"]

    personal_coef = personal_params[client_id]["coef"]
    personal_intercept = personal_params[client_id]["intercept"]

    updated_coef, updated_intercept = local_fedper_elastic_net(
        X_local,
        y_local,
        global_coef,
        global_intercept,
        personal_coef,
        personal_intercept,
        alpha=FEDPER_ALPHA,
        l1_ratio=FEDPER_L1_RATIO,
        learning_rate=FEDPER_LEARNING_RATE,
        personalization_lambda=FEDPER_PERSONALIZATION_LAMBDA,
        epochs=FEDPER_LOCAL_EPOCHS
    )

    # Store personalized model
    personal_params[client_id]["coef"] = updated_coef
    personal_params[client_id]["intercept"] = updated_intercept

    # Local training prediction
    train_pred = (
        X_local @ updated_coef
        + updated_intercept
    )

    train_metrics = regression_metrics(
        y_local,
        train_pred
    )

    # Model sent to server
    client_models.append(
        (
            updated_coef,
            updated_intercept,
            len(y_local)
        )
    )

    print(
        f"{client_id}: "
        f"Train MAE={train_metrics['MAE']:.6f}, "
        f"RMSE={train_metrics['RMSE']:.6f}, "
        f"R²={train_metrics['R2']:.6f}, "
        f"CoefNorm={np.linalg.norm(updated_coef):.6f}, "
        f"Intercept={updated_intercept:.6f}"
    )


# ============================================================
# SERVER-SIDE FEDAVG OF PERSONALIZED CLIENT MODELS
# ============================================================

global_coef, global_intercept = fedavg_aggregate(
    client_models
)


# ============================================================
# PERSONALIZED LOCAL TEST EVALUATION
# ============================================================

all_predictions = []
all_actual = []
client_metrics = []

print("\n" + "=" * 70)
print("ROUND 1 PERSONALIZED TEST PERFORMANCE")
print("=" * 70)

for client_id in sorted(fedper_client_data.keys()):

    data = fedper_client_data[client_id]

    X_local_test = data["X_test"]
    y_local_test = data["y_test"]

    personal_coef = personal_params[client_id]["coef"]
    personal_intercept = personal_params[client_id]["intercept"]

    prediction = (
        X_local_test @ personal_coef
        + personal_intercept
    )

    metrics = regression_metrics(
        y_local_test,
        prediction
    )

    client_metrics.append({
        "Client": client_id,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "Test Samples": len(y_local_test)
    })

    all_predictions.extend(prediction)
    all_actual.extend(y_local_test)

    print(
        f"{client_id}: "
        f"MAE={metrics['MAE']:.6f}, "
        f"RMSE={metrics['RMSE']:.6f}, "
        f"R²={metrics['R2']:.6f}, "
        f"Test={len(y_local_test)}"
    )


# ============================================================
# OVERALL PERSONALIZED PERFORMANCE
# ============================================================

overall_metrics = regression_metrics(
    np.array(all_actual),
    np.array(all_predictions)
)

print("\n" + "=" * 70)
print("ROUND 1 OVERALL FEDPER")
print("=" * 70)

print(f"MAE  : {overall_metrics['MAE']:.6f}")
print(f"RMSE : {overall_metrics['RMSE']:.6f}")
print(f"R²   : {overall_metrics['R2']:.6f}")

print(f"\nGlobal coefficient norm : {np.linalg.norm(global_coef):.6f}")
print(f"Global intercept        : {global_intercept:.6f}")
print(f"Global nonzero features: {np.count_nonzero(global_coef)}")

STABLE-23 FEDPER - ROUND 1 LOCAL TRAINING
C1: Train MAE=0.115342, RMSE=0.117757, R²=-10.802984, CoefNorm=0.028000, Intercept=0.022424
C2: Train MAE=0.108552, RMSE=0.114787, R²=-8.326376, CoefNorm=0.030779, Intercept=0.022088
C3: Train MAE=0.093992, RMSE=0.101443, R²=-13.303301, CoefNorm=0.028940, Intercept=0.019936
C4: Train MAE=0.119460, RMSE=0.124139, R²=-10.000399, CoefNorm=0.025500, Intercept=0.022638
C5: Train MAE=0.127487, RMSE=0.130020, R²=-33.437209, CoefNorm=0.025686, Intercept=0.024003

ROUND 1 PERSONALIZED TEST PERFORMANCE
C1: MAE=0.175729, RMSE=0.184896, R²=-48.255383, Test=3
C2: MAE=0.165822, RMSE=0.166070, R²=-54.538055, Test=3
C3: MAE=0.200459, RMSE=0.203929, R²=-75.468944, Test=3
C4: MAE=0.179672, RMSE=0.180255, R²=-321.247724, Test=2
C5: MAE=0.148392, RMSE=0.153526, R²=-73.577919, Test=2

ROUND 1 OVERALL FEDPER
MAE  : 0.175551
RMSE : 0.180210
R²   : -34.022032

Global coefficient norm : 0.010230
Global intercept        : 0.022218
Global nonzero features: 23


In [175]:
# ============================================================
# STABLE-23 FEDPER: WARM-START FROM CONVERGED FEDAVG
# ============================================================

# IMPORTANT:
# These should be the final parameters from the
# 100-round Stable-23 FedAvg experiment.

fedper_global_coef = global_coef.copy()
fedper_global_intercept = float(global_intercept)

print("=" * 70)
print("STABLE-23 FEDPER WARM START")
print("=" * 70)

print(f"Features              : {len(fedper_global_coef)}")
print(f"Global coefficient norm: {np.linalg.norm(fedper_global_coef):.6f}")
print(f"Global intercept       : {fedper_global_intercept:.6f}")
print(f"Nonzero coefficients   : {np.count_nonzero(fedper_global_coef)}")

assert len(fedper_global_coef) == 23

# ------------------------------------------------------------
# Initialize every client's personalized model
# from the converged shared model.
# ------------------------------------------------------------

personal_params = {}

for client_id in sorted(fedper_client_data.keys()):

    personal_params[client_id] = {
        "coef": fedper_global_coef.copy(),
        "intercept": fedper_global_intercept
    }

print("\nPersonalized model initialization:")

for client_id in sorted(personal_params.keys()):
    print(
        f"{client_id}: "
        f"coef={personal_params[client_id]['coef'].shape}, "
        f"norm={np.linalg.norm(personal_params[client_id]['coef']):.6f}, "
        f"intercept={personal_params[client_id]['intercept']:.6f}"
    )

print("\n✓ All clients initialized from converged FedAvg model")

STABLE-23 FEDPER WARM START
Features              : 23
Global coefficient norm: 0.010230
Global intercept       : 0.022218
Nonzero coefficients   : 23

Personalized model initialization:
C1: coef=(23,), norm=0.010230, intercept=0.022218
C2: coef=(23,), norm=0.010230, intercept=0.022218
C3: coef=(23,), norm=0.010230, intercept=0.022218
C4: coef=(23,), norm=0.010230, intercept=0.022218
C5: coef=(23,), norm=0.010230, intercept=0.022218

✓ All clients initialized from converged FedAvg model


In [178]:
# ============================================================
# REBUILD STABLE-23 GLOBAL TRAIN / TEST SPLIT
# USING INDICES, NOT THE OVERWRITTEN trip_ids VARIABLE
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Stable-23 data
# ------------------------------------------------------------

X_stable23 = X[STABLE_23_FEATURES].copy()
y_stable23 = np.asarray(y).copy()

assert X_stable23.shape == (67, 23)
assert len(y_stable23) == 67

# Use row positions as the split identifier
all_positions = np.arange(len(X_stable23))

train_pos, test_pos = train_test_split(
    all_positions,
    test_size=0.20,
    random_state=42
)

# ------------------------------------------------------------
# Create train/test arrays
# ------------------------------------------------------------

X_stable23_train = X_stable23.iloc[train_pos]
X_stable23_test = X_stable23.iloc[test_pos]

y_train_stable23 = y_stable23[train_pos]
y_test_stable23 = y_stable23[test_pos]

# ------------------------------------------------------------
# Scale using TRAINING DATA ONLY
# ------------------------------------------------------------

stable23_scaler = StandardScaler()

X_stable23_train_scaled = stable23_scaler.fit_transform(
    X_stable23_train
)

X_stable23_test_scaled = stable23_scaler.transform(
    X_stable23_test
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("STABLE-23 GLOBAL SPLIT REBUILT")
print("=" * 70)

print("Full data :", X_stable23.shape)
print("Train     :", X_stable23_train_scaled.shape)
print("Test      :", X_stable23_test_scaled.shape)

print("\nTrain positions:", len(train_pos))
print("Test positions :", len(test_pos))
print("Overlap        :", len(set(train_pos) & set(test_pos)))

print("\nTrain mean:", np.mean(X_stable23_train_scaled))
print("Train std :", np.std(X_stable23_train_scaled))

print(
    "\nMax |train scaled value|:",
    np.max(np.abs(X_stable23_train_scaled))
)

print(
    "Max |test scaled value| :",
    np.max(np.abs(X_stable23_test_scaled))
)

assert len(train_pos) == 53
assert len(test_pos) == 14
assert len(set(train_pos) & set(test_pos)) == 0

print("\n✓ 53/14 Stable-23 split restored")

STABLE-23 GLOBAL SPLIT REBUILT
Full data : (67, 23)
Train     : (53, 23)
Test      : (14, 23)

Train positions: 53
Test positions : 14
Overlap        : 0

Train mean: 3.424477910246586e-17
Train std : 1.0

Max |train scaled value|: 6.0274005457546025
Max |test scaled value| : 5.745878228492991

✓ 53/14 Stable-23 split restored


### Personalized-model checkpoints

Warm-started FedPer results are recorded at multiple communication rounds. Since the local training metrics improve while local held-out metrics worsen across later checkpoints, these cells provide evidence of overfitting under the very small per-client samples rather than evidence that additional rounds improve personalization.


In [180]:
# ============================================================
# STABLE-23 FEDPER
# CLIENT-SPECIFIC PERSONALIZATION EXPERIMENT
# ============================================================

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

FEDPER_ROUNDS = 100
FEDPER_LOCAL_EPOCHS = 10
FEDPER_LEARNING_RATE = 0.05
FEDPER_ALPHA = 0.001
FEDPER_L1_RATIO = 0.3
FEDPER_PERSONALIZATION_LAMBDA = 0.01

# ------------------------------------------------------------
# IMPORTANT:
# Start from the converged Stable-23 FedAvg model
# ------------------------------------------------------------

global_coef = stable23_fedavg_coef.copy()
global_intercept = float(stable23_fedavg_intercept)

n_features = len(global_coef)

assert n_features == 23

# Each client starts from the shared FedAvg model
personal_params = {}

for client_id in sorted(fedper_client_data.keys()):

    personal_params[client_id] = {
        "coef": global_coef.copy(),
        "intercept": global_intercept
    }


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

fedper_history = []
fedper_client_history = []

print("=" * 70)
print("STABLE-23 FEDPER")
print("=" * 70)

print(f"Features              : {n_features}")
print(f"Clients               : {len(personal_params)}")
print(f"Rounds                : {FEDPER_ROUNDS}")
print(f"Local epochs          : {FEDPER_LOCAL_EPOCHS}")
print(f"Learning rate         : {FEDPER_LEARNING_RATE}")
print(f"Alpha                 : {FEDPER_ALPHA}")
print(f"L1 ratio              : {FEDPER_L1_RATIO}")
print(f"Personalization lambda: {FEDPER_PERSONALIZATION_LAMBDA}")

print("\nInitial global coefficient norm:",
      np.linalg.norm(global_coef))

print("Initial global intercept:",
      global_intercept)


# ============================================================
# FEDPER TRAINING
# ============================================================

for round_num in range(1, FEDPER_ROUNDS + 1):

    client_models = []

    # --------------------------------------------------------
    # LOCAL CLIENT TRAINING
    # --------------------------------------------------------

    for client_id in sorted(fedper_client_data.keys()):

        data = fedper_client_data[client_id]

        X_local = data["X_train"]
        y_local = data["y_train"]

        personal_coef = personal_params[client_id]["coef"]
        personal_intercept = personal_params[client_id]["intercept"]

        # Personalized local update
        updated_coef, updated_intercept = (
            local_fedper_elastic_net(
                X_local,
                y_local,
                global_coef,
                global_intercept,
                personal_coef,
                personal_intercept,
                alpha=FEDPER_ALPHA,
                l1_ratio=FEDPER_L1_RATIO,
                learning_rate=FEDPER_LEARNING_RATE,
                personalization_lambda=FEDPER_PERSONALIZATION_LAMBDA,
                epochs=FEDPER_LOCAL_EPOCHS
            )
        )

        # Store personalized parameters
        personal_params[client_id]["coef"] = updated_coef
        personal_params[client_id]["intercept"] = updated_intercept

        # Send personalized model to server
        client_models.append(
            (
                updated_coef,
                updated_intercept,
                len(y_local)
            )
        )

    # --------------------------------------------------------
    # SERVER AGGREGATION
    # --------------------------------------------------------

    global_coef, global_intercept = fedavg_aggregate(
        client_models
    )

    # --------------------------------------------------------
    # PERSONALIZED CLIENT TESTING
    # --------------------------------------------------------

    all_predictions = []
    all_actual = []
    client_round_metrics = []

    for client_id in sorted(fedper_client_data.keys()):

        data = fedper_client_data[client_id]

        X_local_test = data["X_test"]
        y_local_test = data["y_test"]

        personal_coef = personal_params[client_id]["coef"]
        personal_intercept = personal_params[client_id]["intercept"]

        prediction = (
            X_local_test @ personal_coef
            + personal_intercept
        )

        metrics = regression_metrics(
            y_local_test,
            prediction
        )

        client_round_metrics.append({
            "Round": round_num,
            "Client": client_id,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "Test Samples": len(y_local_test),
            "Personal Coef Norm": np.linalg.norm(personal_coef),
            "Personal Intercept": personal_intercept
        })

        all_predictions.extend(prediction)
        all_actual.extend(y_local_test)

    # --------------------------------------------------------
    # OVERALL PERSONALIZED PERFORMANCE
    # --------------------------------------------------------

    overall_metrics = regression_metrics(
        np.asarray(all_actual),
        np.asarray(all_predictions)
    )

    client_maes = [
        row["MAE"]
        for row in client_round_metrics
    ]

    fedper_history.append({
        "Round": round_num,
        "Overall MAE": overall_metrics["MAE"],
        "Overall RMSE": overall_metrics["RMSE"],
        "Overall R2": overall_metrics["R2"],
        "Mean Client MAE": np.mean(client_maes),
        "Std Client MAE": np.std(client_maes),
        "Worst Client MAE": np.max(client_maes),
        "Global Coefficient Norm": np.linalg.norm(global_coef),
        "Global Intercept": global_intercept,
        "Global Nonzero": np.count_nonzero(global_coef),
        "Mean Personal Coef Norm": np.mean([
            np.linalg.norm(
                personal_params[c]["coef"]
            )
            for c in personal_params
        ])
    })

    fedper_client_history.extend(client_round_metrics)

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if (
        round_num == 1
        or round_num % 10 == 0
        or round_num == FEDPER_ROUNDS
    ):

        print(
            f"Round {round_num:3d}: "
            f"MAE={overall_metrics['MAE']:.6f} | "
            f"RMSE={overall_metrics['RMSE']:.6f} | "
            f"R²={overall_metrics['R2']:.6f} | "
            f"Mean Client MAE={np.mean(client_maes):.6f}"
        )


# ============================================================
# FINAL RESULTS
# ============================================================

fedper_history_df = pd.DataFrame(
    fedper_history
)

fedper_client_history_df = pd.DataFrame(
    fedper_client_history
)

final_fedper = fedper_history_df.iloc[-1]

print("\n" + "=" * 70)
print("FINAL STABLE-23 FEDPER RESULTS")
print("=" * 70)

print(
    f"Overall MAE  : "
    f"{final_fedper['Overall MAE']:.6f}"
)

print(
    f"Overall RMSE : "
    f"{final_fedper['Overall RMSE']:.6f}"
)

print(
    f"Overall R²   : "
    f"{final_fedper['Overall R2']:.6f}"
)

print(
    f"Mean Client MAE : "
    f"{final_fedper['Mean Client MAE']:.6f}"
)

print(
    f"Worst Client MAE: "
    f"{final_fedper['Worst Client MAE']:.6f}"
)

print(
    f"Global Coef Norm: "
    f"{final_fedper['Global Coefficient Norm']:.6f}"
)

print(
    f"Mean Personal Coef Norm: "
    f"{final_fedper['Mean Personal Coef Norm']:.6f}"
)


# ============================================================
# FINAL CLIENT-BY-CLIENT RESULTS
# ============================================================

final_client_results = (
    fedper_client_history_df[
        fedper_client_history_df["Round"] == FEDPER_ROUNDS
    ]
    .copy()
)

print("\n" + "=" * 70)
print("FINAL CLIENT-SPECIFIC RESULTS")
print("=" * 70)

print(
    final_client_results[
        [
            "Client",
            "MAE",
            "RMSE",
            "R2",
            "Test Samples",
            "Personal Coef Norm",
            "Personal Intercept"
        ]
    ].to_string(index=False)
)

print("\n✓ Stable-23 FedPer experiment completed")

STABLE-23 FEDPER
Features              : 23
Clients               : 5
Rounds                : 100
Local epochs          : 10
Learning rate         : 0.05
Alpha                 : 0.001
L1 ratio              : 0.3
Personalization lambda: 0.01

Initial global coefficient norm: 0.026500191674142658
Initial global intercept: 0.1698629028185313
Round   1: MAE=0.005181 | RMSE=0.006795 | R²=0.950203 | Mean Client MAE=0.005037
Round  10: MAE=0.008185 | RMSE=0.010289 | R²=0.885838 | Mean Client MAE=0.008303
Round  20: MAE=0.009156 | RMSE=0.011650 | R²=0.853644 | Mean Client MAE=0.009377
Round  30: MAE=0.009562 | RMSE=0.012396 | R²=0.834278 | Mean Client MAE=0.009822
Round  40: MAE=0.010019 | RMSE=0.012967 | R²=0.818661 | Mean Client MAE=0.010313
Round  50: MAE=0.010441 | RMSE=0.013414 | R²=0.805965 | Mean Client MAE=0.010794
Round  60: MAE=0.010770 | RMSE=0.013776 | R²=0.795347 | Mean Client MAE=0.011160
Round  70: MAE=0.011022 | RMSE=0.014119 | R²=0.785036 | Mean Client MAE=0.011421
Round  80: 

In [181]:
# ============================================================
# FIND BEST FEDPER ROUND
# ============================================================

fedper_history_df = pd.DataFrame(fedper_history)

best_mae_row = fedper_history_df.loc[
    fedper_history_df["Overall MAE"].idxmin()
]

best_rmse_row = fedper_history_df.loc[
    fedper_history_df["Overall RMSE"].idxmin()
]

best_r2_row = fedper_history_df.loc[
    fedper_history_df["Overall R2"].idxmax()
]

print("=" * 70)
print("BEST FEDPER ROUNDS")
print("=" * 70)

print(
    f"Best MAE : Round {int(best_mae_row['Round'])} | "
    f"MAE={best_mae_row['Overall MAE']:.6f} | "
    f"RMSE={best_mae_row['Overall RMSE']:.6f} | "
    f"R²={best_mae_row['Overall R2']:.6f}"
)

print(
    f"Best RMSE: Round {int(best_rmse_row['Round'])} | "
    f"MAE={best_rmse_row['Overall MAE']:.6f} | "
    f"RMSE={best_rmse_row['Overall RMSE']:.6f} | "
    f"R²={best_rmse_row['Overall R2']:.6f}"
)

print(
    f"Best R²  : Round {int(best_r2_row['Round'])} | "
    f"MAE={best_r2_row['Overall MAE']:.6f} | "
    f"RMSE={best_r2_row['Overall RMSE']:.6f} | "
    f"R²={best_r2_row['Overall R2']:.6f}"
)

BEST FEDPER ROUNDS
Best MAE : Round 1 | MAE=0.005181 | RMSE=0.006795 | R²=0.950203
Best RMSE: Round 1 | MAE=0.005181 | RMSE=0.006795 | R²=0.950203
Best R²  : Round 1 | MAE=0.005181 | RMSE=0.006795 | R²=0.950203


In [183]:
# ============================================================
# STABLE-23 FEDPER
# STORE PERSONALIZED MODELS AT CHECKPOINTS
# ============================================================

FEDPER_CHECKPOINTS = [1, 10, 30, 50, 100]

# Start from the converged Stable-23 FedAvg model
global_coef = stable23_fedavg_coef.copy()
global_intercept = float(stable23_fedavg_intercept)

# Initialize each client's personalized model
personal_params = {}

for client_id in sorted(fedper_client_data.keys()):
    personal_params[client_id] = {
        "coef": global_coef.copy(),
        "intercept": global_intercept
    }

# Storage
fedper_checkpoint_models = {}
fedper_checkpoint_metrics = []

print("=" * 70)
print("STABLE-23 FEDPER CHECKPOINT RUN")
print("=" * 70)

for round_num in range(1, 101):

    client_models = []

    # --------------------------------------------------------
    # LOCAL PERSONALIZED TRAINING
    # --------------------------------------------------------

    for client_id in sorted(fedper_client_data.keys()):

        data = fedper_client_data[client_id]

        X_local = data["X_train"]
        y_local = data["y_train"]

        personal_coef = personal_params[client_id]["coef"]
        personal_intercept = personal_params[client_id]["intercept"]

        updated_coef, updated_intercept = (
            local_fedper_elastic_net(
                X_local,
                y_local,
                global_coef,
                global_intercept,
                personal_coef,
                personal_intercept,
                alpha=FEDPER_ALPHA,
                l1_ratio=FEDPER_L1_RATIO,
                learning_rate=FEDPER_LEARNING_RATE,
                personalization_lambda=FEDPER_PERSONALIZATION_LAMBDA,
                epochs=FEDPER_LOCAL_EPOCHS
            )
        )

        personal_params[client_id]["coef"] = updated_coef
        personal_params[client_id]["intercept"] = updated_intercept

        client_models.append(
            (
                updated_coef,
                updated_intercept,
                len(y_local)
            )
        )

    # --------------------------------------------------------
    # GLOBAL AGGREGATION
    # --------------------------------------------------------

    global_coef, global_intercept = fedavg_aggregate(
        client_models
    )

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    if round_num in FEDPER_CHECKPOINTS:

        checkpoint_models = {}

        all_predictions = []
        all_actual = []

        for client_id in sorted(fedper_client_data.keys()):

            data = fedper_client_data[client_id]

            coef = personal_params[client_id]["coef"].copy()
            intercept = personal_params[client_id]["intercept"]

            # Save exact personalized model
            checkpoint_models[client_id] = {
                "coef": coef,
                "intercept": intercept
            }

            # Local test prediction
            prediction = (
                data["X_test"] @ coef
                + intercept
            )

            all_predictions.extend(prediction)
            all_actual.extend(data["y_test"])

        metrics = regression_metrics(
            np.asarray(all_actual),
            np.asarray(all_predictions)
        )

        fedper_checkpoint_models[round_num] = checkpoint_models

        fedper_checkpoint_metrics.append({
            "Round": round_num,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"]
        })

        print(
            f"Round {round_num:3d}: "
            f"MAE={metrics['MAE']:.6f} | "
            f"RMSE={metrics['RMSE']:.6f} | "
            f"R²={metrics['R2']:.6f}"
        )


fedper_checkpoint_metrics_df = pd.DataFrame(
    fedper_checkpoint_metrics
)

print("\n✓ Checkpoint models saved")

STABLE-23 FEDPER CHECKPOINT RUN
Round   1: MAE=0.005181 | RMSE=0.006795 | R²=0.950203
Round  10: MAE=0.008185 | RMSE=0.010289 | R²=0.885838
Round  30: MAE=0.009562 | RMSE=0.012396 | R²=0.834278
Round  50: MAE=0.010441 | RMSE=0.013414 | R²=0.805965
Round 100: MAE=0.011808 | RMSE=0.015176 | R²=0.751626

✓ Checkpoint models saved


In [184]:
# ============================================================
# FEDPER: TRAINING VS TEST PERFORMANCE
# ============================================================

comparison_rows = []

for round_num in FEDPER_CHECKPOINTS:

    checkpoint_models = fedper_checkpoint_models[round_num]

    train_actual = []
    train_pred = []

    test_actual = []
    test_pred = []

    for client_id in sorted(fedper_client_data.keys()):

        data = fedper_client_data[client_id]

        coef = checkpoint_models[client_id]["coef"]
        intercept = checkpoint_models[client_id]["intercept"]

        # ----------------------------------------------------
        # Local training predictions
        # ----------------------------------------------------

        pred_train = (
            data["X_train"] @ coef
            + intercept
        )

        train_actual.extend(data["y_train"])
        train_pred.extend(pred_train)

        # ----------------------------------------------------
        # Local test predictions
        # ----------------------------------------------------

        pred_test = (
            data["X_test"] @ coef
            + intercept
        )

        test_actual.extend(data["y_test"])
        test_pred.extend(pred_test)

    train_metrics = regression_metrics(
        np.asarray(train_actual),
        np.asarray(train_pred)
    )

    test_metrics = regression_metrics(
        np.asarray(test_actual),
        np.asarray(test_pred)
    )

    comparison_rows.append({
        "Round": round_num,

        "Train MAE": train_metrics["MAE"],
        "Train RMSE": train_metrics["RMSE"],
        "Train R2": train_metrics["R2"],

        "Test MAE": test_metrics["MAE"],
        "Test RMSE": test_metrics["RMSE"],
        "Test R2": test_metrics["R2"]
    })


fedper_train_test_df = pd.DataFrame(
    comparison_rows
)

print("=" * 70)
print("FEDPER TRAIN VS LOCAL TEST PERFORMANCE")
print("=" * 70)

print(
    fedper_train_test_df.to_string(index=False)
)

FEDPER TRAIN VS LOCAL TEST PERFORMANCE
 Round  Train MAE  Train RMSE  Train R2  Test MAE  Test RMSE  Test R2
     1   0.002510    0.003318  0.989839  0.005181   0.006795 0.950203
    10   0.001003    0.001216  0.998637  0.008185   0.010289 0.885838
    30   0.000828    0.001010  0.999058  0.009562   0.012396 0.834278
    50   0.000810    0.000974  0.999126  0.010441   0.013414 0.805965
   100   0.000819    0.001016  0.999047  0.011808   0.015176 0.751626


In [185]:
# ============================================================
# FEDPER: ROUND 1 VS ROUND 100 CLIENT COMPARISON
# ============================================================

def evaluate_fedper_checkpoint(round_num):
    checkpoint_models = fedper_checkpoint_models[round_num]

    rows = []

    for client_id in sorted(fedper_client_data.keys()):

        data = fedper_client_data[client_id]

        coef = checkpoint_models[client_id]["coef"]
        intercept = checkpoint_models[client_id]["intercept"]

        # Training performance
        train_pred = (
            data["X_train"] @ coef
            + intercept
        )

        train_metrics = regression_metrics(
            data["y_train"],
            train_pred
        )

        # Held-out client performance
        test_pred = (
            data["X_test"] @ coef
            + intercept
        )

        test_metrics = regression_metrics(
            data["y_test"],
            test_pred
        )

        rows.append({
            "Round": round_num,
            "Client": client_id,
            "Train MAE": train_metrics["MAE"],
            "Test MAE": test_metrics["MAE"],
            "Train R2": train_metrics["R2"],
            "Test R2": test_metrics["R2"],
            "Test Samples": len(data["y_test"]),
            "Personal Coef Norm": np.linalg.norm(coef),
            "Personal Intercept": intercept
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Evaluate both checkpoints
# ------------------------------------------------------------

r1_client_df = evaluate_fedper_checkpoint(1)
r100_client_df = evaluate_fedper_checkpoint(100)

fedper_client_comparison = pd.concat(
    [r1_client_df, r100_client_df],
    ignore_index=True
)

print("=" * 70)
print("FEDPER CLIENT PERFORMANCE: ROUND 1 VS ROUND 100")
print("=" * 70)

print(
    fedper_client_comparison[
        [
            "Round",
            "Client",
            "Train MAE",
            "Test MAE",
            "Train R2",
            "Test R2",
            "Test Samples"
        ]
    ].to_string(index=False)
)


# ============================================================
# WIDE COMPARISON
# ============================================================

comparison_wide = r1_client_df[
    [
        "Client",
        "Train MAE",
        "Test MAE",
        "Train R2",
        "Test R2"
    ]
].copy()

comparison_wide = comparison_wide.rename(columns={
    "Train MAE": "R1 Train MAE",
    "Test MAE": "R1 Test MAE",
    "Train R2": "R1 Train R2",
    "Test R2": "R1 Test R2"
})

r100_small = r100_client_df[
    [
        "Client",
        "Train MAE",
        "Test MAE",
        "Train R2",
        "Test R2"
    ]
].copy()

r100_small = r100_small.rename(columns={
    "Train MAE": "R100 Train MAE",
    "Test MAE": "R100 Test MAE",
    "Train R2": "R100 Train R2",
    "Test R2": "R100 Test R2"
})

comparison_wide = comparison_wide.merge(
    r100_small,
    on="Client"
)

# ------------------------------------------------------------
# Change from Round 1 -> Round 100
# ------------------------------------------------------------

comparison_wide["Test MAE Change"] = (
    comparison_wide["R100 Test MAE"]
    - comparison_wide["R1 Test MAE"]
)

comparison_wide["Test R2 Change"] = (
    comparison_wide["R100 Test R2"]
    - comparison_wide["R1 Test R2"]
)

print("\n" + "=" * 70)
print("CLIENT-LEVEL CHANGE: ROUND 1 → ROUND 100")
print("=" * 70)

print(
    comparison_wide.to_string(index=False)
)

FEDPER CLIENT PERFORMANCE: ROUND 1 VS ROUND 100
 Round Client  Train MAE  Test MAE  Train R2   Test R2  Test Samples
     1     C1   0.002719  0.009568  0.989284  0.822515             3
     1     C2   0.002356  0.004096  0.994043  0.951569             3
     1     C3   0.002252  0.003322  0.991358  0.976743             3
     1     C4   0.002212  0.001119  0.995239  0.987234             2
     1     C5   0.003011  0.007081  0.956876  0.813212             2
   100     C1   0.000774  0.012719  0.999271  0.732965             3
   100     C2   0.000556  0.005133  0.999513  0.880030             3
   100     C3   0.001214  0.013897  0.997199  0.535774             3
   100     C4   0.000676  0.002790  0.999555  0.917219             2
   100     C5   0.000875  0.026338  0.998007 -1.349714             2

CLIENT-LEVEL CHANGE: ROUND 1 → ROUND 100
Client  R1 Train MAE  R1 Test MAE  R1 Train R2  R1 Test R2  R100 Train MAE  R100 Test MAE  R100 Train R2  R100 Test R2  Test MAE Change  Test R2 Change

## Stable-23 Extension: Results and Interpretation

Within the 23-feature Stable-23 branch, the 100-round FedAvg endpoint reports MAE = **0.013600**, RMSE = **0.017133**, and R² = **0.740032** on the untouched 14-trip global test set. The matched centralized 23-feature benchmark reports MAE = **0.011506**, RMSE = **0.013549**, and R² = **0.837410**. Thus, in this branch centralized pooling again performs better, while FedAvg remains useful with a moderate performance cost.

The Stable-23 FedPer checkpoints use client-specific validation sets of only 2–3 trips. Their best recorded local-validation checkpoint is Round 1 (MAE = 0.005181, RMSE = 0.006795, R² = 0.950203), with later-round held-out performance declining while training error decreases. This suggests overfitting in this small-sample personalized setting, but is not a direct global-test comparison with FedAvg and should not support strong client-level conclusions.

The original 77-feature fixed-holdout comparison remains the primary Module 04 comparison unless a subsequent protocol explicitly selects Stable-23 as the final representation.
